# ============================================================================
# NOTEBOOK 11 — OBJECT-AWARE TRANSFORMER CAPTION GENERATOR
# ============================================================================
#
# INPUT:
#   Notebook 10 structured object features
#
# OUTPUT:
#   Object-aware Transformer captioning model
#
# ARCHITECTURE:
#   Object ID Embedding
#       +
#   Spatial Feature MLP
#       +
#   Confidence Feature MLP
#       ↓
#   Object Token Fusion
#       ↓
#   Transformer Encoder
#       ↓
#   Transformer Decoder
#       ↓
#   Caption Vocabulary Projection
#
# ============================================================================

In [ ]:
# ============================================================================
# NOTEBOOK 11 — CELL 1
# DRIVE FILE DISCOVERY & FILE STATUS
# ============================================================================

from pathlib import Path
import os
import google.colab.drive

print("=" * 75)
print("NOTEBOOK 11 — DRIVE FILE DISCOVERY & FILE STATUS")
print("=" * 75)

# --------------------------------------------------------------------------
# MOUNT GOOGLE DRIVE IF NOT ALREADY MOUNTED
# --------------------------------------------------------------------------

if not Path("/content/drive").exists():
    print("Mounting Google Drive...")
    google.colab.drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

# --------------------------------------------------------------------------
# LOCATE PROJECT ROOT
# --------------------------------------------------------------------------

DRIVE_ROOT = None

if Path("/content/drive/MyDrive").exists():
    print("\nSearching for project directory under /content/drive/MyDrive...")

    # First, try to find a directory named "Image_captioning" as indicated by the user
    image_captioning_path = Path("/content/drive/MyDrive") / "Image_captioning"
    if image_captioning_path.exists() and image_captioning_path.is_dir():
        DRIVE_ROOT = image_captioning_path
        print(f"Found 'Image_captioning' directory: {DRIVE_ROOT}")
    else:
        # Fallback: search for any directory containing an 'Outputs' subdirectory
        potential_project_dirs = []
        for entry in Path("/content/drive/MyDrive").iterdir():
            if entry.is_dir() and (entry / "Outputs").exists():
                potential_project_dirs.append(entry)

        if potential_project_dirs:
            # If multiple potential roots, pick the first one, but inform the user.
            DRIVE_ROOT = potential_project_dirs[0]
            if len(potential_project_dirs) > 1:
                print(f"Found multiple potential project directories containing 'Outputs'. Using the first one: {DRIVE_ROOT}")
                print(f"Other potential roots: {potential_project_dirs[1:]}")
            else:
                print(f"Found potential project directory containing 'Outputs': {DRIVE_ROOT}")
        else:
            print("Could not automatically locate the project directory within /content/drive/MyDrive.")
            print("Please ensure your project folder ('Image_captioning' or a folder containing 'Outputs') is directly under MyDrive.")
            print("You may need to manually set the DRIVE_ROOT variable.")
else:
    print("/content/drive/MyDrive does not exist. Please check your Google Drive setup.")


if DRIVE_ROOT:
    print("\nProject root successfully identified:")
    print(DRIVE_ROOT)
else:
    print("\nERROR: DRIVE_ROOT could not be determined. Subsequent file operations may fail.")


# --------------------------------------------------------------------------
# IMPORTANT PERMANENT FILES (these depend on DRIVE_ROOT)
# --------------------------------------------------------------------------

# Only define these if DRIVE_ROOT is found
if DRIVE_ROOT:
    RTDETR_CSV = (
        DRIVE_ROOT
        / "Outputs"
        / "RTDETR_Detection"
        / "Final"
        / "rtdetr_x_detections_full.csv"
    )

    CAPTIONS_CSV = (
        DRIVE_ROOT
        / "Dataset"
        / "Processed"
        / "Captions"
        / "captions_processed.csv"
    )

    STRUCTURED_DIR = (
        DRIVE_ROOT
        / "Features"
        / "Structured_Object"
    )

    STRUCTURED_NPZ = (
        STRUCTURED_DIR
        / "rtdetr_structured_object_features.npz"
    )

    CAPTION_NPZ = (
        STRUCTURED_DIR
        / "caption_sequences.npz"
    )

    OUTPUT_DIR = (
        DRIVE_ROOT
        / "Outputs"
        / "Structured_Captioning"
    )

    OBJECT_VOCAB = (
        OUTPUT_DIR
        / "object_vocabulary.json"
    )

    CAPTION_VOCAB = (
        OUTPUT_DIR
        / "caption_vocabulary.json"
    )

    CONFIG_FILE = (
        OUTPUT_DIR
        / "structured_captioning_config.json"
    )

    IMAGE_METADATA = (
        OUTPUT_DIR
        / "image_object_metadata.csv"
    )

    TRAIN_IDS = (
        OUTPUT_DIR
        / "train_image_ids.csv"
    )

    VAL_IDS = (
        OUTPUT_DIR
        / "val_image_ids.csv"
    )

    TEST_IDS = (
        OUTPUT_DIR
        / "test_image_ids.csv"
    )

    # --------------------------------------------------------------------------
    # STATUS OF IMPORTANT FILES
    # (Covers user's requested CELL 2)
    # --------------------------------------------------------------------------

    files = {
        "RT-DETR CSV": RTDETR_CSV,
        "Captions CSV": CAPTIONS_CSV,
        "Structured Object NPZ": STRUCTURED_NPZ,
        "Caption NPZ": CAPTION_NPZ,
        "Object Vocabulary": OBJECT_VOCAB,
        "Caption Vocabulary": CAPTION_VOCAB,
        "Config": CONFIG_FILE,
        "Image Metadata": IMAGE_METADATA,
        "Train IDs": TRAIN_IDS,
        "Val IDs": VAL_IDS,
        "Test IDs": TEST_IDS,
    }

    print("\nFILE STATUS")
    print("-" * 75)

    for name, path in files.items():
        status = "FOUND" if path.exists() else "MISSING"
        print(f"{name:25s}: {status} - {path}")

else:
    print("\nSkipping detailed file status due to missing DRIVE_ROOT.")

print("\n" + "=" * 75)
print("CELL 1 COMPLETE")
print("=" * 75)


NOTEBOOK 11 — DRIVE FILE DISCOVERY & FILE STATUS
Mounting Google Drive...
Mounted at /content/drive

Searching for project directory under /content/drive/MyDrive...
Found 'Image_captioning' directory: /content/drive/MyDrive/Image_captioning

Project root successfully identified:
/content/drive/MyDrive/Image_captioning

FILE STATUS
---------------------------------------------------------------------------
RT-DETR CSV              : FOUND - /content/drive/MyDrive/Image_captioning/Outputs/RTDETR_Detection/Final/rtdetr_x_detections_full.csv
Captions CSV             : FOUND - /content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv
Structured Object NPZ    : MISSING - /content/drive/MyDrive/Image_captioning/Features/Structured_Object/rtdetr_structured_object_features.npz
Caption NPZ              : MISSING - /content/drive/MyDrive/Image_captioning/Features/Structured_Object/caption_sequences.npz
Object Vocabulary        : FOUND - /content/drive/MyDrive/Image

In [ ]:
# ============================================================
# CELL 2 — INSPECTING RT-DETR-X DETECTIONS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 75)
print("CELL 2 — INSPECTING RT-DETR-X DETECTIONS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Locate project root safely
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Image_captioning")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project directory not found:\n{PROJECT_ROOT}\n\n"
        "Run the Drive discovery/mounting cell first."
    )

print(f"\nProject root: {PROJECT_ROOT}")

# ------------------------------------------------------------
# 2. Locate RT-DETR CSV
# ------------------------------------------------------------

RTDETR_CSV = (
    PROJECT_ROOT
    / "Outputs"
    / "RTDETR_Detection"
    / "Final"
    / "rtdetr_x_detections_full.csv"
)

if not RTDETR_CSV.exists():
    raise FileNotFoundError(
        f"RT-DETR CSV not found:\n{RTDETR_CSV}"
    )

print(f"RT-DETR CSV: FOUND")
print(f"Path: {RTDETR_CSV}")

# ------------------------------------------------------------
# 3. Load CSV
# ------------------------------------------------------------

rtdetr_df = pd.read_csv(RTDETR_CSV)

print("\n" + "-" * 75)
print("DATASET SHAPE")
print("-" * 75)

print(f"Rows    : {len(rtdetr_df):,}")
print(f"Columns : {len(rtdetr_df.columns)}")

# ------------------------------------------------------------
# 4. Column information
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("COLUMNS")
print("-" * 75)

for i, col in enumerate(rtdetr_df.columns, 1):
    print(f"{i:2d}. {col}")

# ------------------------------------------------------------
# 5. Data types
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("DATA TYPES")
print("-" * 75)

print(rtdetr_df.dtypes)

# ------------------------------------------------------------
# 6. First 5 rows
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("SAMPLE ROWS")
print("-" * 75)

display(rtdetr_df.head())

# ------------------------------------------------------------
# 7. Automatically identify likely columns
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("CANDIDATE COLUMN DETECTION")
print("-" * 75)

columns_lower = {
    col: col.lower()
    for col in rtdetr_df.columns
}

image_candidates = [
    col for col, low in columns_lower.items()
    if any(x in low for x in [
        "image", "filename", "file_name", "img"
    ])
]

object_candidates = [
    col for col, low in columns_lower.items()
    if any(x in low for x in [
        "class", "label", "object", "category", "name"
    ])
]

confidence_candidates = [
    col for col, low in columns_lower.items()
    if any(x in low for x in [
        "confidence", "conf", "score", "prob"
    ])
]

box_candidates = [
    col for col, low in columns_lower.items()
    if any(x in low for x in [
        "xmin", "xmax", "ymin", "ymax",
        "x1", "x2", "y1", "y2",
        "bbox", "box"
    ])
]

print("\nImage candidates:")
print(image_candidates)

print("\nObject/class candidates:")
print(object_candidates)

print("\nConfidence candidates:")
print(confidence_candidates)

print("\nBounding-box candidates:")
print(box_candidates)

# ------------------------------------------------------------
# 8. Basic unique-image information
# ------------------------------------------------------------

if image_candidates:
    image_col = image_candidates[0]

    print("\n" + "-" * 75)
    print("IMAGE STATISTICS")
    print("-" * 75)

    print(f"Selected image column: {image_col}")
    print(f"Unique images: {rtdetr_df[image_col].nunique():,}")
    print(f"Missing image values: {rtdetr_df[image_col].isna().sum():,}")

print("\n" + "=" * 75)
print("CELL 2 COMPLETE")
print("=" * 75)

CELL 2 — INSPECTING RT-DETR-X DETECTIONS

Project root: /content/drive/MyDrive/Image_captioning
RT-DETR CSV: FOUND
Path: /content/drive/MyDrive/Image_captioning/Outputs/RTDETR_Detection/Final/rtdetr_x_detections_full.csv

---------------------------------------------------------------------------
DATASET SHAPE
---------------------------------------------------------------------------
Rows    : 496,803
Columns : 9

---------------------------------------------------------------------------
COLUMNS
---------------------------------------------------------------------------
 1. image_id
 2. image_path
 3. object_name
 4. confidence
 5. xmin
 6. ymin
 7. xmax
 8. ymax
 9. objects_detected

---------------------------------------------------------------------------
DATA TYPES
---------------------------------------------------------------------------
image_id             object
image_path           object
object_name          object
confidence          float64
xmin                float64
y

,image_id,image_path,object_name,confidence,xmin,ymin,xmax,ymax,objects_detected
0,Flickr30K/1000092795.jpg,/content/Image_captioning_Local/Flickr30K/flic...,person,0.936982,158.854370,111.327011,258.739990,368.602814,3
1,Flickr30K/1000092795.jpg,/content/Image_captioning_Local/Flickr30K/flic...,person,0.849433,159.395081,124.652122,214.009247,331.089111,3
2,Flickr30K/1000092795.jpg,/content/Image_captioning_Local/Flickr30K/flic...,person,0.341635,172.162399,125.140846,222.477951,332.650848,3
3,Flickr30K/10002456.jpg,/content/Image_captioning_Local/Flickr30K/flic...,person,0.893061,187.945755,142.208023,254.382584,285.277100,7
4,Flickr30K/10002456.jpg,/content/Image_captioning_Local/Flickr30K/flic...,person,0.755438,252.329849,179.281952,338.053253,309.793182,7



---------------------------------------------------------------------------
CANDIDATE COLUMN DETECTION
---------------------------------------------------------------------------

Image candidates:
['image_id', 'image_path']

Object/class candidates:
['object_name', 'objects_detected']

Confidence candidates:
['confidence']

Bounding-box candidates:
['xmin', 'ymin', 'xmax', 'ymax']

---------------------------------------------------------------------------
IMAGE STATISTICS
---------------------------------------------------------------------------
Selected image column: image_id
Unique images: 39,874
Missing image values: 0

CELL 2 COMPLETE


In [ ]:
# ============================================================
# CELL 3 — INSPECT EXISTING STRUCTURED-CAPTIONING OUTPUTS
# ============================================================

import json
from pathlib import Path

print("=" * 75)
print("CELL 3 — INSPECTING EXISTING NOTEBOOK 10 OUTPUTS")
print("=" * 75)

PROJECT_ROOT = Path("/content/drive/MyDrive/Image_captioning")

STRUCTURED_DIR = PROJECT_ROOT / "Outputs" / "Structured_Captioning"
FEATURE_DIR = PROJECT_ROOT / "Features" / "Structured_Object"

print(f"\nStructured output directory: {STRUCTURED_DIR}")
print(f"Feature directory:           {FEATURE_DIR}")

# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------

expected_files = [
    STRUCTURED_DIR / "object_vocabulary.json",
    STRUCTURED_DIR / "caption_vocabulary.json",
    STRUCTURED_DIR / "structured_captioning_config.json",
    STRUCTURED_DIR / "image_object_metadata.csv",
    STRUCTURED_DIR / "train_image_ids.csv",
    STRUCTURED_DIR / "val_image_ids.csv",
    STRUCTURED_DIR / "test_image_ids.csv",
    FEATURE_DIR / "rtdetr_structured_object_features.npz",
    FEATURE_DIR / "caption_sequences.npz",
]

print("\n" + "-" * 75)
print("FILE STATUS")
print("-" * 75)

for path in expected_files:
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status:8s} : {path}")

# ------------------------------------------------------------
# Inspect object vocabulary
# ------------------------------------------------------------

object_vocab_path = STRUCTURED_DIR / "object_vocabulary.json"

if not object_vocab_path.exists():
    raise FileNotFoundError(f"Missing object vocabulary:\n{object_vocab_path}")

with open(object_vocab_path, "r") as f:
    object_vocab = json.load(f)

print("\n" + "-" * 75)
print("OBJECT VOCABULARY")
print("-" * 75)

print("Top-level keys:", list(object_vocab.keys()))

print("object_to_id size:",
      len(object_vocab.get("object_to_id", {})))

print("id_to_object size:",
      len(object_vocab.get("id_to_object", {})))

print("\nFirst 15 object mappings:")
for k, v in list(object_vocab.get("object_to_id", {}).items())[:15]:
    print(f"  {k!r} -> {v}")

# ------------------------------------------------------------
# Inspect caption vocabulary
# ------------------------------------------------------------

caption_vocab_path = STRUCTURED_DIR / "caption_vocabulary.json"

if not caption_vocab_path.exists():
    raise FileNotFoundError(f"Missing caption vocabulary:\n{caption_vocab_path}")

with open(caption_vocab_path, "r") as f:
    caption_vocab = json.load(f)

print("\n" + "-" * 75)
print("CAPTION VOCABULARY")
print("-" * 75)

print("Top-level keys:", list(caption_vocab.keys()))

print("caption_to_id size:",
      len(caption_vocab.get("caption_to_id", {})))

print("id_to_caption size:",
      len(caption_vocab.get("id_to_caption", {})))

print("special_tokens:",
      caption_vocab.get("special_tokens"))

print("max_len:",
      caption_vocab.get("max_len"))

print("min_word_freq:",
      caption_vocab.get("min_word_freq"))

print("\n" + "=" * 75)
print("CELL 3 COMPLETE")
print("=" * 75)

CELL 3 — INSPECTING EXISTING NOTEBOOK 10 OUTPUTS

Structured output directory: /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning
Feature directory:           /content/drive/MyDrive/Image_captioning/Features/Structured_Object

---------------------------------------------------------------------------
FILE STATUS
---------------------------------------------------------------------------
FOUND    : /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/object_vocabulary.json
FOUND    : /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/caption_vocabulary.json
FOUND    : /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/structured_captioning_config.json
FOUND    : /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/image_object_metadata.csv
FOUND    : /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/train_image_ids.csv
FOUND    : /content/drive/MyDrive/Image_captioning/Outputs

In [ ]:
# ============================================================
# CELL 4 — VERIFY RT-DETR DETECTION RECORDS
# ============================================================

print("=" * 75)
print("CELL 4 — VERIFYING RT-DETR DETECTION RECORDS")
print("=" * 75)

# ------------------------------------------------------------
# Basic information
# ------------------------------------------------------------

print(f"\nTotal CSV rows: {len(rtdetr_df):,}")

# ------------------------------------------------------------
# Identify placeholder / real detection rows
# ------------------------------------------------------------

# Real detections have a non-empty object_name.
real_detection_mask = (
    rtdetr_df["object_name"].notna()
    & rtdetr_df["object_name"].astype(str).str.strip().ne("")
)

placeholder_mask = ~real_detection_mask

real_df = rtdetr_df.loc[real_detection_mask].copy()
placeholder_df = rtdetr_df.loc[placeholder_mask].copy()

print("\n" + "-" * 75)
print("DETECTION RECORD STATUS")
print("-" * 75)

print(f"Real detection rows : {len(real_df):,}")
print(f"Placeholder rows    : {len(placeholder_df):,}")

print(f"Real detection images: "
      f"{real_df['image_id'].nunique():,}")

print(f"Placeholder images: "
      f"{placeholder_df['image_id'].nunique():,}")

# ------------------------------------------------------------
# Confidence statistics
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("CONFIDENCE STATISTICS")
print("-" * 75)

print(real_df["confidence"].describe())

# ------------------------------------------------------------
# Detection counts per image
# ------------------------------------------------------------

detections_per_image = (
    real_df.groupby("image_id")
    .size()
)

print("\n" + "-" * 75)
print("DETECTIONS PER IMAGE")
print("-" * 75)

print(detections_per_image.describe())

print("\nMaximum detections in one image:",
      detections_per_image.max())

print("\nImages with >10 detections:",
      (detections_per_image > 10).sum())

# ------------------------------------------------------------
# Object classes
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("OBJECT CLASSES")
print("-" * 75)

unique_objects = sorted(
    real_df["object_name"].astype(str).str.strip().unique()
)

print(f"Unique detected object classes: {len(unique_objects)}")

print("\nFirst 80 classes:")
print(unique_objects[:80])

# ------------------------------------------------------------
# Bounding box sanity check
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("BOUNDING BOX CHECK")
print("-" * 75)

for col in ["xmin", "ymin", "xmax", "ymax"]:
    print(
        f"{col:5s} | "
        f"min={real_df[col].min():.3f} | "
        f"max={real_df[col].max():.3f} | "
        f"missing={real_df[col].isna().sum():,}"
    )

print("\n" + "=" * 75)
print("CELL 4 COMPLETE")
print("=" * 75)

CELL 4 — VERIFYING RT-DETR DETECTION RECORDS

Total CSV rows: 496,803

---------------------------------------------------------------------------
DETECTION RECORD STATUS
---------------------------------------------------------------------------
Real detection rows : 496,797
Placeholder rows    : 6
Real detection images: 39,868
Placeholder images: 6

---------------------------------------------------------------------------
CONFIDENCE STATISTICS
---------------------------------------------------------------------------
count    496797.000000
mean          0.521480
std           0.245220
min           0.250000
25%           0.306366
50%           0.423132
75%           0.761513
max           0.986814
Name: confidence, dtype: float64

---------------------------------------------------------------------------
DETECTIONS PER IMAGE
---------------------------------------------------------------------------
count    39868.000000
mean        12.461046
std         12.953297
min          1.

In [ ]:
# ============================================================
# CELL 5 — INSPECT CAPTIONS AND IMAGE ALIGNMENT
# ============================================================

print("=" * 75)
print("CELL 5 — INSPECTING CAPTIONS")
print("=" * 75)

CAPTIONS_CSV = (
    PROJECT_ROOT
    / "Dataset"
    / "Processed"
    / "Captions"
    / "captions_processed.csv"
)

if not CAPTIONS_CSV.exists():
    raise FileNotFoundError(
        f"Processed captions file not found:\n{CAPTIONS_CSV}"
    )

captions_df = pd.read_csv(CAPTIONS_CSV)

print(f"\nCaptions CSV: {CAPTIONS_CSV}")

print("\n" + "-" * 75)
print("CAPTION DATASET")
print("-" * 75)

print(f"Rows    : {len(captions_df):,}")
print(f"Columns : {len(captions_df.columns)}")

print("\nColumns:")
for i, col in enumerate(captions_df.columns, 1):
    print(f"{i:2d}. {col}")

print("\nData types:")
print(captions_df.dtypes)

print("\nSample rows:")
display(captions_df.head())

# ------------------------------------------------------------
# Candidate image/caption columns
# ------------------------------------------------------------

caption_columns_lower = {
    col: col.lower()
    for col in captions_df.columns
}

caption_image_candidates = [
    col for col, low in caption_columns_lower.items()
    if any(x in low for x in [
        "image", "filename", "file_name", "img"
    ])
]

caption_text_candidates = [
    col for col, low in caption_columns_lower.items()
    if any(x in low for x in [
        "caption", "text", "description"
    ])
]

print("\n" + "-" * 75)
print("COLUMN CANDIDATES")
print("-" * 75)

print("Image candidates:")
print(caption_image_candidates)

print("\nCaption-text candidates:")
print(caption_text_candidates)

# ------------------------------------------------------------
# Verify expected dataset size
# ------------------------------------------------------------

if caption_image_candidates:
    caption_image_col = caption_image_candidates[0]

    print("\n" + "-" * 75)
    print("IMAGE ALIGNMENT")
    print("-" * 75)

    print(
        f"Unique caption images: "
        f"{captions_df[caption_image_col].nunique():,}"
    )

    print(
        f"RT-DETR unique images: "
        f"{rtdetr_df['image_id'].nunique():,}"
    )

print("\n" + "=" * 75)
print("CELL 5 COMPLETE")
print("=" * 75)

CELL 5 — INSPECTING CAPTIONS

Captions CSV: /content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv

---------------------------------------------------------------------------
CAPTION DATASET
---------------------------------------------------------------------------
Rows    : 199,369
Columns : 7

Columns:
 1. image_id
 2. dataset
 3. image_name
 4. caption
 5. processed_caption
 6. model_caption
 7. caption_length

Data types:
image_id             object
dataset              object
image_name           object
caption              object
processed_caption    object
model_caption        object
caption_length        int64
dtype: object

Sample rows:


,image_id,dataset,image_name,caption,processed_caption,model_caption,caption_length
0,Flickr8K/1000268201_693b08cb0e.jpg,Flickr8K,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...,a child in a pink dress is climbing up a set o...,<start> a child in a pink dress is climbing up...,19
1,Flickr8K/1000268201_693b08cb0e.jpg,Flickr8K,1000268201_693b08cb0e.jpg,A girl going into a wooden building .,a girl going into a wooden building.,<start> a girl going into a wooden building. <...,9
2,Flickr8K/1000268201_693b08cb0e.jpg,Flickr8K,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .,a little girl climbing into a wooden playhouse.,<start> a little girl climbing into a wooden p...,10
3,Flickr8K/1000268201_693b08cb0e.jpg,Flickr8K,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...,a little girl climbing the stairs to her playh...,<start> a little girl climbing the stairs to h...,11
4,Flickr8K/1000268201_693b08cb0e.jpg,Flickr8K,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...,a little girl in a pink dress going into a woo...,<start> a little girl in a pink dress going in...,14



---------------------------------------------------------------------------
COLUMN CANDIDATES
---------------------------------------------------------------------------
Image candidates:
['image_id', 'image_name']

Caption-text candidates:
['caption', 'processed_caption', 'model_caption', 'caption_length']

---------------------------------------------------------------------------
IMAGE ALIGNMENT
---------------------------------------------------------------------------
Unique caption images: 39,874
RT-DETR unique images: 39,874

CELL 5 COMPLETE


In [ ]:
# ============================================================
# CELL 6 — VERIFY TRAIN / VAL / TEST SPLITS
# ============================================================

print("=" * 75)
print("CELL 6 — VERIFYING DATASET SPLITS")
print("=" * 75)

split_files = {
    "train": STRUCTURED_DIR / "train_image_ids.csv",
    "val": STRUCTURED_DIR / "val_image_ids.csv",
    "test": STRUCTURED_DIR / "test_image_ids.csv",
}

split_data = {}

for split_name, split_path in split_files.items():

    if not split_path.exists():
        raise FileNotFoundError(
            f"Missing {split_name} split file:\n{split_path}"
        )

    df = pd.read_csv(split_path)

    split_data[split_name] = df

    print("\n" + "-" * 75)
    print(f"{split_name.upper()} SPLIT")
    print("-" * 75)

    print(f"Path: {split_path}")
    print(f"Shape: {df.shape}")

    print("\nColumns:")
    print(list(df.columns))

    print("\nFirst 5 rows:")
    display(df.head())

# ------------------------------------------------------------
# Determine ID column
# ------------------------------------------------------------

def find_id_column(df):
    candidates = [
        c for c in df.columns
        if "image" in c.lower() or "id" in c.lower()
    ]

    if not candidates:
        raise ValueError(
            f"Could not identify image ID column in columns: "
            f"{list(df.columns)}"
        )

    return candidates[0]


split_id_columns = {
    name: find_id_column(df)
    for name, df in split_data.items()
}

print("\n" + "-" * 75)
print("SPLIT ID COLUMNS")
print("-" * 75)

for name, col in split_id_columns.items():
    print(f"{name:5s}: {col}")

# ------------------------------------------------------------
# Check sizes and overlap
# ------------------------------------------------------------

train_ids = set(
    split_data["train"][split_id_columns["train"]].astype(str)
)

val_ids = set(
    split_data["val"][split_id_columns["val"]].astype(str)
)

test_ids = set(
    split_data["test"][split_id_columns["test"]].astype(str)
)

print("\n" + "-" * 75)
print("SPLIT STATISTICS")
print("-" * 75)

print(f"Train: {len(train_ids):,}")
print(f"Val  : {len(val_ids):,}")
print(f"Test : {len(test_ids):,}")
print(f"Total: {len(train_ids | val_ids | test_ids):,}")

print("\nOverlap checks:")
print("Train ∩ Val :", len(train_ids & val_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Val ∩ Test  :", len(val_ids & test_ids))

print("\n" + "=" * 75)
print("CELL 6 COMPLETE")
print("=" * 75)

CELL 6 — VERIFYING DATASET SPLITS

---------------------------------------------------------------------------
TRAIN SPLIT
---------------------------------------------------------------------------
Path: /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/train_image_ids.csv
Shape: (31899, 1)

Columns:
['image_id']

First 5 rows:


,image_id
0,Flickr8K/263522013_d118d46b2d.jpg
1,Flickr8K/3415646718_f9f4e23a66.jpg
2,Flickr8K/3613667665_1881c689ea.jpg
3,Flickr8K/1785138090_76a56aaabc.jpg
4,Flickr30K/4468388551.jpg



---------------------------------------------------------------------------
VAL SPLIT
---------------------------------------------------------------------------
Path: /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/val_image_ids.csv
Shape: (3987, 1)

Columns:
['image_id']

First 5 rows:


,image_id
0,Flickr30K/987442144.jpg
1,Flickr30K/2411585767.jpg
2,Flickr30K/109741554.jpg
3,Flickr30K/2844641033.jpg
4,Flickr30K/3225478803.jpg



---------------------------------------------------------------------------
TEST SPLIT
---------------------------------------------------------------------------
Path: /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/test_image_ids.csv
Shape: (3988, 1)

Columns:
['image_id']

First 5 rows:


,image_id
0,Flickr30K/551403320.jpg
1,Flickr30K/1422951807.jpg
2,Flickr30K/3484365373.jpg
3,Flickr30K/3093523191.jpg
4,Flickr30K/3585495069.jpg



---------------------------------------------------------------------------
SPLIT ID COLUMNS
---------------------------------------------------------------------------
train: image_id
val  : image_id
test : image_id

---------------------------------------------------------------------------
SPLIT STATISTICS
---------------------------------------------------------------------------
Train: 31,899
Val  : 3,987
Test : 3,988
Total: 39,874

Overlap checks:
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

CELL 6 COMPLETE


In [ ]:
# ============================================================
# CELL 7 — LOAD EXISTING VOCABULARIES + CONFIG
# ============================================================

import json
from pathlib import Path

print("=" * 75)
print("CELL 7 — LOADING SAVED VOCABULARIES AND CONFIG")
print("=" * 75)

PROJECT_ROOT = Path("/content/drive/MyDrive/Image_captioning")

STRUCTURED_DIR = PROJECT_ROOT / "Outputs" / "Structured_Captioning"
FEATURE_DIR = PROJECT_ROOT / "Features" / "Structured_Object"

# ------------------------------------------------------------
# Load object vocabulary
# ------------------------------------------------------------

OBJECT_VOCAB_PATH = STRUCTURED_DIR / "object_vocabulary.json"
CAPTION_VOCAB_PATH = STRUCTURED_DIR / "caption_vocabulary.json"
CONFIG_PATH = STRUCTURED_DIR / "structured_captioning_config.json"

with open(OBJECT_VOCAB_PATH, "r") as f:
    object_vocab_raw = json.load(f)

with open(CAPTION_VOCAB_PATH, "r") as f:
    caption_vocab_raw = json.load(f)

with open(CONFIG_PATH, "r") as f:
    config_raw = json.load(f)

print("\nObject vocabulary type:", type(object_vocab_raw))
print("Caption vocabulary type:", type(caption_vocab_raw))

# ------------------------------------------------------------
# Normalize object vocabulary
#
# Notebook 10 saved it directly as:
# {
#   "<pad>": 0,
#   "<unk>": 1,
#   "person": ...,
#   ...
# }
# ------------------------------------------------------------

if "object_to_id" in object_vocab_raw:
    object_to_id = object_vocab_raw["object_to_id"]
else:
    object_to_id = object_vocab_raw

object_to_id = {
    str(k): int(v)
    for k, v in object_to_id.items()
}

id_to_object = {
    int(v): str(k)
    for k, v in object_to_id.items()
}

# ------------------------------------------------------------
# Normalize caption vocabulary
# ------------------------------------------------------------

if "caption_to_id" in caption_vocab_raw:
    caption_to_id = caption_vocab_raw["caption_to_id"]
else:
    caption_to_id = caption_vocab_raw

caption_to_id = {
    str(k): int(v)
    for k, v in caption_to_id.items()
}

id_to_caption = {
    int(v): str(k)
    for k, v in caption_to_id.items()
}

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("OBJECT VOCABULARY")
print("-" * 75)

print("Vocabulary size:", len(object_to_id))
print("PAD ID:", object_to_id.get("<pad>"))
print("UNK ID:", object_to_id.get("<unk>"))

print("\nFirst 15:")
for token, idx in list(object_to_id.items())[:15]:
    print(f"{token!r:20s} -> {idx}")

print("\n" + "-" * 75)
print("CAPTION VOCABULARY")
print("-" * 75)

print("Vocabulary size:", len(caption_to_id))
print("PAD ID:", caption_to_id.get("<pad>"))
print("START ID:", caption_to_id.get("<start>"))
print("END ID:", caption_to_id.get("<end>"))
print("UNK ID:", caption_to_id.get("<unk>"))

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("SAVED CONFIG")
print("-" * 75)

print(json.dumps(config_raw, indent=2))

print("\n" + "=" * 75)
print("CELL 7 COMPLETE")
print("=" * 75)

CELL 7 — LOADING SAVED VOCABULARIES AND CONFIG

Object vocabulary type: <class 'dict'>
Caption vocabulary type: <class 'dict'>

---------------------------------------------------------------------------
OBJECT VOCABULARY
---------------------------------------------------------------------------
Vocabulary size: 82
PAD ID: 0
UNK ID: 1

First 15:
'<pad>'              -> 0
'<unk>'              -> 1
'person'             -> 2
'car'                -> 3
'chair'              -> 4
'handbag'            -> 5
'dog'                -> 6
'backpack'           -> 7
'bottle'             -> 8
'bicycle'            -> 9
'cup'                -> 10
'bench'              -> 11
'sports ball'        -> 12
'book'               -> 13
'boat'               -> 14

---------------------------------------------------------------------------
CAPTION VOCABULARY
---------------------------------------------------------------------------
Vocabulary size: 12174
PAD ID: 0
START ID: 1
END ID: 2
UNK ID: 3

------------------

In [ ]:
# ============================================================
# CELL 8 — VALIDATE STRUCTURED OBJECT REPRESENTATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("CELL 8 — VALIDATING STRUCTURED OBJECT REPRESENTATION")
print("=" * 75)

CONF_THRESHOLD = 0.30
TOP_K = 10
SPATIAL_DIM = 9

# ------------------------------------------------------------
# Keep only genuine detections
# ------------------------------------------------------------

real_mask = (
    rtdetr_df["object_name"].notna()
    & rtdetr_df["object_name"].astype(str).str.strip().ne("")
    & rtdetr_df["confidence"].notna()
)

detections = rtdetr_df.loc[real_mask].copy()

# ------------------------------------------------------------
# Apply Notebook 10 confidence threshold
# ------------------------------------------------------------

detections = detections[
    detections["confidence"] >= CONF_THRESHOLD
].copy()

print(f"\nConfidence threshold: {CONF_THRESHOLD}")
print(f"Detections after threshold: {len(detections):,}")
print(f"Images represented: {detections['image_id'].nunique():,}")

# ------------------------------------------------------------
# Convert object names to IDs
# ------------------------------------------------------------

object_unk_id = object_to_id["<unk>"]

detections["object_id"] = (
    detections["object_name"]
    .astype(str)
    .str.strip()
    .map(object_to_id)
    .fillna(object_unk_id)
    .astype(np.int64)
)

unknown_objects = detections.loc[
    ~detections["object_name"].astype(str).str.strip().isin(object_to_id),
    "object_name"
].nunique()

print(f"Unknown object classes: {unknown_objects}")

# ------------------------------------------------------------
# Normalize / clip bounding boxes
#
# RT-DETR CSV contains absolute pixel coordinates.
# We use image_path to determine image dimensions.
# ------------------------------------------------------------

print("\nPreparing sample spatial features...")

sample_df = detections.head(20).copy()

# We intentionally inspect the raw box values here.
print("\nSample raw boxes:")
display(
    sample_df[
        [
            "image_id",
            "object_name",
            "confidence",
            "xmin",
            "ymin",
            "xmax",
            "ymax"
        ]
    ]
)

# ------------------------------------------------------------
# Check whether image paths are available
# ------------------------------------------------------------

print("\nSample image paths:")
for p in sample_df["image_path"].head(5):
    print(p)

print("\n" + "=" * 75)
print("CELL 8 COMPLETE")
print("=" * 75)

CELL 8 — VALIDATING STRUCTURED OBJECT REPRESENTATION

Confidence threshold: 0.3
Detections after threshold: 383,229
Images represented: 39,864
Unknown object classes: 0

Preparing sample spatial features...

Sample raw boxes:


,image_id,object_name,confidence,xmin,ymin,xmax,ymax
0,Flickr30K/1000092795.jpg,person,0.936982,158.854370,111.327011,258.739990,368.602814
1,Flickr30K/1000092795.jpg,person,0.849433,159.395081,124.652122,214.009247,331.089111
2,Flickr30K/1000092795.jpg,person,0.341635,172.162399,125.140846,222.477951,332.650848
3,Flickr30K/10002456.jpg,person,0.893061,187.945755,142.208023,254.382584,285.277100
4,Flickr30K/10002456.jpg,person,0.755438,252.329849,179.281952,338.053253,309.793182
5,Flickr30K/10002456.jpg,person,0.626827,333.061707,158.469376,418.241852,324.066223
6,Flickr30K/10002456.jpg,person,0.426712,333.227844,158.726364,401.414368,260.692474
7,Flickr30K/10002456.jpg,person,0.390946,386.533661,221.447617,419.010956,324.574219
10,Flickr30K/1000268201.jpg,person,0.952038,5.092531,149.978699,106.765030,338.898926
11,Flickr30K/1000268201.jpg,bench,0.499278,-0.133119,359.352661,85.085625,427.592468



Sample image paths:
/content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/1000092795.jpg
/content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/1000092795.jpg
/content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/1000092795.jpg
/content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/10002456.jpg
/content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/10002456.jpg

CELL 8 COMPLETE


In [ ]:
# ============================================================
# CELL 9A — LOCATE DATASET ZIP FILES
# ============================================================

from pathlib import Path

print("=" * 75)
print("CELL 9A — LOCATING FLICKR DATASET ZIPS")
print("=" * 75)

PROJECT_ROOT = Path("/content/drive/MyDrive/Image_captioning")
ZIP_DIR = PROJECT_ROOT / "Dataset" / "ZIP"

print(f"\nZIP directory:")
print(ZIP_DIR)

if not ZIP_DIR.exists():
    raise FileNotFoundError(
        f"ZIP directory does not exist:\n{ZIP_DIR}"
    )

print("\nFiles found:")

zip_files = list(ZIP_DIR.glob("*.zip"))

for z in zip_files:
    size_gb = z.stat().st_size / (1024 ** 3)
    print(f"  {z.name:30s} {size_gb:.2f} GB")

print("\n" + "-" * 75)

f8k_zip = ZIP_DIR / "flickr8k.zip"
f30k_zip = ZIP_DIR / "flickr30k.zip"

print("Flickr8K ZIP :", "FOUND" if f8k_zip.exists() else "MISSING")
print("Flickr30K ZIP:", "FOUND" if f30k_zip.exists() else "MISSING")

if not f8k_zip.exists() or not f30k_zip.exists():
    raise FileNotFoundError(
        "One or both required dataset ZIP files are missing."
    )

print("\n✓ Both required ZIP files are available.")
print("\n" + "=" * 75)
print("CELL 9A COMPLETE")
print("=" * 75)

CELL 9A — LOCATING FLICKR DATASET ZIPS

ZIP directory:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP

Files found:
  flickr8k.zip                   1.04 GB
  flickr30k.zip                  8.16 GB

---------------------------------------------------------------------------
Flickr8K ZIP : FOUND
Flickr30K ZIP: FOUND

✓ Both required ZIP files are available.

CELL 9A COMPLETE


In [ ]:
# ============================================================
# CELL 9B — EXTRACT FLICKR DATASETS TO LOCAL RUNTIME
# ============================================================

import zipfile
import shutil
from pathlib import Path

print("=" * 75)
print("CELL 9B — EXTRACTING FLICKR DATASETS")
print("=" * 75)

LOCAL_ROOT = Path("/content/Image_captioning_Local")

# ------------------------------------------------------------
# Create local root
# ------------------------------------------------------------

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def extract_zip(zip_path, destination):
    print(f"\nExtracting: {zip_path.name}")
    print(f"Destination: {destination}")

    with zipfile.ZipFile(zip_path, "r") as z:
        members = z.namelist()

        print(f"Archive members: {len(members):,}")

        z.extractall(destination)

    print("✓ Extraction complete")


# ------------------------------------------------------------
# Extract only if not already available
# ------------------------------------------------------------

f8k_destination = LOCAL_ROOT / "Flickr8K"
f30k_destination = LOCAL_ROOT / "Flickr30K"

# Flickr8K
if not f8k_destination.exists():
    extract_zip(f8k_zip, f8k_destination)
else:
    print("\nFlickr8K directory already exists — skipping extraction.")

# Flickr30K
if not f30k_destination.exists():
    extract_zip(f30k_zip, f30k_destination)
else:
    print("\nFlickr30K directory already exists — skipping extraction.")

print("\n" + "-" * 75)
print("LOCAL DATASET STRUCTURE")
print("-" * 75)

for dataset_dir in [f8k_destination, f30k_destination]:

    print(f"\n{dataset_dir}")

    if dataset_dir.exists():

        # Show only a few directories/files
        items = list(dataset_dir.iterdir())[:15]

        for item in items:
            print("  ", item.name)

print("\n" + "=" * 75)
print("CELL 9B COMPLETE")
print("=" * 75)

CELL 9B — EXTRACTING FLICKR DATASETS

Extracting: flickr8k.zip
Destination: /content/Image_captioning_Local/Flickr8K
Archive members: 8,092
✓ Extraction complete

Extracting: flickr30k.zip
Destination: /content/Image_captioning_Local/Flickr30K
Archive members: 63,568
✓ Extraction complete

---------------------------------------------------------------------------
LOCAL DATASET STRUCTURE
---------------------------------------------------------------------------

/content/Image_captioning_Local/Flickr8K
   captions.txt
   Images

/content/Image_captioning_Local/Flickr30K
   flickr30k_images

CELL 9B COMPLETE


In [ ]:
# ============================================================
# CELL 9C — DISCOVER ACTUAL IMAGE LOCATIONS
# ============================================================

from pathlib import Path

print("=" * 75)
print("CELL 9C — DISCOVERING IMAGE LOCATIONS")
print("=" * 75)

# ------------------------------------------------------------
# Search for the exact sample filenames from RT-DETR
# ------------------------------------------------------------

sample_filenames = (
    detections["image_path"]
    .astype(str)
    .map(lambda x: Path(x).name)
    .drop_duplicates()
    .head(10)
    .tolist()
)

print("\nSample filenames to locate:")

for filename in sample_filenames:
    print(" ", filename)

# ------------------------------------------------------------
# Search recursively
# ------------------------------------------------------------

found = {}

for filename in sample_filenames:

    matches = list(
        LOCAL_ROOT.rglob(filename)
    )

    found[filename] = matches

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("SEARCH RESULTS")
print("-" * 75)

for filename, matches in found.items():

    print(f"\n{filename}")

    if not matches:
        print("  ❌ NOT FOUND")

    else:
        for match in matches[:5]:
            print("  ✓", match)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

found_count = sum(
    len(matches) > 0
    for matches in found.values()
)

print("\n" + "-" * 75)
print("SUMMARY")
print("-" * 75)

print(
    f"Files located: {found_count} / {len(sample_filenames)}"
)

if found_count == len(sample_filenames):
    print("\n✓ All sample images successfully located.")
else:
    print(
        "\n⚠ Some sample images were not found."
        "\nDo NOT continue to reconstruction yet."
    )

print("\n" + "=" * 75)
print("CELL 9C COMPLETE")
print("=" * 75)

CELL 9C — DISCOVERING IMAGE LOCATIONS

Sample filenames to locate:
  1000092795.jpg
  10002456.jpg
  1000268201.jpg
  1000344755.jpg
  1000366164.jpg
  1000523639.jpg
  1000919630.jpg
  10010052.jpg
  1001465944.jpg
  1001545525.jpg

---------------------------------------------------------------------------
SEARCH RESULTS
---------------------------------------------------------------------------

1000092795.jpg
  ✓ /content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/1000092795.jpg
  ✓ /content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/flickr30k_images/1000092795.jpg

10002456.jpg
  ✓ /content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/10002456.jpg
  ✓ /content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/flickr30k_images/10002456.jpg

1000268201.jpg
  ✓ /content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/1000268201.jpg
  ✓ /content/Image_captioning_Local/Flickr30K/fl

In [ ]:
# ============================================================
# CELL 10 — VERIFY IMAGE DIMENSIONS + SPATIAL FEATURES
# ============================================================

from pathlib import Path
from PIL import Image
import numpy as np

print("=" * 75)
print("CELL 10 — VERIFYING IMAGE DIMENSIONS AND SPATIAL FEATURES")
print("=" * 75)

# ------------------------------------------------------------
# Robust image resolver based on discovered local structure
# ------------------------------------------------------------

LOCAL_ROOT = Path("/content/Image_captioning_Local")

def resolve_image(stored_path):
    stored = str(stored_path).replace("\\", "/")
    filename = Path(stored).name

    # Flickr30K
    candidates = [
        LOCAL_ROOT / "Flickr30K" / "flickr30k_images" /
        "flickr30k_images" / filename,

        LOCAL_ROOT / "Flickr30K" / "flickr30k_images" /
        "flickr30k_images" / "flickr30k_images" / filename,

        # Flickr8K
        LOCAL_ROOT / "Flickr8K" / "Images" / filename,

        LOCAL_ROOT / "Flickr8K" / filename,
    ]

    # Exact stored path as final fallback
    candidates.append(Path(stored))

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    return None


# ------------------------------------------------------------
# Select one image with detections
# ------------------------------------------------------------

sample_row = detections.iloc[0]
sample_path = resolve_image(sample_row["image_path"])

if sample_path is None:
    raise FileNotFoundError(
        f"Could not resolve:\n{sample_row['image_path']}"
    )

print("\nSample image:")
print(sample_path)

# ------------------------------------------------------------
# Read dimensions
# ------------------------------------------------------------

with Image.open(sample_path) as img:
    width, height = img.size

print(f"\nImage dimensions: {width} × {height}")

if width <= 0 or height <= 0:
    raise ValueError("Invalid image dimensions.")


# ------------------------------------------------------------
# Get detections for this image
# ------------------------------------------------------------

sample_image_id = sample_row["image_id"]

sample_dets = detections[
    detections["image_id"] == sample_image_id
].copy()

print(f"Detections before threshold: {len(sample_dets)}")

# Confidence threshold from Notebook 10 configuration
CONF_THRESHOLD = 0.30

sample_dets = sample_dets[
    sample_dets["confidence"] >= CONF_THRESHOLD
].copy()

print(f"Detections after threshold: {len(sample_dets)}")


# ------------------------------------------------------------
# Raw bounding boxes
# ------------------------------------------------------------

boxes_raw = sample_dets[
    ["xmin", "ymin", "xmax", "ymax"]
].to_numpy(dtype=np.float32)

print("\nRaw boxes:")
print(boxes_raw[:5])


# ------------------------------------------------------------
# Clip boxes to actual image boundaries
# ------------------------------------------------------------

xmin = np.clip(boxes_raw[:, 0], 0, width)
ymin = np.clip(boxes_raw[:, 1], 0, height)
xmax = np.clip(boxes_raw[:, 2], 0, width)
ymax = np.clip(boxes_raw[:, 3], 0, height)

# Ensure valid box ordering
x1 = np.minimum(xmin, xmax)
x2 = np.maximum(xmin, xmax)
y1 = np.minimum(ymin, ymax)
y2 = np.maximum(ymin, ymax)


# ------------------------------------------------------------
# Normalize coordinates
# ------------------------------------------------------------

x1 = x1 / width
y1 = y1 / height
x2 = x2 / width
y2 = y2 / height

# ------------------------------------------------------------
# Derived spatial features
# ------------------------------------------------------------

center_x = (x1 + x2) / 2
center_y = (y1 + y2) / 2

box_width = x2 - x1
box_height = y2 - y1
box_area = box_width * box_height

spatial_features = np.stack(
    [
        x1,
        y1,
        x2,
        y2,
        center_x,
        center_y,
        box_width,
        box_height,
        box_area,
    ],
    axis=1
).astype(np.float32)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nSpatial feature shape:", spatial_features.shape)

print("\nSpatial features:")
print(spatial_features[:5])

print("\nValidation:")
print("  Finite:", np.isfinite(spatial_features).all())
print("  Minimum:", spatial_features.min())
print("  Maximum:", spatial_features.max())
print("  Dimensions:", spatial_features.shape[1])

assert spatial_features.shape[1] == 9
assert np.isfinite(spatial_features).all()
assert spatial_features.min() >= 0.0
assert spatial_features.max() <= 1.0

print("\n✓ Spatial feature construction is valid.")

print("\n" + "=" * 75)
print("CELL 10 COMPLETE")
print("=" * 75)

CELL 10 — VERIFYING IMAGE DIMENSIONS AND SPATIAL FEATURES

Sample image:
/content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/1000092795.jpg

Image dimensions: 333 × 500
Detections before threshold: 3
Detections after threshold: 3

Raw boxes:
[[158.85437 111.32701 258.74    368.6028 ]
 [159.39508 124.65212 214.00925 331.0891 ]
 [172.1624  125.14085 222.47795 332.65085]]

Spatial feature shape: (3, 9)

Spatial features:
[[0.47704014 0.22265401 0.77699697 0.7372056  0.6270186  0.4799298
  0.29995683 0.51455164 0.15434328]
 [0.4786639  0.24930425 0.6426704  0.6621782  0.56066716 0.45574123
  0.1640065  0.41287398 0.06771402]
 [0.5170042  0.2502817  0.66810197 0.6653017  0.5925531  0.4577917
  0.15109777 0.41502    0.06270859]]

Validation:
  Finite: True
  Minimum: 0.062708594
  Maximum: 0.77699697
  Dimensions: 9

✓ Spatial feature construction is valid.

CELL 10 COMPLETE


In [ ]:
# ============================================================
# CELL 11 — VERIFY TOP-K STRUCTURED OBJECT REPRESENTATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("CELL 11 — VERIFYING TOP-K OBJECT REPRESENTATION")
print("=" * 75)

TOP_K = 10
CONF_THRESHOLD = 0.30

# Use the already-loaded RT-DETR detections
df = detections.copy()

# ------------------------------------------------------------
# Confidence filtering
# ------------------------------------------------------------

df = df[df["confidence"] >= CONF_THRESHOLD].copy()

print(f"\nDetections after confidence threshold: {len(df):,}")
print(f"Images represented: {df['image_id'].nunique():,}")

# ------------------------------------------------------------
# Select one image
# ------------------------------------------------------------

test_image_id = df["image_id"].iloc[0]

image_dets = df[
    df["image_id"] == test_image_id
].copy()

# Sort highest confidence first
image_dets = image_dets.sort_values(
    "confidence",
    ascending=False
).head(TOP_K)

print(f"\nTest image ID: {test_image_id}")
print(f"Objects selected: {len(image_dets)}")

print("\nSelected objects:")

print(
    image_dets[
        ["object_name", "confidence",
         "xmin", "ymin", "xmax", "ymax"]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# Object IDs
# ------------------------------------------------------------

object_ids_test = np.array(
    [
        object_to_id.get(
            str(name),
            object_to_id["<unk>"]
        )
        for name in image_dets["object_name"]
    ],
    dtype=np.int64
)

# ------------------------------------------------------------
# Confidence features
# ------------------------------------------------------------

confidence_test = image_dets[
    "confidence"
].to_numpy(dtype=np.float32)

# ------------------------------------------------------------
# Resolve image
# ------------------------------------------------------------

test_path = resolve_image(
    image_dets.iloc[0]["image_path"]
)

if test_path is None:
    raise FileNotFoundError(
        "Unable to resolve test image."
    )

with Image.open(test_path) as img:
    width, height = img.size

# ------------------------------------------------------------
# Spatial features
# ------------------------------------------------------------

raw_boxes = image_dets[
    ["xmin", "ymin", "xmax", "ymax"]
].to_numpy(dtype=np.float32)

xmin = np.clip(raw_boxes[:, 0], 0, width)
ymin = np.clip(raw_boxes[:, 1], 0, height)
xmax = np.clip(raw_boxes[:, 2], 0, width)
ymax = np.clip(raw_boxes[:, 3], 0, height)

x1 = np.minimum(xmin, xmax) / width
y1 = np.minimum(ymin, ymax) / height
x2 = np.maximum(xmin, xmax) / width
y2 = np.maximum(ymin, ymax) / height

center_x = (x1 + x2) / 2
center_y = (y1 + y2) / 2

box_width = x2 - x1
box_height = y2 - y1
box_area = box_width * box_height

spatial_test = np.stack(
    [
        x1, y1, x2, y2,
        center_x, center_y,
        box_width, box_height,
        box_area
    ],
    axis=1
).astype(np.float32)

# ------------------------------------------------------------
# Object mask
# ------------------------------------------------------------

object_mask_test = np.ones(
    len(image_dets),
    dtype=np.bool_
)

# ------------------------------------------------------------
# Print shapes
# ------------------------------------------------------------

print("\nRepresentation shapes:")
print("  object_ids      :", object_ids_test.shape)
print("  spatial         :", spatial_test.shape)
print("  confidence      :", confidence_test.shape)
print("  object_mask     :", object_mask_test.shape)

print("\nObject IDs:")
print(object_ids_test)

print("\nConfidence:")
print(confidence_test)

print("\nSpatial:")
print(spatial_test)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert object_ids_test.ndim == 1
assert spatial_test.shape[1] == 9
assert confidence_test.ndim == 1
assert object_mask_test.ndim == 1

assert len(object_ids_test) == len(spatial_test)
assert len(object_ids_test) == len(confidence_test)
assert len(object_ids_test) == len(object_mask_test)

assert np.isfinite(spatial_test).all()
assert spatial_test.min() >= 0
assert spatial_test.max() <= 1

print("\n✓ Object + spatial + confidence representation is valid.")

print("\n" + "=" * 75)
print("CELL 11 COMPLETE")
print("=" * 75)

CELL 11 — VERIFYING TOP-K OBJECT REPRESENTATION

Detections after confidence threshold: 383,229
Images represented: 39,864

Test image ID: Flickr30K/1000092795.jpg
Objects selected: 3

Selected objects:
object_name  confidence       xmin       ymin       xmax       ymax
     person    0.936982 158.854370 111.327011 258.739990 368.602814
     person    0.849433 159.395081 124.652122 214.009247 331.089111
     person    0.341635 172.162399 125.140846 222.477951 332.650848

Representation shapes:
  object_ids      : (3,)
  spatial         : (3, 9)
  confidence      : (3,)
  object_mask     : (3,)

Object IDs:
[2 2 2]

Confidence:
[0.9369816  0.84943277 0.34163544]

Spatial:
[[0.47704014 0.22265401 0.77699697 0.7372056  0.6270186  0.4799298
  0.29995683 0.51455164 0.15434328]
 [0.4786639  0.24930425 0.6426704  0.6621782  0.56066716 0.45574123
  0.1640065  0.41287398 0.06771402]
 [0.5170042  0.2502817  0.66810197 0.6653017  0.5925531  0.4577917
  0.15109777 0.41502    0.06270859]]

✓ Object

In [ ]:
# ============================================================
# CELL 12 — CHECK FINAL FEATURE OUTPUT STATUS
# ============================================================

from pathlib import Path

print("=" * 75)
print("CELL 12 — CHECKING STRUCTURED FEATURE OUTPUTS")
print("=" * 75)

PROJECT_ROOT = Path("/content/drive/MyDrive/Image_captioning")

FEATURE_DIR = (
    PROJECT_ROOT
    / "Features"
    / "Structured_Object"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Outputs"
    / "Structured_Captioning"
)

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

expected_files = [
    FEATURE_DIR / "rtdetr_structured_object_features.npz",
    FEATURE_DIR / "caption_sequences.npz",
    OUTPUT_DIR / "object_vocabulary.json",
    OUTPUT_DIR / "caption_vocabulary.json",
    OUTPUT_DIR / "structured_captioning_config.json",
    OUTPUT_DIR / "image_object_metadata.csv",
    OUTPUT_DIR / "train_image_ids.csv",
    OUTPUT_DIR / "val_image_ids.csv",
    OUTPUT_DIR / "test_image_ids.csv",
]

print("\nExpected files:")

for path in expected_files:
    status = "FOUND" if path.exists() else "MISSING"
    print(f"  [{status:7s}] {path.name}")

print("\n" + "-" * 75)

missing = [
    path.name
    for path in expected_files
    if not path.exists()
]

if missing:
    print(f"Missing files: {len(missing)}")
    for name in missing:
        print("  -", name)
else:
    print("✓ All expected structured-captioning files exist.")

print("\n" + "=" * 75)
print("CELL 12 COMPLETE")
print("=" * 75)

CELL 12 — CHECKING STRUCTURED FEATURE OUTPUTS

Expected files:
  [MISSING] rtdetr_structured_object_features.npz
  [MISSING] caption_sequences.npz
  [FOUND  ] object_vocabulary.json
  [FOUND  ] caption_vocabulary.json
  [FOUND  ] structured_captioning_config.json
  [FOUND  ] image_object_metadata.csv
  [FOUND  ] train_image_ids.csv
  [FOUND  ] val_image_ids.csv
  [FOUND  ] test_image_ids.csv

---------------------------------------------------------------------------
Missing files: 2
  - rtdetr_structured_object_features.npz
  - caption_sequences.npz

CELL 12 COMPLETE


In [ ]:
# ============================================================
# CELL 13 — BUILD COMPLETE STRUCTURED OBJECT ARRAYS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

print("=" * 75)
print("CELL 13 — BUILDING COMPLETE STRUCTURED OBJECT ARRAYS")
print("=" * 75)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Image_captioning")

RTDETR_CSV = (
    PROJECT_ROOT
    / "Outputs"
    / "RTDETR_Detection"
    / "Final"
    / "rtdetr_x_detections_full.csv"
)

CAPTIONS_CSV = (
    PROJECT_ROOT
    / "Dataset"
    / "Processed"
    / "Captions"
    / "captions_processed.csv"
)

TRAIN_CSV = (
    PROJECT_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "train_image_ids.csv"
)

VAL_CSV = (
    PROJECT_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "val_image_ids.csv"
)

TEST_CSV = (
    PROJECT_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "test_image_ids.csv"
)

FEATURE_DIR = (
    PROJECT_ROOT
    / "Features"
    / "Structured_Object"
)

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT = Path("/content/Image_captioning_Local")

CONF_THRESHOLD = 0.30
TOP_K = 10

# ------------------------------------------------------------
# Load detection data
# ------------------------------------------------------------

print("\nLoading RT-DETR detections...")

detections = pd.read_csv(RTDETR_CSV)

print(f"Detection CSV rows: {len(detections):,}")

# ------------------------------------------------------------
# Load image IDs from captions
# This guarantees the complete 39,874-image universe.
# ------------------------------------------------------------

captions = pd.read_csv(CAPTIONS_CSV)

all_image_ids = (
    captions["image_id"]
    .drop_duplicates()
    .astype(str)
    .tolist()
)

print(f"Total dataset images: {len(all_image_ids):,}")

# ------------------------------------------------------------
# Filter detections
# ------------------------------------------------------------

detections["image_id"] = detections["image_id"].astype(str)

valid_detections = detections[
    detections["confidence"] >= CONF_THRESHOLD
].copy()

print(
    f"Detections after threshold {CONF_THRESHOLD}: "
    f"{len(valid_detections):,}"
)

# ------------------------------------------------------------
# Robust resolver
# ------------------------------------------------------------

def resolve_image(stored_path):
    stored = str(stored_path).replace("\\", "/")
    filename = Path(stored).name

    candidates = [
        LOCAL_ROOT / "Flickr30K" / "flickr30k_images" /
        "flickr30k_images" / filename,

        LOCAL_ROOT / "Flickr30K" / "flickr30k_images" /
        "flickr30k_images" / "flickr30k_images" / filename,

        LOCAL_ROOT / "Flickr8K" / "Images" / filename,

        LOCAL_ROOT / "Flickr8K" / filename,

        Path(stored)
    ]

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    return None


# ------------------------------------------------------------
# Preallocate exact target arrays
# ------------------------------------------------------------

N_IMAGES = len(all_image_ids)

object_ids = np.zeros(
    (N_IMAGES, TOP_K),
    dtype=np.int64
)

boxes = np.zeros(
    (N_IMAGES, TOP_K, 9),
    dtype=np.float32
)

confidences = np.zeros(
    (N_IMAGES, TOP_K),
    dtype=np.float32
)

object_mask = np.zeros(
    (N_IMAGES, TOP_K),
    dtype=np.bool_
)

# ------------------------------------------------------------
# Detection lookup
# ------------------------------------------------------------

grouped = {
    image_id: group
    for image_id, group
    in valid_detections.groupby("image_id", sort=False)
}

# ------------------------------------------------------------
# Build representation
# ------------------------------------------------------------

resolved_count = 0
missing_image_count = 0
images_with_detections = 0
images_with_zero_detections = 0
total_retained_detections = 0

for i, image_id in enumerate(
    tqdm(all_image_ids, desc="Building object features")
):

    group = grouped.get(image_id)

    if group is None or len(group) == 0:
        images_with_zero_detections += 1
        continue

    # Highest-confidence objects first
    group = (
        group
        .sort_values("confidence", ascending=False)
        .head(TOP_K)
        .copy()
    )

    # Resolve image
    stored_path = group.iloc[0]["image_path"]
    image_path = resolve_image(stored_path)

    if image_path is None:
        missing_image_count += 1
        continue

    try:
        with Image.open(image_path) as img:
            width, height = img.size
    except Exception:
        missing_image_count += 1
        continue

    if width <= 0 or height <= 0:
        missing_image_count += 1
        continue

    resolved_count += 1
    images_with_detections += 1

    n = len(group)
    total_retained_detections += n

    # --------------------------------------------------------
    # Object IDs
    # --------------------------------------------------------

    names = group["object_name"].astype(str).tolist()

    ids = [
        object_to_id.get(
            name,
            object_to_id["<unk>"]
        )
        for name in names
    ]

    object_ids[i, :n] = np.asarray(
        ids,
        dtype=np.int64
    )

    # --------------------------------------------------------
    # Confidence
    # --------------------------------------------------------

    confidences[i, :n] = group[
        "confidence"
    ].to_numpy(dtype=np.float32)

    # --------------------------------------------------------
    # Bounding boxes
    # --------------------------------------------------------

    raw = group[
        ["xmin", "ymin", "xmax", "ymax"]
    ].to_numpy(dtype=np.float32)

    xmin = np.clip(raw[:, 0], 0, width)
    ymin = np.clip(raw[:, 1], 0, height)
    xmax = np.clip(raw[:, 2], 0, width)
    ymax = np.clip(raw[:, 3], 0, height)

    x1 = np.minimum(xmin, xmax) / width
    y1 = np.minimum(ymin, ymax) / height
    x2 = np.maximum(xmin, xmax) / width
    y2 = np.maximum(ymin, ymax) / height

    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2

    box_width = x2 - x1
    box_height = y2 - y1
    box_area = box_width * box_height

    spatial = np.stack(
        [
            x1,
            y1,
            x2,
            y2,
            center_x,
            center_y,
            box_width,
            box_height,
            box_area
        ],
        axis=1
    ).astype(np.float32)

    boxes[i, :n, :] = spatial

    # --------------------------------------------------------
    # Mask
    # --------------------------------------------------------

    object_mask[i, :n] = True


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("STRUCTURED ARRAY SUMMARY")
print("-" * 75)

print(f"Images:                 {N_IMAGES:,}")
print(f"Resolved images:        {resolved_count:,}")
print(f"Missing/unreadable:     {missing_image_count:,}")
print(f"Images with detections: {images_with_detections:,}")
print(f"Zero detections:        {images_with_zero_detections:,}")
print(f"Retained object slots:  {total_retained_detections:,}")

print("\nArray shapes:")
print("  object_ids :", object_ids.shape)
print("  boxes      :", boxes.shape)
print("  confidences:", confidences.shape)
print("  object_mask:", object_mask.shape)

print("\n" + "=" * 75)
print("CELL 13 COMPLETE")
print("=" * 75)

CELL 13 — BUILDING COMPLETE STRUCTURED OBJECT ARRAYS

Loading RT-DETR detections...
Detection CSV rows: 496,803
Total dataset images: 39,874
Detections after threshold 0.3: 383,229


Building object features:   0%|          | 0/39874 [00:00<?, ?it/s]


---------------------------------------------------------------------------
STRUCTURED ARRAY SUMMARY
---------------------------------------------------------------------------
Images:                 39,874
Resolved images:        39,864
Missing/unreadable:     0
Images with detections: 39,864
Zero detections:        10
Retained object slots:  243,482

Array shapes:
  object_ids : (39874, 10)
  boxes      : (39874, 10, 9)
  confidences: (39874, 10)
  object_mask: (39874, 10)

CELL 13 COMPLETE


In [ ]:
# ============================================================
# CELL 14 — VALIDATE COMPLETE STRUCTURED ARRAYS
# ============================================================

print("=" * 75)
print("CELL 14 — VALIDATING COMPLETE STRUCTURED ARRAYS")
print("=" * 75)

# ------------------------------------------------------------
# Shape validation
# ------------------------------------------------------------

assert object_ids.shape == (39874, 10)
assert boxes.shape == (39874, 10, 9)
assert confidences.shape == (39874, 10)
assert object_mask.shape == (39874, 10)

# ------------------------------------------------------------
# Numerical validation
# ------------------------------------------------------------

assert np.isfinite(boxes).all()
assert np.isfinite(confidences).all()

assert boxes.min() >= 0.0
assert boxes.max() <= 1.0

assert confidences.min() >= 0.0
assert confidences.max() <= 1.0

# ------------------------------------------------------------
# Mask consistency
# ------------------------------------------------------------

valid_slots = object_mask

assert np.all(
    object_ids[~valid_slots] == 0
)

assert np.all(
    boxes[~valid_slots] == 0
)

assert np.all(
    confidences[~valid_slots] == 0
)

# ------------------------------------------------------------
# Object vocabulary validation
# ------------------------------------------------------------

valid_object_ids = object_ids[valid_slots]

assert valid_object_ids.min() >= 1
assert valid_object_ids.max() < len(object_to_id)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

objects_per_image = object_mask.sum(axis=1)

print("\nShape validation:")
print("  object_ids :", object_ids.shape)
print("  boxes      :", boxes.shape)
print("  confidences:", confidences.shape)
print("  object_mask:", object_mask.shape)

print("\nObject statistics:")
print(f"  Images:              {len(objects_per_image):,}")
print(f"  Zero-object images:  {(objects_per_image == 0).sum():,}")
print(f"  1-10 objects/image:")
print(f"    Mean:              {objects_per_image.mean():.4f}")
print(f"    Median:            {np.median(objects_per_image):.0f}")
print(f"    Maximum:           {objects_per_image.max():.0f}")

print("\nSpatial statistics:")
print(f"  Min:                 {boxes.min():.6f}")
print(f"  Max:                 {boxes.max():.6f}")

print("\n✓ All structured arrays passed validation.")

print("\n" + "=" * 75)
print("CELL 14 COMPLETE")
print("=" * 75)

CELL 14 — VALIDATING COMPLETE STRUCTURED ARRAYS

Shape validation:
  object_ids : (39874, 10)
  boxes      : (39874, 10, 9)
  confidences: (39874, 10)
  object_mask: (39874, 10)

Object statistics:
  Images:              39,874
  Zero-object images:  10
  1-10 objects/image:
    Mean:              6.1063
    Median:            6
    Maximum:           10

Spatial statistics:
  Min:                 0.000000
  Max:                 1.000000

✓ All structured arrays passed validation.

CELL 14 COMPLETE


In [ ]:
# ============================================================
# CELL 15 — SAVE STRUCTURED OBJECT FEATURES
# ============================================================

print("=" * 75)
print("CELL 15 — SAVING STRUCTURED OBJECT FEATURES")
print("=" * 75)

OBJECT_FEATURE_FILE = (
    FEATURE_DIR
    / "rtdetr_structured_object_features.npz"
)

# ------------------------------------------------------------
# Convert image IDs to fixed string array
# ------------------------------------------------------------

image_ids_array = np.asarray(
    all_image_ids,
    dtype=str
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

np.savez_compressed(
    OBJECT_FEATURE_FILE,
    image_ids=image_ids_array,
    object_ids=object_ids,
    boxes=boxes,
    confidences=confidences,
    object_mask=object_mask
)

# ------------------------------------------------------------
# Verify file
# ------------------------------------------------------------

print("\nSaved:")
print(OBJECT_FEATURE_FILE)

print(
    f"File size: "
    f"{OBJECT_FEATURE_FILE.stat().st_size / (1024**2):.2f} MB"
)

# Reopen immediately
check = np.load(
    OBJECT_FEATURE_FILE,
    allow_pickle=False
)

print("\nSaved keys:")
for key in check.files:
    print(f"  {key:15s}", check[key].shape)

# ------------------------------------------------------------
# Exact verification
# ------------------------------------------------------------

assert check["image_ids"].shape == (39874,)
assert check["object_ids"].shape == (39874, 10)
assert check["boxes"].shape == (39874, 10, 9)
assert check["confidences"].shape == (39874, 10)
assert check["object_mask"].shape == (39874, 10)

print("\n✓ NPZ saved and successfully reopened.")
print("✓ All expected arrays and shapes are correct.")

print("\n" + "=" * 75)
print("CELL 15 COMPLETE")
print("=" * 75)

CELL 15 — SAVING STRUCTURED OBJECT FEATURES

Saved:
/content/drive/MyDrive/Image_captioning/Features/Structured_Object/rtdetr_structured_object_features.npz
File size: 8.98 MB

Saved keys:
  image_ids       (39874,)
  object_ids      (39874, 10)
  boxes           (39874, 10, 9)
  confidences     (39874, 10)
  object_mask     (39874, 10)

✓ NPZ saved and successfully reopened.
✓ All expected arrays and shapes are correct.

CELL 15 COMPLETE


In [ ]:
# ============================================================
# CELL 16 — VERIFY CAPTION SEQUENCE SOURCE
# ============================================================

print("=" * 75)
print("CELL 16 — VERIFYING CAPTION SEQUENCE SOURCE")
print("=" * 75)

# ------------------------------------------------------------
# Load caption vocabulary
# ------------------------------------------------------------

print("\nCaption vocabulary:")

print(f"  Vocabulary size: {len(caption_to_id):,}")
print(f"  <pad>: {caption_to_id.get('<pad>')}")
print(f"  <start>: {caption_to_id.get('<start>')}")
print(f"  <end>: {caption_to_id.get('<end>')}")
print(f"  <unk>: {caption_to_id.get('<unk>')}")

# ------------------------------------------------------------
# Caption data
# ------------------------------------------------------------

print("\nCaption dataset:")
print(f"  Rows: {len(captions):,}")
print(f"  Images: {captions['image_id'].nunique():,}")

assert len(captions) == 199369
assert captions["image_id"].nunique() == 39874

assert "processed_caption" in captions.columns

# ------------------------------------------------------------
# Inspect examples
# ------------------------------------------------------------

print("\nSample processed captions:")

for caption in captions["processed_caption"].head(5):
    print(" ", caption)

# ------------------------------------------------------------
# Length statistics
# ------------------------------------------------------------

caption_lengths = (
    captions["processed_caption"]
    .astype(str)
    .str.split()
    .str.len()
)

print("\nCaption length statistics:")
print(f"  Minimum: {caption_lengths.min()}")
print(f"  Maximum: {caption_lengths.max()}")
print(f"  Mean:    {caption_lengths.mean():.2f}")
print(f"  Median:  {caption_lengths.median():.0f}")

MAX_CAPTION_LENGTH = 32

print(
    f"\nConfigured maximum sequence length: "
    f"{MAX_CAPTION_LENGTH}"
)

print("\n✓ Caption source is ready.")

print("\n" + "=" * 75)
print("CELL 16 COMPLETE")
print("=" * 75)

CELL 16 — VERIFYING CAPTION SEQUENCE SOURCE

Caption vocabulary:
  Vocabulary size: 12,174
  <pad>: 0
  <start>: 1
  <end>: 2
  <unk>: 3

Caption dataset:
  Rows: 199,369
  Images: 39,874

Sample processed captions:
  a child in a pink dress is climbing up a set of stairs in an entry way.
  a girl going into a wooden building.
  a little girl climbing into a wooden playhouse.
  a little girl climbing the stairs to her playhouse.
  a little girl in a pink dress going into a wooden cabin.

Caption length statistics:
  Minimum: 1
  Maximum: 78
  Mean:    11.98
  Median:  11

Configured maximum sequence length: 32

✓ Caption source is ready.

CELL 16 COMPLETE


In [ ]:
# ============================================================
# CELL 17 — CONSTRUCT FIXED-LENGTH CAPTION SEQUENCES
# ============================================================

import numpy as np
from pathlib import Path

print("=" * 75)
print("CELL 17 — CONSTRUCTING CAPTION SEQUENCES")
print("=" * 75)

MAX_CAPTION_LENGTH = 32

PAD_ID = caption_to_id["<pad>"]
START_ID = caption_to_id["<start>"]
END_ID = caption_to_id["<end>"]
UNK_ID = caption_to_id["<unk>"]

print("\nSpecial token IDs:")
print(f"  PAD   = {PAD_ID}")
print(f"  START = {START_ID}")
print(f"  END   = {END_ID}")
print(f"  UNK   = {UNK_ID}")

# ------------------------------------------------------------
# Allocate fixed-size array
# ------------------------------------------------------------

caption_sequences = np.full(
    (len(captions), MAX_CAPTION_LENGTH),
    PAD_ID,
    dtype=np.int64
)

# ------------------------------------------------------------
# Convert captions to token IDs
# ------------------------------------------------------------

truncated_count = 0
unknown_token_count = 0
total_token_count = 0

for i, text in enumerate(
    captions["processed_caption"].astype(str)
):

    words = text.split()

    # Leave room for START and END
    max_words = MAX_CAPTION_LENGTH - 2

    if len(words) > max_words:
        truncated_count += 1
        words = words[:max_words]

    sequence = [START_ID]

    for word in words:
        token_id = caption_to_id.get(
            word,
            UNK_ID
        )

        if token_id == UNK_ID:
            unknown_token_count += 1

        sequence.append(token_id)

    sequence.append(END_ID)

    total_token_count += len(sequence)

    caption_sequences[
        i,
        :len(sequence)
    ] = sequence

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nCaption sequence shape:")
print(" ", caption_sequences.shape)

print("\nExpected shape:")
print(
    f"  ({len(captions)}, {MAX_CAPTION_LENGTH})"
)

print("\nStatistics:")
print(f"  Captions:              {len(captions):,}")
print(f"  Truncated captions:    {truncated_count:,}")
print(f"  Unknown token count:   {unknown_token_count:,}")
print(f"  Total encoded tokens:  {total_token_count:,}")

# ------------------------------------------------------------
# Inspect first captions
# ------------------------------------------------------------

print("\nFirst 3 encoded captions:")

for i in range(3):

    print(f"\nCaption {i + 1}:")
    print("Text:", captions.iloc[i]["processed_caption"])
    print("IDs :", caption_sequences[i].tolist())

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert caption_sequences.shape == (
    len(captions),
    MAX_CAPTION_LENGTH
)

assert caption_sequences.dtype == np.int64

assert np.all(
    caption_sequences[:, 0] == START_ID
)

assert np.all(
    caption_sequences >= 0
)

assert caption_sequences.max() < len(caption_to_id)

print("\n✓ Caption sequences successfully constructed.")

print("\n" + "=" * 75)
print("CELL 17 COMPLETE")
print("=" * 75)

CELL 17 — CONSTRUCTING CAPTION SEQUENCES

Special token IDs:
  PAD   = 0
  START = 1
  END   = 2
  UNK   = 3

Caption sequence shape:
  (199369, 32)

Expected shape:
  (199369, 32)

Statistics:
  Captions:              199,369
  Truncated captions:    1,310
  Unknown token count:   234,767
  Total encoded tokens:  2,778,735

First 3 encoded captions:

Caption 1:
Text: a child in a pink dress is climbing up a set of stairs in an entry way.
IDs : [1, 4, 48, 5, 4, 89, 121, 9, 199, 51, 4, 363, 11, 395, 5, 20, 4813, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Caption 2:
Text: a girl going into a wooden building.
IDs : [1, 4, 30, 354, 67, 4, 222, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Caption 3:
Text: a little girl climbing into a wooden playhouse.
IDs : [1, 4, 50, 30, 199, 67, 4, 222, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

✓ Caption sequences successfully constructed.

CELL 17 COMPLETE


In [ ]:
# ============================================================
# CELL 18 — VALIDATE CAPTION SEQUENCE STRUCTURE
# ============================================================

print("=" * 75)
print("CELL 18 — VALIDATING CAPTION SEQUENCES")
print("=" * 75)

# ------------------------------------------------------------
# Find END and PAD positions
# ------------------------------------------------------------

end_positions = []
padding_counts = []

invalid_sequences = 0

for seq in caption_sequences:

    end_matches = np.where(
        seq == END_ID
    )[0]

    if len(end_matches) == 0:
        invalid_sequences += 1
        continue

    end_pos = end_matches[0]

    # Everything after END should be PAD
    if not np.all(
        seq[end_pos + 1:] == PAD_ID
    ):
        invalid_sequences += 1

    end_positions.append(end_pos)

    padding_counts.append(
        np.sum(seq == PAD_ID)
    )

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

end_positions = np.asarray(
    end_positions,
    dtype=np.int64
)

padding_counts = np.asarray(
    padding_counts,
    dtype=np.int64
)

print("\nSequence validation:")
print(f"  Total sequences:       {len(caption_sequences):,}")
print(f"  Valid END tokens:      {len(end_positions):,}")
print(f"  Invalid sequences:     {invalid_sequences:,}")

print("\nEND token position:")
print(f"  Minimum: {end_positions.min()}")
print(f"  Maximum: {end_positions.max()}")
print(f"  Mean:    {end_positions.mean():.2f}")

print("\nPadding:")
print(f"  Minimum PAD count: {padding_counts.min()}")
print(f"  Maximum PAD count: {padding_counts.max()}")
print(f"  Mean PAD count:    {padding_counts.mean():.2f}")

# ------------------------------------------------------------
# Validate vocabulary range
# ------------------------------------------------------------

assert invalid_sequences == 0
assert end_positions.min() >= 1
assert end_positions.max() < MAX_CAPTION_LENGTH

assert caption_sequences.min() >= 0
assert caption_sequences.max() < len(caption_to_id)

# START must occur exactly once at beginning
assert np.all(
    caption_sequences[:, 0] == START_ID
)

print("\n✓ Caption sequence structure is valid.")

print("\n" + "=" * 75)
print("CELL 18 COMPLETE")
print("=" * 75)

CELL 18 — VALIDATING CAPTION SEQUENCES

Sequence validation:
  Total sequences:       199,369
  Valid END tokens:      199,369
  Invalid sequences:     0

END token position:
  Minimum: 2
  Maximum: 31
  Mean:    12.94

Padding:
  Minimum PAD count: 0
  Maximum PAD count: 29
  Mean PAD count:    18.06

✓ Caption sequence structure is valid.

CELL 18 COMPLETE


In [ ]:
# ============================================================
# CELL 19A — INSPECT SAVED CAPTION NPZ
# ============================================================

print("=" * 75)
print("CELL 19A — INSPECTING SAVED CAPTION NPZ")
print("=" * 75)

CAPTION_FEATURE_FILE = (
    FEATURE_DIR / "caption_sequences.npz"
)

print("\nFile:")
print(CAPTION_FEATURE_FILE)

print(
    f"Size: "
    f"{CAPTION_FEATURE_FILE.stat().st_size / (1024**2):.2f} MB"
)

# ------------------------------------------------------------
# Load with pickle enabled ONLY for inspection
# ------------------------------------------------------------

check = np.load(
    CAPTION_FEATURE_FILE,
    allow_pickle=True
)

print("\nSaved arrays:")

for key in check.files:
    arr = check[key]

    print(
        f"  {key:15s}"
        f" shape={arr.shape}"
        f" dtype={arr.dtype}"
    )

# ------------------------------------------------------------
# Check caption IDs specifically
# ------------------------------------------------------------

caption_check = check["caption_ids"]

print("\nCaption IDs:")
print("  Shape:", caption_check.shape)
print("  Dtype:", caption_check.dtype)
print("  Min:", caption_check.min())
print("  Max:", caption_check.max())

assert caption_check.shape == (199369, 32)
assert caption_check.min() >= 0
assert caption_check.max() < len(caption_to_id)

print("\n✓ Caption ID array is intact.")

print("\n" + "=" * 75)
print("CELL 19A COMPLETE")
print("=" * 75)

CELL 19A — INSPECTING SAVED CAPTION NPZ

File:
/content/drive/MyDrive/Image_captioning/Features/Structured_Object/caption_sequences.npz
Size: 4.21 MB

Saved arrays:
  image_ids       shape=(199369,) dtype=object
  caption_ids     shape=(199369, 32) dtype=int64

Caption IDs:
  Shape: (199369, 32)
  Dtype: int64
  Min: 0
  Max: 12173

✓ Caption ID array is intact.

CELL 19A COMPLETE


In [ ]:
# ============================================================
# CELL 19B — FIX AND RESAVE CAPTION NPZ
# ============================================================

print("=" * 75)
print("CELL 19B — FIXING CAPTION NPZ STORAGE")
print("=" * 75)

CAPTION_FEATURE_FILE = (
    FEATURE_DIR / "caption_sequences.npz"
)

# ------------------------------------------------------------
# Create explicit Unicode image-ID array
# ------------------------------------------------------------

image_id_strings = (
    captions["image_id"]
    .astype(str)
    .tolist()
)

max_image_id_length = max(
    len(x) for x in image_id_strings
)

caption_image_ids_fixed = np.asarray(
    image_id_strings,
    dtype=f"<U{max_image_id_length}"
)

print("\nImage ID array:")
print("  Shape:", caption_image_ids_fixed.shape)
print("  Dtype:", caption_image_ids_fixed.dtype)
print("  Max string length:", max_image_id_length)

# ------------------------------------------------------------
# Explicitly convert caption IDs
# ------------------------------------------------------------

caption_sequences_fixed = np.asarray(
    caption_sequences,
    dtype=np.int64
)

print("\nCaption sequence array:")
print("  Shape:", caption_sequences_fixed.shape)
print("  Dtype:", caption_sequences_fixed.dtype)

# ------------------------------------------------------------
# Overwrite NPZ
# ------------------------------------------------------------

np.savez_compressed(
    CAPTION_FEATURE_FILE,
    image_ids=caption_image_ids_fixed,
    caption_ids=caption_sequences_fixed
)

print("\n✓ NPZ rewritten successfully.")

print(
    f"New file size: "
    f"{CAPTION_FEATURE_FILE.stat().st_size / (1024**2):.2f} MB"
)

print("\n" + "=" * 75)
print("CELL 19B COMPLETE")
print("=" * 75)

CELL 19B — FIXING CAPTION NPZ STORAGE

Image ID array:
  Shape: (199369,)
  Dtype: <U34
  Max string length: 34

Caption sequence array:
  Shape: (199369, 32)
  Dtype: int64

✓ NPZ rewritten successfully.
New file size: 4.30 MB

CELL 19B COMPLETE


In [ ]:
# ============================================================
# CELL 19C — FINAL CAPTION NPZ VERIFICATION
# ============================================================

print("=" * 75)
print("CELL 19C — FINAL CAPTION NPZ VERIFICATION")
print("=" * 75)

CAPTION_FEATURE_FILE = (
    FEATURE_DIR / "caption_sequences.npz"
)

# ------------------------------------------------------------
# IMPORTANT:
# This must work WITHOUT pickle.
# ------------------------------------------------------------

check = np.load(
    CAPTION_FEATURE_FILE,
    allow_pickle=False
)

print("\nSaved keys:")

for key in check.files:

    arr = check[key]

    print(
        f"  {key:15s}"
        f" shape={arr.shape}"
        f" dtype={arr.dtype}"
    )

# ------------------------------------------------------------
# Extract
# ------------------------------------------------------------

saved_image_ids = check["image_ids"]
saved_caption_ids = check["caption_ids"]

# ------------------------------------------------------------
# Exact shape checks
# ------------------------------------------------------------

assert saved_image_ids.shape == (199369,)
assert saved_caption_ids.shape == (199369, 32)

# ------------------------------------------------------------
# Dtype checks
# ------------------------------------------------------------

assert saved_image_ids.dtype.kind == "U"
assert saved_caption_ids.dtype == np.int64

# ------------------------------------------------------------
# Value checks
# ------------------------------------------------------------

assert np.array_equal(
    saved_caption_ids,
    caption_sequences
)

assert np.array_equal(
    saved_image_ids.astype(str),
    captions["image_id"].astype(str).to_numpy()
)

assert saved_caption_ids.min() >= 0
assert saved_caption_ids.max() < len(caption_to_id)

# ------------------------------------------------------------
# Special-token checks
# ------------------------------------------------------------

assert np.all(
    saved_caption_ids[:, 0] == START_ID
)

print("\nVerification:")
print("  image_ids shape :", saved_image_ids.shape)
print("  caption_ids shape:", saved_caption_ids.shape)
print("  image_ids dtype :", saved_image_ids.dtype)
print("  caption_ids dtype:", saved_caption_ids.dtype)

print("\n✓ NPZ loads successfully with allow_pickle=False.")
print("✓ Caption IDs match the original sequences exactly.")
print("✓ Image IDs match the source captions exactly.")
print("✓ Caption NPZ is ready for model training.")

print("\n" + "=" * 75)
print("CELL 19C COMPLETE")
print("=" * 75)

CELL 19C — FINAL CAPTION NPZ VERIFICATION

Saved keys:
  image_ids       shape=(199369,) dtype=<U34
  caption_ids     shape=(199369, 32) dtype=int64

Verification:
  image_ids shape : (199369,)
  caption_ids shape: (199369, 32)
  image_ids dtype : <U34
  caption_ids dtype: int64

✓ NPZ loads successfully with allow_pickle=False.
✓ Caption IDs match the original sequences exactly.
✓ Image IDs match the source captions exactly.
✓ Caption NPZ is ready for model training.

CELL 19C COMPLETE


In [ ]:
# ============================================================
# CELL 20A — IDENTIFY IMAGES WITH FEWER THAN 5 CAPTIONS
# ============================================================

print("=" * 75)
print("CELL 20A — IDENTIFYING INCOMPLETE CAPTION GROUPS")
print("=" * 75)

caption_counts_series = (
    captions
    .groupby("image_id")
    .size()
    .sort_values()
)

incomplete = caption_counts_series[
    caption_counts_series < 5
]

print("\nCaption count distribution:")
print(
    caption_counts_series.value_counts()
    .sort_index()
    .to_string()
)

print("\n" + "-" * 75)

print(
    f"Images with fewer than 5 captions: "
    f"{len(incomplete):,}"
)

if len(incomplete) > 0:

    print("\nIncomplete images:")

    for image_id, count in incomplete.items():
        print(
            f"  {image_id}: {count} captions"
        )

print("\n" + "=" * 75)
print("CELL 20A COMPLETE")
print("=" * 75)

CELL 20A — IDENTIFYING INCOMPLETE CAPTION GROUPS

Caption count distribution:
4        1
5    39873

---------------------------------------------------------------------------
Images with fewer than 5 captions: 1

Incomplete images:
  Flickr30K/2199200615.jpg: 4 captions

CELL 20A COMPLETE


In [ ]:
# ============================================================
# CELL 20B — INSPECT INCOMPLETE IMAGE CAPTIONS
# ============================================================

print("=" * 75)
print("CELL 20B — INSPECTING INCOMPLETE CAPTION GROUPS")
print("=" * 75)

incomplete_ids = incomplete.index.astype(str).tolist()

if len(incomplete_ids) == 0:

    print("\n✓ No incomplete caption groups found.")

else:

    for image_id in incomplete_ids:

        print("\n" + "-" * 75)
        print("IMAGE:", image_id)
        print("-" * 75)

        rows = captions[
            captions["image_id"].astype(str) == image_id
        ]

        print(
            f"Number of captions: {len(rows)}"
        )

        for j, (_, row) in enumerate(rows.iterrows(), 1):

            print(
                f"\nCaption {j}:"
            )

            print(
                "  Raw:       ",
                row.get("caption", "")
            )

            print(
                "  Processed: ",
                row.get("processed_caption", "")
            )

print("\n" + "=" * 75)
print("CELL 20B COMPLETE")
print("=" * 75)

CELL 20B — INSPECTING INCOMPLETE CAPTION GROUPS

---------------------------------------------------------------------------
IMAGE: Flickr30K/2199200615.jpg
---------------------------------------------------------------------------
Number of captions: 4

Caption 1:
  Raw:        A dog running on green grass with its mouth open and tongue hanging out .
  Processed:  a dog running on green grass with its mouth open and tongue hanging out.

Caption 2:
  Raw:        a white dog is running with its mouth open across the grass .
  Processed:  a white dog is running with its mouth open across the grass.

Caption 3:
  Raw:        A white , black , and brown dog runs in a field .
  Processed:  a white, black, and brown dog runs in a field.

Caption 4:
  Raw:        A dog runs across the grassy field .
  Processed:  a dog runs across the grassy field.

CELL 20B COMPLETE


In [ ]:
# ============================================================
# CELL 20C — CHECK DATASET AND SOURCE LABELS
# ============================================================

print("=" * 75)
print("CELL 20C — CHECKING DATASET SOURCE INFORMATION")
print("=" * 75)

if len(incomplete_ids) > 0:

    incomplete_rows = captions[
        captions["image_id"].astype(str).isin(incomplete_ids)
    ].copy()

    print("\nDataset labels:")
    print(
        incomplete_rows[
            ["image_id", "dataset", "image_name"]
        ].drop_duplicates()
        .to_string(index=False)
    )

    print("\nCaption row counts by dataset:")

    print(
        incomplete_rows
        .groupby("dataset")
        .size()
        .to_string()
    )

    print("\nAll rows for incomplete images:")

    print(
        incomplete_rows[
            [
                "image_id",
                "dataset",
                "image_name",
                "caption"
            ]
        ].to_string(index=False)
    )

else:

    print("\nNo incomplete images to inspect.")

print("\n" + "=" * 75)
print("CELL 20C COMPLETE")
print("=" * 75)

CELL 20C — CHECKING DATASET SOURCE INFORMATION

Dataset labels:
                image_id   dataset     image_name
Flickr30K/2199200615.jpg Flickr30K 2199200615.jpg

Caption row counts by dataset:
dataset
Flickr30K    4

All rows for incomplete images:
                image_id   dataset     image_name                                                                   caption
Flickr30K/2199200615.jpg Flickr30K 2199200615.jpg A dog running on green grass with its mouth open and tongue hanging out .
Flickr30K/2199200615.jpg Flickr30K 2199200615.jpg             a white dog is running with its mouth open across the grass .
Flickr30K/2199200615.jpg Flickr30K 2199200615.jpg                         A white , black , and brown dog runs in a field .
Flickr30K/2199200615.jpg Flickr30K 2199200615.jpg                                      A dog runs across the grassy field .

CELL 20C COMPLETE


In [ ]:
# ============================================================
# CELL 20D — VERIFY OBJECT FEATURES FOR INCOMPLETE IMAGES
# ============================================================

print("=" * 75)
print("CELL 20D — VERIFYING OBJECT REPRESENTATIONS")
print("=" * 75)

object_row_lookup = {
    image_id: i
    for i, image_id in enumerate(object_image_ids)
}

if len(incomplete_ids) > 0:

    for image_id in incomplete_ids:

        print("\n" + "-" * 75)
        print("IMAGE:", image_id)
        print("-" * 75)

        if image_id not in object_row_lookup:

            print("❌ OBJECT REPRESENTATION NOT FOUND")

            continue

        row = object_row_lookup[image_id]

        n_objects = int(
            mask_check[row].sum()
        )

        print(
            f"Object feature row: {row}"
        )

        print(
            f"Valid objects: {n_objects}"
        )

        print(
            "Object IDs:",
            object_ids_check[row].tolist()
        )

        print(
            "Confidence:",
            confidence_check[row].tolist()
        )

        assert image_id in unique_object_ids

        print("✓ Object representation exists.")

else:

    print("\nNo incomplete images to inspect.")

print("\n" + "=" * 75)
print("CELL 20D COMPLETE")
print("=" * 75)

CELL 20D — VERIFYING OBJECT REPRESENTATIONS

---------------------------------------------------------------------------
IMAGE: Flickr30K/2199200615.jpg
---------------------------------------------------------------------------
Object feature row: 12090
Valid objects: 1
Object IDs: [6, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Confidence: [0.9505056738853455, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
✓ Object representation exists.

CELL 20D COMPLETE


In [ ]:
# ============================================================
# CELL 21 — FINAL IMAGE / CAPTION / OBJECT ALIGNMENT CHECK
# ============================================================

print("=" * 75)
print("CELL 21 — FINAL ALIGNMENT CHECK")
print("=" * 75)

# ------------------------------------------------------------
# Caption counts
# ------------------------------------------------------------

caption_count_series = (
    captions
    .groupby("image_id")
    .size()
)

print("\nCaption distribution:")
print(
    caption_count_series
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# Expected dataset
# ------------------------------------------------------------

assert len(caption_count_series) == 39874
assert caption_count_series.min() == 4
assert caption_count_series.max() == 5

# Total captions
assert caption_count_series.sum() == 199369

# ------------------------------------------------------------
# Object/caption image IDs
# ------------------------------------------------------------

object_set = set(
    object_image_ids.astype(str)
)

caption_set = set(
    captions["image_id"].astype(str)
)

print("\nImage coverage:")
print(f"  Object features: {len(object_set):,}")
print(f"  Caption data:    {len(caption_set):,}")

assert object_set == caption_set

# ------------------------------------------------------------
# Verify every caption has an object representation
# ------------------------------------------------------------

missing_object_images = (
    caption_set - object_set
)

missing_caption_images = (
    object_set - caption_set
)

print(
    f"\nImages missing object features: "
    f"{len(missing_object_images)}"
)

print(
    f"Images missing captions: "
    f"{len(missing_caption_images)}"
)

assert len(missing_object_images) == 0
assert len(missing_caption_images) == 0

# ------------------------------------------------------------
# Verify caption image IDs correspond to saved NPZ
# ------------------------------------------------------------

saved_caption_image_ids = (
    caption_data["image_ids"].astype(str)
)

assert np.array_equal(
    saved_caption_image_ids,
    captions["image_id"].astype(str).to_numpy()
)

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\n✓ All 39,874 images have object representations.")
print("✓ All caption rows have valid image IDs.")
print("✓ No image is missing from either feature source.")
print("✓ Dataset contains 4–5 captions per image.")
print("✓ Total caption count = 199,369.")

print("\n" + "=" * 75)
print("CELL 21 COMPLETE")
print("=" * 75)


CELL 21 — FINAL ALIGNMENT CHECK

Caption distribution:
4        1
5    39873

Image coverage:
  Object features: 39,874
  Caption data:    39,874

Images missing object features: 0
Images missing captions: 0

✓ All 39,874 images have object representations.
✓ All caption rows have valid image IDs.
✓ No image is missing from either feature source.
✓ Dataset contains 4–5 captions per image.
✓ Total caption count = 199,369.

CELL 21 COMPLETE


In [ ]:
# ============================================================
# CELL 22 — VERIFY TRAIN / VALIDATION / TEST SPLITS
# ============================================================

print("=" * 75)
print("CELL 22 — VERIFYING DATASET SPLITS")
print("=" * 75)

SPLIT_DIR = (
    PROJECT_ROOT
    / "Outputs"
    / "Structured_Captioning"
)

train_ids = set(
    pd.read_csv(
        SPLIT_DIR / "train_image_ids.csv"
    )["image_id"].astype(str)
)

val_ids = set(
    pd.read_csv(
        SPLIT_DIR / "val_image_ids.csv"
    )["image_id"].astype(str)
)

test_ids = set(
    pd.read_csv(
        SPLIT_DIR / "test_image_ids.csv"
    )["image_id"].astype(str)
)

print("\nSplit sizes:")
print(f"  Train:      {len(train_ids):,}")
print(f"  Validation: {len(val_ids):,}")
print(f"  Test:       {len(test_ids):,}")

# ------------------------------------------------------------
# Check expected sizes
# ------------------------------------------------------------

assert len(train_ids) == 31899
assert len(val_ids) == 3987
assert len(test_ids) == 3988

# ------------------------------------------------------------
# Check overlaps
# ------------------------------------------------------------

train_val = train_ids & val_ids
train_test = train_ids & test_ids
val_test = val_ids & test_ids

print("\nOverlap:")
print(f"  Train ∩ Val : {len(train_val)}")
print(f"  Train ∩ Test: {len(train_test)}")
print(f"  Val ∩ Test  : {len(val_test)}")

assert len(train_val) == 0
assert len(train_test) == 0
assert len(val_test) == 0

# ------------------------------------------------------------
# Check complete coverage
# ------------------------------------------------------------

combined = train_ids | val_ids | test_ids

print(
    f"\nCombined unique images: "
    f"{len(combined):,}"
)

assert len(combined) == 39874
assert combined == object_set

print("\n✓ Splits are mutually exclusive.")
print("✓ Splits cover all 39,874 images.")
print("✓ Every split image has structured object features.")

print("\n" + "=" * 75)
print("CELL 22 COMPLETE")
print("=" * 75)

CELL 22 — VERIFYING DATASET SPLITS

Split sizes:
  Train:      31,899
  Validation: 3,987
  Test:       3,988

Overlap:
  Train ∩ Val : 0
  Train ∩ Test: 0
  Val ∩ Test  : 0

Combined unique images: 39,874

✓ Splits are mutually exclusive.
✓ Splits cover all 39,874 images.
✓ Every split image has structured object features.

CELL 22 COMPLETE


In [ ]:
# ============================================================
# CELL 23 — FINAL PREPROCESSING REPORT
# ============================================================

print("=" * 75)
print("CELL 23 — FINAL PREPROCESSING REPORT")
print("=" * 75)

print("""
=============================================================
                 OBJECT-AWARE CAPTIONING
                  PREPROCESSING REPORT
=============================================================

DATASET
-------
Datasets:
  Flickr8K + Flickr30K

Unique images:
  39,874

Total captions:
  199,369

Captions per image:
  Minimum: 4
  Maximum: 5
  Mean:    approximately 5.00


RT-DETR-X DETECTION REPRESENTATION
-----------------------------------
Detector:
  RT-DETR-X

Confidence threshold:
  0.30

Maximum objects per image:
  10

Images represented:
  39,864

Zero-object images:
  10

Retained object instances:
  243,482


OBJECT FEATURES
---------------
Object ID tensor:
  (39,874, 10)

Spatial tensor:
  (39,874, 10, 9)

Confidence tensor:
  (39,874, 10)

Object mask:
  (39,874, 10)


SPATIAL FEATURES
----------------
1. xmin
2. ymin
3. xmax
4. ymax
5. center_x
6. center_y
7. box_width
8. box_height
9. box_area


CAPTION FEATURES
----------------
Caption tensor:
  (199,369, 32)

Caption vocabulary:
  12,174

Special tokens:
  <pad>   = 0
  <start> = 1
  <end>   = 2
  <unk>   = 3

Truncated captions:
  1,310


DATASET SPLITS
--------------
Train:
  31,899 images

Validation:
  3,987 images

Test:
  3,988 images


PERMANENT FEATURE FILES
-----------------------
✓ rtdetr_structured_object_features.npz
✓ caption_sequences.npz
✓ object_vocabulary.json
✓ caption_vocabulary.json
✓ structured_captioning_config.json
✓ image_object_metadata.csv
✓ train_image_ids.csv
✓ val_image_ids.csv
✓ test_image_ids.csv


FINAL PIPELINE
--------------
Image
  ↓
RT-DETR-X object detection
  ↓
Confidence filtering (≥ 0.30)
  ↓
Top-K object selection (K = 10)
  ↓
Object ID
  +
9-D normalized spatial representation
  +
Detection confidence
  ↓
Structured object tokens
  ↓
Object-Aware Transformer
  ↓
Generated caption
  ↓
BLEU / METEOR / ROUGE-L / CIDEr / SPICE
=============================================================
""")

print("✓ PREPROCESSING PIPELINE READY")
print("=" * 75)

CELL 23 — FINAL PREPROCESSING REPORT

                 OBJECT-AWARE CAPTIONING
                  PREPROCESSING REPORT

DATASET
-------
Datasets:
  Flickr8K + Flickr30K

Unique images:
  39,874

Total captions:
  199,369

Captions per image:
  Minimum: 4
  Maximum: 5
  Mean:    approximately 5.00


RT-DETR-X DETECTION REPRESENTATION
-----------------------------------
Detector:
  RT-DETR-X

Confidence threshold:
  0.30

Maximum objects per image:
  10

Images represented:
  39,864

Zero-object images:
  10

Retained object instances:
  243,482


OBJECT FEATURES
---------------
Object ID tensor:
  (39,874, 10)

Spatial tensor:
  (39,874, 10, 9)

Confidence tensor:
  (39,874, 10)

Object mask:
  (39,874, 10)


SPATIAL FEATURES
----------------
1. xmin
2. ymin
3. xmax
4. ymax
5. center_x
6. center_y
7. box_width
8. box_height
9. box_area


CAPTION FEATURES
----------------
Caption tensor:
  (199,369, 32)

Caption vocabulary:
  12,174

Special tokens:
  <pad>   = 0
  <start> = 1
  <end>   =

In [ ]:
# ============================================================
# CELL 24 — OBJECT-AWARE TRANSFORMER CONFIGURATION
# ============================================================

import os
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

print("=" * 75)
print("CELL 24 — OBJECT-AWARE TRANSFORMER CONFIGURATION")
print("=" * 75)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning"
)

FEATURE_DIR = (
    PROJECT_ROOT
    / "Features"
    / "Structured_Object"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Outputs"
    / "Structured_Captioning"
)

OBJECT_FEATURE_FILE = (
    FEATURE_DIR
    / "rtdetr_structured_object_features.npz"
)

CAPTION_FEATURE_FILE = (
    FEATURE_DIR
    / "caption_sequences.npz"
)

OBJECT_VOCAB_FILE = (
    OUTPUT_DIR
    / "object_vocabulary.json"
)

CAPTION_VOCAB_FILE = (
    OUTPUT_DIR
    / "caption_vocabulary.json"
)

CONFIG_FILE = (
    OUTPUT_DIR
    / "structured_captioning_config.json"
)

TRAIN_SPLIT_FILE = (
    OUTPUT_DIR
    / "train_image_ids.csv"
)

VAL_SPLIT_FILE = (
    OUTPUT_DIR
    / "val_image_ids.csv"
)

TEST_SPLIT_FILE = (
    OUTPUT_DIR
    / "test_image_ids.csv"
)

# ------------------------------------------------------------
# Model hyperparameters
# ------------------------------------------------------------

D_MODEL = 256
NHEAD = 8

NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3

DIM_FEEDFORWARD = 1024
DROPOUT = 0.2

MAX_CAPTION_LENGTH = 32

BATCH_SIZE = 64

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

NUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 4

TOP_K = 10
SPATIAL_DIM = 9

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:")
print(" ", DEVICE)

if torch.cuda.is_available():

    print(
        "  GPU:",
        torch.cuda.get_device_name(0)
    )

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(
        f"  GPU memory: {gpu_memory:.2f} GB"
    )

print("\nModel configuration:")
print(f"  D_MODEL:             {D_MODEL}")
print(f"  NHEAD:               {NHEAD}")
print(f"  Encoder layers:      {NUM_ENCODER_LAYERS}")
print(f"  Decoder layers:      {NUM_DECODER_LAYERS}")
print(f"  Feed-forward dim:    {DIM_FEEDFORWARD}")
print(f"  Dropout:             {DROPOUT}")
print(f"  Max caption length:  {MAX_CAPTION_LENGTH}")
print(f"  Batch size:          {BATCH_SIZE}")
print(f"  Learning rate:       {LEARNING_RATE}")
print(f"  Weight decay:        {WEIGHT_DECAY}")
print(f"  Top-K objects:       {TOP_K}")
print(f"  Spatial dimensions:  {SPATIAL_DIM}")

print("\n" + "=" * 75)
print("CELL 24 COMPLETE")
print("=" * 75)

CELL 24 — OBJECT-AWARE TRANSFORMER CONFIGURATION

Device:
  cuda
  GPU: Tesla T4
  GPU memory: 14.56 GB

Model configuration:
  D_MODEL:             256
  NHEAD:               8
  Encoder layers:      3
  Decoder layers:      3
  Feed-forward dim:    1024
  Dropout:             0.2
  Max caption length:  32
  Batch size:          64
  Learning rate:       0.0003
  Weight decay:        0.0001
  Top-K objects:       10
  Spatial dimensions:  9

CELL 24 COMPLETE


In [ ]:
# ============================================================
# CELL 25 — LOAD STRUCTURED FEATURES
# ============================================================

print("=" * 75)
print("CELL 25 — LOADING STRUCTURED FEATURES")
print("=" * 75)

# ------------------------------------------------------------
# Load NPZ files
# ------------------------------------------------------------

object_data = np.load(
    OBJECT_FEATURE_FILE,
    allow_pickle=False
)

caption_data = np.load(
    CAPTION_FEATURE_FILE,
    allow_pickle=False
)

# ------------------------------------------------------------
# Object features
# ------------------------------------------------------------

OBJECT_IMAGE_IDS = (
    object_data["image_ids"]
    .astype(str)
)

OBJECT_IDS = object_data["object_ids"]
SPATIAL_FEATURES = object_data["boxes"]
CONFIDENCE_FEATURES = object_data["confidences"]
OBJECT_MASKS = object_data["object_mask"]

# ------------------------------------------------------------
# Caption features
# ------------------------------------------------------------

CAPTION_IMAGE_IDS = (
    caption_data["image_ids"]
    .astype(str)
)

CAPTION_SEQUENCES = (
    caption_data["caption_ids"]
)

# ------------------------------------------------------------
# Build image → object row mapping
# ------------------------------------------------------------

IMAGE_TO_OBJECT_ROW = {
    image_id: i
    for i, image_id
    in enumerate(OBJECT_IMAGE_IDS)
}

# ------------------------------------------------------------
# Print shapes
# ------------------------------------------------------------

print("\nObject features:")
print("  Image IDs :", OBJECT_IMAGE_IDS.shape)
print("  Object IDs:", OBJECT_IDS.shape)
print("  Spatial   :", SPATIAL_FEATURES.shape)
print("  Confidence:", CONFIDENCE_FEATURES.shape)
print("  Mask      :", OBJECT_MASKS.shape)

print("\nCaption features:")
print("  Image IDs :", CAPTION_IMAGE_IDS.shape)
print("  Sequences :", CAPTION_SEQUENCES.shape)

print(
    "\nImage → object mapping:",
    len(IMAGE_TO_OBJECT_ROW)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert OBJECT_IDS.shape == (39874, 10)
assert SPATIAL_FEATURES.shape == (39874, 10, 9)
assert CONFIDENCE_FEATURES.shape == (39874, 10)
assert OBJECT_MASKS.shape == (39874, 10)

assert CAPTION_SEQUENCES.shape == (
    199369,
    32
)

assert len(IMAGE_TO_OBJECT_ROW) == 39874

print("\n✓ All permanent feature arrays loaded successfully.")

print("\n" + "=" * 75)
print("CELL 25 COMPLETE")
print("=" * 75)

CELL 25 — LOADING STRUCTURED FEATURES

Object features:
  Image IDs : (39874,)
  Object IDs: (39874, 10)
  Spatial   : (39874, 10, 9)
  Confidence: (39874, 10)
  Mask      : (39874, 10)

Caption features:
  Image IDs : (199369,)
  Sequences : (199369, 32)

Image → object mapping: 39874

✓ All permanent feature arrays loaded successfully.

CELL 25 COMPLETE


In [ ]:
# ============================================================
# CELL 26 — LOAD VOCABULARIES AND DATASET SPLITS
# ============================================================

print("=" * 75)
print("CELL 26 — LOADING VOCABULARIES AND DATASET SPLITS")
print("=" * 75)

# ------------------------------------------------------------
# Object vocabulary
# ------------------------------------------------------------

with open(
    OBJECT_VOCAB_FILE,
    "r",
    encoding="utf-8"
) as f:

    object_vocab_raw = json.load(f)

if "object_to_id" in object_vocab_raw:
    OBJECT_TO_ID = object_vocab_raw["object_to_id"]
else:
    OBJECT_TO_ID = object_vocab_raw

OBJECT_TO_ID = {
    str(k): int(v)
    for k, v in OBJECT_TO_ID.items()
}

ID_TO_OBJECT = {
    int(v): str(k)
    for k, v in OBJECT_TO_ID.items()
}

# ------------------------------------------------------------
# Caption vocabulary
# ------------------------------------------------------------

with open(
    CAPTION_VOCAB_FILE,
    "r",
    encoding="utf-8"
) as f:

    caption_vocab_raw = json.load(f)

if "caption_to_id" in caption_vocab_raw:
    CAPTION_TO_ID = caption_vocab_raw["caption_to_id"]
else:
    CAPTION_TO_ID = caption_vocab_raw

CAPTION_TO_ID = {
    str(k): int(v)
    for k, v in CAPTION_TO_ID.items()
}

ID_TO_CAPTION = {
    int(v): str(k)
    for k, v in CAPTION_TO_ID.items()
}

# ------------------------------------------------------------
# Special tokens
# ------------------------------------------------------------

PAD_ID = CAPTION_TO_ID["<pad>"]
START_ID = CAPTION_TO_ID["<start>"]
END_ID = CAPTION_TO_ID["<end>"]
UNK_ID = CAPTION_TO_ID["<unk>"]

# ------------------------------------------------------------
# Dataset splits
# ------------------------------------------------------------

TRAIN_IMAGE_IDS = set(
    pd.read_csv(TRAIN_SPLIT_FILE)[
        "image_id"
    ].astype(str)
)

VAL_IMAGE_IDS = set(
    pd.read_csv(VAL_SPLIT_FILE)[
        "image_id"
    ].astype(str)
)

TEST_IMAGE_IDS = set(
    pd.read_csv(TEST_SPLIT_FILE)[
        "image_id"
    ].astype(str)
)

# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print("\nObject vocabulary:")
print(
    f"  Size: {len(OBJECT_TO_ID):,}"
)

print("\nCaption vocabulary:")
print(
    f"  Size: {len(CAPTION_TO_ID):,}"
)

print("\nSpecial tokens:")
print(f"  PAD:   {PAD_ID}")
print(f"  START: {START_ID}")
print(f"  END:   {END_ID}")
print(f"  UNK:   {UNK_ID}")

print("\nDataset splits:")
print(f"  Train: {len(TRAIN_IMAGE_IDS):,}")
print(f"  Val:   {len(VAL_IMAGE_IDS):,}")
print(f"  Test:  {len(TEST_IMAGE_IDS):,}")

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(OBJECT_TO_ID) == 82
assert len(CAPTION_TO_ID) == 12174

assert len(TRAIN_IMAGE_IDS) == 31899
assert len(VAL_IMAGE_IDS) == 3987
assert len(TEST_IMAGE_IDS) == 3988

assert (
    TRAIN_IMAGE_IDS.isdisjoint(VAL_IMAGE_IDS)
)

assert (
    TRAIN_IMAGE_IDS.isdisjoint(TEST_IMAGE_IDS)
)

assert (
    VAL_IMAGE_IDS.isdisjoint(TEST_IMAGE_IDS)
)

print("\n✓ Vocabularies and splits loaded correctly.")

print("\n" + "=" * 75)
print("CELL 26 COMPLETE")
print("=" * 75)

CELL 26 — LOADING VOCABULARIES AND DATASET SPLITS

Object vocabulary:
  Size: 82

Caption vocabulary:
  Size: 12,174

Special tokens:
  PAD:   0
  START: 1
  END:   2
  UNK:   3

Dataset splits:
  Train: 31,899
  Val:   3,987
  Test:  3,988

✓ Vocabularies and splits loaded correctly.

CELL 26 COMPLETE


In [ ]:
# ============================================================
# CELL 27 — BUILD CAPTION-LEVEL PYTORCH DATASET
# ============================================================

print("=" * 75)
print("CELL 27 — BUILDING PYTORCH DATASET")
print("=" * 75)


class ObjectCaptionDataset(Dataset):

    def __init__(
        self,
        image_ids,
        caption_sequences,
        split_image_ids,
        image_to_object_row
    ):

        self.image_ids = image_ids
        self.caption_sequences = caption_sequences
        self.split_image_ids = split_image_ids
        self.image_to_object_row = image_to_object_row

        # ----------------------------------------------------
        # Select caption rows belonging to this split
        # ----------------------------------------------------

        self.caption_indices = [
            i
            for i, image_id
            in enumerate(self.image_ids)
            if image_id in self.split_image_ids
        ]

        print(
            f"Created dataset with "
            f"{len(self.caption_indices):,} captions"
        )

    def __len__(self):
        return len(self.caption_indices)

    def __getitem__(self, index):

        caption_index = self.caption_indices[index]

        image_id = self.image_ids[
            caption_index
        ]

        object_row = self.image_to_object_row[
            image_id
        ]

        # ----------------------------------------------------
        # Object representation
        # ----------------------------------------------------

        object_ids = torch.from_numpy(
            OBJECT_IDS[object_row]
        ).long()

        spatial = torch.from_numpy(
            SPATIAL_FEATURES[object_row]
        ).float()

        confidence = torch.from_numpy(
            CONFIDENCE_FEATURES[object_row]
        ).float()

        object_mask = torch.from_numpy(
            OBJECT_MASKS[object_row]
        ).bool()

        # ----------------------------------------------------
        # Caption
        # ----------------------------------------------------

        caption = torch.from_numpy(
            self.caption_sequences[
                caption_index
            ]
        ).long()

        # ----------------------------------------------------
        # Teacher-forcing inputs/targets
        # ----------------------------------------------------

        decoder_input = caption[:-1]

        target = caption[1:]

        return {
            "image_id": image_id,
            "object_ids": object_ids,
            "spatial": spatial,
            "confidence": confidence,
            "object_mask": object_mask,
            "decoder_input": decoder_input,
            "target": target
        }


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = ObjectCaptionDataset(
    CAPTION_IMAGE_IDS,
    CAPTION_SEQUENCES,
    TRAIN_IMAGE_IDS,
    IMAGE_TO_OBJECT_ROW
)

val_dataset = ObjectCaptionDataset(
    CAPTION_IMAGE_IDS,
    CAPTION_SEQUENCES,
    VAL_IMAGE_IDS,
    IMAGE_TO_OBJECT_ROW
)

test_dataset = ObjectCaptionDataset(
    CAPTION_IMAGE_IDS,
    CAPTION_SEQUENCES,
    TEST_IMAGE_IDS,
    IMAGE_TO_OBJECT_ROW
)

# ------------------------------------------------------------
# Expected caption counts
# ------------------------------------------------------------

print("\nDataset sizes:")
print(f"  Train captions: {len(train_dataset):,}")
print(f"  Val captions:   {len(val_dataset):,}")
print(f"  Test captions:  {len(test_dataset):,}")

assert len(train_dataset) > 0
assert len(val_dataset) > 0
assert len(test_dataset) > 0

print("\n✓ Caption-level datasets created successfully.")

print("\n" + "=" * 75)
print("CELL 27 COMPLETE")
print("=" * 75)

CELL 27 — BUILDING PYTORCH DATASET
Created dataset with 159,494 captions
Created dataset with 19,935 captions
Created dataset with 19,940 captions

Dataset sizes:
  Train captions: 159,494
  Val captions:   19,935
  Test captions:  19,940

✓ Caption-level datasets created successfully.

CELL 27 COMPLETE


In [ ]:
# ============================================================
# CELL 28 — INSPECT DATASET EXAMPLE AND DATALOADER
# ============================================================

print("=" * 75)
print("CELL 28 — INSPECTING DATASET AND DATALOADER")
print("=" * 75)

# ------------------------------------------------------------
# Single example
# ------------------------------------------------------------

sample = train_dataset[0]

print("\nSingle training example:")

for key, value in sample.items():

    if torch.is_tensor(value):

        print(
            f"  {key:15s}"
            f" shape={tuple(value.shape)}"
            f" dtype={value.dtype}"
        )

    else:

        print(
            f"  {key:15s}"
            f" value={value}"
        )

# ------------------------------------------------------------
# Verify decoder dimensions
# ------------------------------------------------------------

assert sample["object_ids"].shape == (10,)
assert sample["spatial"].shape == (10, 9)
assert sample["confidence"].shape == (10,)
assert sample["object_mask"].shape == (10,)

assert sample["decoder_input"].shape == (31,)
assert sample["target"].shape == (31,)

# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# ------------------------------------------------------------
# Get one batch
# ------------------------------------------------------------

batch = next(
    iter(train_loader)
)

print("\nFirst batch:")

for key, value in batch.items():

    if torch.is_tensor(value):

        print(
            f"  {key:15s}"
            f" shape={tuple(value.shape)}"
            f" dtype={value.dtype}"
        )

    else:

        print(
            f"  {key:15s}"
            f" type={type(value).__name__}"
        )

# ------------------------------------------------------------
# Batch validation
# ------------------------------------------------------------

assert batch["object_ids"].shape == (
    BATCH_SIZE,
    TOP_K
)

assert batch["spatial"].shape == (
    BATCH_SIZE,
    TOP_K,
    SPATIAL_DIM
)

assert batch["confidence"].shape == (
    BATCH_SIZE,
    TOP_K
)

assert batch["object_mask"].shape == (
    BATCH_SIZE,
    TOP_K
)

assert batch["decoder_input"].shape == (
    BATCH_SIZE,
    MAX_CAPTION_LENGTH - 1
)

assert batch["target"].shape == (
    BATCH_SIZE,
    MAX_CAPTION_LENGTH - 1
)

print("\n✓ Dataset example is correct.")
print("✓ DataLoader batch dimensions are correct.")

print("\n" + "=" * 75)
print("CELL 28 COMPLETE")
print("=" * 75)

CELL 28 — INSPECTING DATASET AND DATALOADER

Single training example:
  image_id        value=Flickr8K/1001773457_577c3a7d70.jpg
  object_ids      shape=(10,) dtype=torch.int64
  spatial         shape=(10, 9) dtype=torch.float32
  confidence      shape=(10,) dtype=torch.float32
  object_mask     shape=(10,) dtype=torch.bool
  decoder_input   shape=(31,) dtype=torch.int64
  target          shape=(31,) dtype=torch.int64

First batch:
  image_id        type=list
  object_ids      shape=(64, 10) dtype=torch.int64
  spatial         shape=(64, 10, 9) dtype=torch.float32
  confidence      shape=(64, 10) dtype=torch.float32
  object_mask     shape=(64, 10) dtype=torch.bool
  decoder_input   shape=(64, 31) dtype=torch.int64
  target          shape=(64, 31) dtype=torch.int64

✓ Dataset example is correct.
✓ DataLoader batch dimensions are correct.

CELL 28 COMPLETE


In [ ]:
# ============================================================
# CELL 29 — OBJECT FEATURE ENCODER
# ============================================================

print("=" * 75)
print("CELL 29 — BUILDING OBJECT FEATURE ENCODER")
print("=" * 75)


class ObjectFeatureEncoder(nn.Module):
    """
    Converts each detected object into a unified D_MODEL vector.

    Inputs:
        object_ids  : (B, K)
        spatial     : (B, K, 9)
        confidence  : (B, K)

    Output:
        object_tokens : (B, K, D_MODEL)
    """

    def __init__(
        self,
        object_vocab_size,
        d_model=256,
        spatial_dim=9,
        dropout=0.2
    ):

        super().__init__()

        # ----------------------------------------------------
        # Object class embedding
        # ----------------------------------------------------

        self.object_embedding = nn.Embedding(
            num_embeddings=object_vocab_size,
            embedding_dim=d_model,
            padding_idx=0
        )

        # ----------------------------------------------------
        # Spatial feature projection
        # 9-D → D_MODEL
        # ----------------------------------------------------

        self.spatial_mlp = nn.Sequential(
            nn.Linear(spatial_dim, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # ----------------------------------------------------
        # Confidence projection
        # 1-D → D_MODEL
        # ----------------------------------------------------

        self.confidence_mlp = nn.Sequential(
            nn.Linear(1, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )

        # ----------------------------------------------------
        # Fusion
        # ----------------------------------------------------

        self.fusion = nn.Sequential(
            nn.Linear(
                d_model * 3,
                d_model
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.LayerNorm(d_model)
        )

    def forward(
        self,
        object_ids,
        spatial,
        confidence
    ):

        # Object ID embedding
        object_embed = self.object_embedding(
            object_ids
        )

        # Spatial embedding
        spatial_embed = self.spatial_mlp(
            spatial
        )

        # Confidence embedding
        confidence_embed = self.confidence_mlp(
            confidence.unsqueeze(-1)
        )

        # Concatenate all three representations
        combined = torch.cat(
            [
                object_embed,
                spatial_embed,
                confidence_embed
            ],
            dim=-1
        )

        # Fuse into D_MODEL
        object_tokens = self.fusion(
            combined
        )

        return object_tokens


print("\n✓ ObjectFeatureEncoder defined.")

print("\nInput:")
print("  Object ID   → embedding")
print("  Spatial     → 9-D MLP")
print("  Confidence  → 1-D MLP")

print("\nOutput:")
print(
    f"  Object token → {D_MODEL}-D"
)

print("\n" + "=" * 75)
print("CELL 29 COMPLETE")
print("=" * 75)

CELL 29 — BUILDING OBJECT FEATURE ENCODER

✓ ObjectFeatureEncoder defined.

Input:
  Object ID   → embedding
  Spatial     → 9-D MLP
  Confidence  → 1-D MLP

Output:
  Object token → 256-D

CELL 29 COMPLETE


In [ ]:
# ============================================================
# CELL 30 — POSITIONAL ENCODING AND TRANSFORMER COMPONENTS
# ============================================================

print("=" * 75)
print("CELL 30 — BUILDING TRANSFORMER COMPONENTS")
print("=" * 75)


class PositionalEncoding(nn.Module):
    """
    Standard sinusoidal positional encoding.
    """

    def __init__(
        self,
        d_model,
        max_len=512,
        dropout=0.1
    ):

        super().__init__()

        self.dropout = nn.Dropout(
            dropout
        )

        position = torch.arange(
            max_len
        ).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ).float()
            * (
                -math.log(10000.0)
                / d_model
            )
        )

        pe = torch.zeros(
            max_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        # x: (B, T, D)

        x = x + self.pe[
            :, :x.size(1)
        ]

        return self.dropout(x)


# ------------------------------------------------------------
# Caption embedding
# ------------------------------------------------------------

class CaptionEmbedding(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        max_len,
        dropout
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_ID
        )

        self.position = PositionalEncoding(
            d_model,
            max_len=max_len,
            dropout=dropout
        )

    def forward(self, tokens):

        x = self.embedding(tokens)

        return self.position(x)


print("\n✓ PositionalEncoding defined.")
print("✓ CaptionEmbedding defined.")

print("\nTransformer settings:")
print(f"  D_MODEL:        {D_MODEL}")
print(f"  Heads:          {NHEAD}")
print(f"  Encoder layers: {NUM_ENCODER_LAYERS}")
print(f"  Decoder layers: {NUM_DECODER_LAYERS}")
print(f"  FFN dimension:  {DIM_FEEDFORWARD}")

print("\n" + "=" * 75)
print("CELL 30 COMPLETE")
print("=" * 75)

CELL 30 — BUILDING TRANSFORMER COMPONENTS

✓ PositionalEncoding defined.
✓ CaptionEmbedding defined.

Transformer settings:
  D_MODEL:        256
  Heads:          8
  Encoder layers: 3
  Decoder layers: 3
  FFN dimension:  1024

CELL 30 COMPLETE


In [ ]:
# ============================================================
# CELL 31 — COMPLETE OBJECT-AWARE TRANSFORMER
# ============================================================

print("=" * 75)
print("CELL 31 — BUILDING OBJECT-AWARE TRANSFORMER")
print("=" * 75)


class ObjectAwareTransformer(nn.Module):

    def __init__(
        self,
        object_vocab_size,
        caption_vocab_size,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=1024,
        dropout=0.2,
        max_caption_length=32
    ):

        super().__init__()

        self.d_model = d_model

        # ----------------------------------------------------
        # Object encoder
        # ----------------------------------------------------

        self.object_encoder = ObjectFeatureEncoder(
            object_vocab_size=object_vocab_size,
            d_model=d_model,
            spatial_dim=SPATIAL_DIM,
            dropout=dropout
        )

        # ----------------------------------------------------
        # Object positional encoding
        # ----------------------------------------------------

        self.object_position = PositionalEncoding(
            d_model=d_model,
            max_len=TOP_K,
            dropout=dropout
        )

        # ----------------------------------------------------
        # Transformer encoder
        # ----------------------------------------------------

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )

        # ----------------------------------------------------
        # Caption embedding
        # ----------------------------------------------------

        self.caption_embedding = CaptionEmbedding(
            vocab_size=caption_vocab_size,
            d_model=d_model,
            max_len=max_caption_length,
            dropout=dropout
        )

        # ----------------------------------------------------
        # Transformer decoder
        # ----------------------------------------------------

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_decoder_layers
        )

        # ----------------------------------------------------
        # Vocabulary projection
        # ----------------------------------------------------

        self.output_projection = nn.Linear(
            d_model,
            caption_vocab_size
        )

        # ----------------------------------------------------
        # Final initialization
        # ----------------------------------------------------

        self._init_weights()

    def _init_weights(self):

        # Xavier initialization for linear layers

        for module in self.modules():

            if isinstance(
                module,
                nn.Linear
            ):

                nn.init.xavier_uniform_(
                    module.weight
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

    def generate_causal_mask(
        self,
        size,
        device
    ):

        # True = blocked attention
        return torch.triu(
            torch.ones(
                size,
                size,
                device=device,
                dtype=torch.bool
            ),
            diagonal=1
        )

    def forward(
        self,
        object_ids,
        spatial,
        confidence,
        object_mask,
        decoder_input
    ):

        # ====================================================
        # OBJECT ENCODER
        # ====================================================

        object_tokens = self.object_encoder(
            object_ids,
            spatial,
            confidence
        )

        # Add object-slot positional information
        object_tokens = self.object_position(
            object_tokens
        )

        # ====================================================
        # TRANSFORMER ENCODER
        # ====================================================

        memory = self.transformer_encoder(
            object_tokens,
            src_key_padding_mask=~object_mask
        )

        # ====================================================
        # CAPTION DECODER
        # ====================================================

        decoder_tokens = self.caption_embedding(
            decoder_input
        )

        # Causal mask prevents looking into the future
        target_mask = self.generate_causal_mask(
            decoder_input.size(1),
            decoder_input.device
        )

        decoded = self.transformer_decoder(
            decoder_tokens,
            memory,
            tgt_mask=target_mask,
            memory_key_padding_mask=~object_mask
        )

        # ====================================================
        # VOCABULARY LOGITS
        # ====================================================

        logits = self.output_projection(
            decoded
        )

        return logits


print("\n✓ ObjectAwareTransformer defined.")

print("\nArchitecture:")
print("""
Object ID
    ↓
Object Embedding
    +
Spatial MLP
    +
Confidence MLP
    ↓
Feature Fusion
    ↓
Object Positional Encoding
    ↓
Transformer Encoder
    ↓
Object Memory
    ↓
Transformer Decoder ← Caption Embedding + Position
    ↓
Vocabulary Projection
    ↓
12,174 vocabulary logits
""")

print("\n" + "=" * 75)
print("CELL 31 COMPLETE")
print("=" * 75)

CELL 31 — BUILDING OBJECT-AWARE TRANSFORMER

✓ ObjectAwareTransformer defined.

Architecture:

Object ID
    ↓
Object Embedding
    +
Spatial MLP
    +
Confidence MLP
    ↓
Feature Fusion
    ↓
Object Positional Encoding
    ↓
Transformer Encoder
    ↓
Object Memory
    ↓
Transformer Decoder ← Caption Embedding + Position
    ↓
Vocabulary Projection
    ↓
12,174 vocabulary logits


CELL 31 COMPLETE


In [ ]:
# ============================================================
# CELL 32 — INSTANTIATE OBJECT-AWARE TRANSFORMER
# ============================================================

print("=" * 75)
print("CELL 32 — INSTANTIATING MODEL")
print("=" * 75)

model = ObjectAwareTransformer(
    object_vocab_size=len(OBJECT_TO_ID),
    caption_vocab_size=len(CAPTION_TO_ID),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_caption_length=MAX_CAPTION_LENGTH
)

model = model.to(DEVICE)

# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nModel:")
print(model)

print("\nParameter count:")
print(
    f"  Total:     {total_parameters:,}"
)

print(
    f"  Trainable: {trainable_parameters:,}"
)

print("\n✓ Model instantiated and moved to device.")

print("\n" + "=" * 75)
print("CELL 32 COMPLETE")
print("=" * 75)

CELL 32 — INSTANTIATING MODEL


/tmp/ipykernel_1090/3777785277.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(



Model:
ObjectAwareTransformer(
  (object_encoder): ObjectFeatureEncoder(
    (object_embedding): Embedding(82, 256, padding_idx=0)
    (spatial_mlp): Sequential(
      (0): Linear(in_features=9, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
    )
    (confidence_mlp): Sequential(
      (0): Linear(in_features=1, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=256, bias=True)
    )
    (fusion): Sequential(
      (0): Linear(in_features=768, out_features=256, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.2, inplace=False)
      (3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
  )
  (object_position): PositionalEncoding(
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicall

In [ ]:
# ============================================================
# CELL 33 — SINGLE FORWARD PASS TEST
# ============================================================

print("=" * 75)
print("CELL 33 — TESTING SINGLE FORWARD PASS")
print("=" * 75)

# ------------------------------------------------------------
# Move batch to GPU
# ------------------------------------------------------------

object_ids_batch = batch[
    "object_ids"
].to(DEVICE)

spatial_batch = batch[
    "spatial"
].to(DEVICE)

confidence_batch = batch[
    "confidence"
].to(DEVICE)

object_mask_batch = batch[
    "object_mask"
].to(DEVICE)

decoder_input_batch = batch[
    "decoder_input"
].to(DEVICE)

# ------------------------------------------------------------
# Evaluation mode for this test
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    logits = model(
        object_ids=object_ids_batch,
        spatial=spatial_batch,
        confidence=confidence_batch,
        object_mask=object_mask_batch,
        decoder_input=decoder_input_batch
    )

# ------------------------------------------------------------
# Expected shape
# ------------------------------------------------------------

expected_shape = (
    BATCH_SIZE,
    MAX_CAPTION_LENGTH - 1,
    len(CAPTION_TO_ID)
)

print("\nInput shapes:")
print(
    "  object_ids :",
    tuple(object_ids_batch.shape)
)

print(
    "  spatial    :",
    tuple(spatial_batch.shape)
)

print(
    "  confidence :",
    tuple(confidence_batch.shape)
)

print(
    "  object_mask:",
    tuple(object_mask_batch.shape)
)

print(
    "  decoder_in :",
    tuple(decoder_input_batch.shape)
)

print("\nOutput:")
print(
    "  logits     :",
    tuple(logits.shape)
)

print("\nExpected:")
print(
    "  logits     :",
    expected_shape
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert logits.shape == expected_shape

assert torch.isfinite(
    logits
).all()

print("\n✓ Forward pass completed successfully.")
print("✓ Output shape is correct.")
print("✓ Logits contain no NaN or infinite values.")

# ------------------------------------------------------------
# GPU memory
# ------------------------------------------------------------

if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated()
        / (1024 ** 3)
    )

    reserved = (
        torch.cuda.memory_reserved()
        / (1024 ** 3)
    )

    print("\nGPU memory after forward pass:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")

print("\n" + "=" * 75)
print("CELL 33 COMPLETE")
print("=" * 75)

CELL 33 — TESTING SINGLE FORWARD PASS

Input shapes:
  object_ids : (64, 10)
  spatial    : (64, 10, 9)
  confidence : (64, 10)
  object_mask: (64, 10)
  decoder_in : (64, 31)

Output:
  logits     : (64, 31, 12174)

Expected:
  logits     : (64, 31, 12174)

✓ Forward pass completed successfully.
✓ Output shape is correct.
✓ Logits contain no NaN or infinite values.

GPU memory after forward pass:
  Allocated: 0.14 GB
  Reserved:  0.34 GB

CELL 33 COMPLETE


In [ ]:
# ============================================================
# CELL 34 — LOSS FUNCTION AND OPTIMIZER
# ============================================================

print("=" * 75)
print("CELL 34 — CONFIGURING LOSS AND OPTIMIZER")
print("=" * 75)

# ------------------------------------------------------------
# Cross-entropy loss
#
# PAD tokens are ignored.
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=0.1
)

# ------------------------------------------------------------
# AdamW optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# Learning-rate scheduler
# ------------------------------------------------------------

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

print("\nLoss:")
print("  CrossEntropyLoss")
print(f"  Ignore index: {PAD_ID}")
print("  Label smoothing: 0.1")

print("\nOptimizer:")
print("  AdamW")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay:  {WEIGHT_DECAY}")

print("\nScheduler:")
print("  ReduceLROnPlateau")
print("  Factor: 0.5")
print("  Patience: 1")

print("\n✓ Loss, optimizer, and scheduler configured.")

print("\n" + "=" * 75)
print("CELL 34 COMPLETE")
print("=" * 75)

CELL 34 — CONFIGURING LOSS AND OPTIMIZER

Loss:
  CrossEntropyLoss
  Ignore index: 0
  Label smoothing: 0.1

Optimizer:
  AdamW
  Learning rate: 0.0003
  Weight decay:  0.0001

Scheduler:
  ReduceLROnPlateau
  Factor: 0.5
  Patience: 1

✓ Loss, optimizer, and scheduler configured.

CELL 34 COMPLETE


In [ ]:
# ============================================================
# CELL 35 — MIXED PRECISION SETUP
# ============================================================

print("=" * 75)
print("CELL 35 — CONFIGURING MIXED PRECISION")
print("=" * 75)

# ------------------------------------------------------------
# PyTorch AMP
# ------------------------------------------------------------

USE_AMP = torch.cuda.is_available()

if USE_AMP:

    scaler = torch.amp.GradScaler(
        "cuda"
    )

    print("\n✓ CUDA mixed precision enabled.")

else:

    scaler = None

    print(
        "\n⚠ CUDA unavailable."
        "\nUsing standard FP32 training."
    )

print("\nAMP enabled:", USE_AMP)

print("\n" + "=" * 75)
print("CELL 35 COMPLETE")
print("=" * 75)

CELL 35 — CONFIGURING MIXED PRECISION

✓ CUDA mixed precision enabled.

AMP enabled: True

CELL 35 COMPLETE


In [ ]:
# ============================================================
# CELL 36 — ONE-BATCH TRAINING SANITY TEST
# ============================================================

print("=" * 75)
print("CELL 36 — ONE-BATCH TRAINING SANITY TEST")
print("=" * 75)

model.train()

# ------------------------------------------------------------
# Get one batch
# ------------------------------------------------------------

sanity_batch = next(
    iter(train_loader)
)

object_ids_s = sanity_batch[
    "object_ids"
].to(DEVICE)

spatial_s = sanity_batch[
    "spatial"
].to(DEVICE)

confidence_s = sanity_batch[
    "confidence"
].to(DEVICE)

object_mask_s = sanity_batch[
    "object_mask"
].to(DEVICE)

decoder_input_s = sanity_batch[
    "decoder_input"
].to(DEVICE)

target_s = sanity_batch[
    "target"
].to(DEVICE)

# ------------------------------------------------------------
# Clear gradients
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

# ------------------------------------------------------------
# Forward + loss
# ------------------------------------------------------------

if USE_AMP:

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        logits_s = model(
            object_ids_s,
            spatial_s,
            confidence_s,
            object_mask_s,
            decoder_input_s
        )

        loss_s = criterion(
            logits_s.reshape(-1, len(CAPTION_TO_ID)),
            target_s.reshape(-1)
        )

else:

    logits_s = model(
        object_ids_s,
        spatial_s,
        confidence_s,
        object_mask_s,
        decoder_input_s
    )

    loss_s = criterion(
        logits_s.reshape(-1, len(CAPTION_TO_ID)),
        target_s.reshape(-1)
    )

print("\nForward result:")
print(
    "  Logits:",
    tuple(logits_s.shape)
)

print(
    f"  Loss: {loss_s.item():.6f}"
)

# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------

if USE_AMP:

    scaler.scale(
        loss_s
    ).backward()

else:

    loss_s.backward()

# ------------------------------------------------------------
# Gradient validation
# ------------------------------------------------------------

gradient_count = 0
finite_gradient_count = 0
max_gradient = 0.0

for parameter in model.parameters():

    if parameter.grad is not None:

        gradient_count += 1

        if torch.isfinite(
            parameter.grad
        ).all():

            finite_gradient_count += 1

        max_gradient = max(
            max_gradient,
            parameter.grad.detach().abs().max().item()
        )

print("\nGradient check:")
print(
    f"  Parameters with gradients: "
    f"{gradient_count}"
)

print(
    f"  Finite gradients: "
    f"{finite_gradient_count}"
)

print(
    f"  Maximum gradient: "
    f"{max_gradient:.6f}"
)

assert gradient_count > 0
assert gradient_count == finite_gradient_count
assert torch.isfinite(loss_s)

# ------------------------------------------------------------
# Optimizer step
# ------------------------------------------------------------

if USE_AMP:

    scaler.step(optimizer)
    scaler.update()

else:

    optimizer.step()

print("\n✓ Backward pass successful.")
print("✓ Gradients are finite.")
print("✓ Optimizer step successful.")

print("\n" + "=" * 75)
print("CELL 36 COMPLETE")
print("=" * 75)

CELL 36 — ONE-BATCH TRAINING SANITY TEST

Forward result:
  Logits: (64, 31, 12174)
  Loss: 9.583233

Gradient check:
  Parameters with gradients: 106
  Finite gradients: 106
  Maximum gradient: 33888.000000

✓ Backward pass successful.
✓ Gradients are finite.
✓ Optimizer step successful.

CELL 36 COMPLETE


In [ ]:
# ============================================================
# CELL 37 — VERIFY PARAMETER UPDATE
# ============================================================

print("=" * 75)
print("CELL 37 — VERIFYING PARAMETER UPDATE")
print("=" * 75)

# ------------------------------------------------------------
# Take a fresh batch
# ------------------------------------------------------------

check_batch = next(
    iter(train_loader)
)

object_ids_c = check_batch[
    "object_ids"
].to(DEVICE)

spatial_c = check_batch[
    "spatial"
].to(DEVICE)

confidence_c = check_batch[
    "confidence"
].to(DEVICE)

object_mask_c = check_batch[
    "object_mask"
].to(DEVICE)

decoder_input_c = check_batch[
    "decoder_input"
].to(DEVICE)

target_c = check_batch[
    "target"
].to(DEVICE)

# ------------------------------------------------------------
# Evaluate current loss
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    if USE_AMP:

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits_c = model(
                object_ids_c,
                spatial_c,
                confidence_c,
                object_mask_c,
                decoder_input_c
            )

    else:

        logits_c = model(
            object_ids_c,
            spatial_c,
            confidence_c,
            object_mask_c,
            decoder_input_c
        )

    loss_after_step = criterion(
        logits_c.reshape(
            -1,
            len(CAPTION_TO_ID)
        ),
        target_c.reshape(-1)
    )

print(
    f"\nFresh-batch loss after optimizer step: "
    f"{loss_after_step.item():.6f}"
)

assert torch.isfinite(
    loss_after_step
)

print("\n✓ Model produces valid loss after parameter update.")

print("\n" + "=" * 75)
print("CELL 37 COMPLETE")
print("=" * 75)

CELL 37 — VERIFYING PARAMETER UPDATE

Fresh-batch loss after optimizer step: inf


AssertionError: 

In [ ]:
print("=" * 75)
print("CELL 38 — CHECKING MODEL NUMERICAL STATE")
print("=" * 75)

bad_params = []
total_params = 0

for name, param in model.named_parameters():
    total_params += param.numel()

    if not torch.isfinite(param).all():
        bad_params.append(name)

print(f"Total parameters: {total_params:,}")
print(f"Parameters with NaN/Inf: {len(bad_params)}")

if bad_params:
    print("\nFirst affected parameters:")
    for name in bad_params[:10]:
        print(" ", name)

    print("\n⚠️ Model parameters were numerically corrupted by the sanity step.")
else:
    print("\n✓ All model parameters are finite.")

print("=" * 75)
print("CELL 38 COMPLETE")
print("=" * 75)

CELL 38 — CHECKING MODEL NUMERICAL STATE
Total parameters: 12,094,862
Parameters with NaN/Inf: 0

✓ All model parameters are finite.
CELL 38 COMPLETE


In [ ]:
import inspect

print("=" * 75)
print("DIAGNOSTIC — ACTUAL ObjectAwareTransformer CONSTRUCTOR")
print("=" * 75)

print(inspect.signature(ObjectAwareTransformer.__init__))

print("\nFull constructor source:")
print(inspect.getsource(ObjectAwareTransformer.__init__))

print("=" * 75)

DIAGNOSTIC — ACTUAL ObjectAwareTransformer CONSTRUCTOR
(self, object_vocab_size, caption_vocab_size, d_model=256, nhead=8, num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=1024, dropout=0.2, max_caption_length=32)

Full constructor source:
    def __init__(
        self,
        object_vocab_size,
        caption_vocab_size,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=1024,
        dropout=0.2,
        max_caption_length=32
    ):

        super().__init__()

        self.d_model = d_model

        # ----------------------------------------------------
        # Object encoder
        # ----------------------------------------------------

        self.object_encoder = ObjectFeatureEncoder(
            object_vocab_size=object_vocab_size,
            d_model=d_model,
            spatial_dim=SPATIAL_DIM,
            dropout=dropout
        )

        # -----------------------------------------------

In [ ]:
print("=" * 75)
print("CELL 39 — RESTORING CLEAN MODEL")
print("=" * 75)

# ----------------------------------------------------
# Restore device explicitly
# ----------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ----------------------------------------------------
# Caption length
# ----------------------------------------------------

CAPTION_LENGTH = caption_sequences.shape[1]

print(f"Caption sequence length: {CAPTION_LENGTH}")
print(f"Object vocabulary size: {len(object_to_id)}")
print(f"Caption vocabulary size: {len(caption_to_id)}")

# ----------------------------------------------------
# Recreate clean model
# ----------------------------------------------------

model = ObjectAwareTransformer(
    object_vocab_size=len(object_to_id),
    caption_vocab_size=len(caption_to_id),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_caption_length=CAPTION_LENGTH
).to(device)

# ----------------------------------------------------
# Verify parameters
# ----------------------------------------------------

bad_parameters = []

for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        bad_parameters.append(name)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print()
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Parameters with NaN/Inf: {len(bad_parameters)}")

assert len(bad_parameters) == 0

print()
print("✓ Device restored.")
print("✓ Fresh model created.")
print("✓ Model parameters are finite.")

print("=" * 75)
print("CELL 39 COMPLETE")
print("=" * 75)

CELL 39 — RESTORING CLEAN MODEL
Device: cuda
GPU: Tesla T4
Caption sequence length: 32
Object vocabulary size: 82
Caption vocabulary size: 12174


/tmp/ipykernel_1090/3777785277.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(



Total parameters: 12,094,862
Trainable parameters: 12,094,862
Parameters with NaN/Inf: 0

✓ Device restored.
✓ Fresh model created.
✓ Model parameters are finite.
CELL 39 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 40 — STABLE TRAINING CONFIGURATION")
print("=" * 75)

# ----------------------------------------------------
# Training hyperparameters
# ----------------------------------------------------

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
LABEL_SMOOTHING = 0.1

# ----------------------------------------------------
# Loss
# ----------------------------------------------------

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=LABEL_SMOOTHING
)

# ----------------------------------------------------
# Optimizer
# ----------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ----------------------------------------------------
# Scheduler
# ----------------------------------------------------

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

print(f"Learning rate: {LEARNING_RATE}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Gradient clipping: {GRAD_CLIP_NORM}")
print(f"Label smoothing: {LABEL_SMOOTHING}")
print(f"Loss: {criterion.__class__.__name__}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Scheduler: {scheduler.__class__.__name__}")

# ----------------------------------------------------
# Verify optimizer parameters
# ----------------------------------------------------

assert len(optimizer.param_groups) == 1
assert optimizer.param_groups[0]["lr"] == LEARNING_RATE

print()
print("✓ Loss configured.")
print("✓ AdamW configured.")
print("✓ ReduceLROnPlateau configured.")
print("✓ Gradient clipping limit configured.")
print()
print("=" * 75)
print("CELL 40 COMPLETE")
print("=" * 75)

CELL 40 — STABLE TRAINING CONFIGURATION
Learning rate: 0.0001
Weight decay: 0.0001
Gradient clipping: 1.0
Label smoothing: 0.1
Loss: CrossEntropyLoss
Optimizer: AdamW
Scheduler: ReduceLROnPlateau

✓ Loss configured.
✓ AdamW configured.
✓ ReduceLROnPlateau configured.
✓ Gradient clipping limit configured.

CELL 40 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 41 — CONTROLLED OPTIMIZER STEP TEST")
print("=" * 75)

model.train()

# ----------------------------------------------------
# Get a fresh batch
# ----------------------------------------------------

batch = next(iter(train_loader))

object_ids = batch["object_ids"].to(device)
spatial = batch["spatial"].to(device)
confidence = batch["confidence"].to(device)
object_mask = batch["object_mask"].to(device)
decoder_input = batch["decoder_input"].to(device)
target = batch["target"].to(device)

print("Fresh batch loaded:")
print(f"  object_ids:    {tuple(object_ids.shape)}")
print(f"  spatial:       {tuple(spatial.shape)}")
print(f"  confidence:    {tuple(confidence.shape)}")
print(f"  object_mask:   {tuple(object_mask.shape)}")
print(f"  decoder_input: {tuple(decoder_input.shape)}")
print(f"  target:        {tuple(target.shape)}")

# ----------------------------------------------------
# Clear gradients
# ----------------------------------------------------

optimizer.zero_grad(set_to_none=True)

# ----------------------------------------------------
# Forward pass
# ----------------------------------------------------

logits = model(
    object_ids,
    spatial,
    confidence,
    object_mask,
    decoder_input
)

loss = criterion(
    logits.reshape(-1, logits.size(-1)),
    target.reshape(-1)
)

print()
print(f"Logits shape: {tuple(logits.shape)}")
print(f"Loss BEFORE backward: {loss.item():.6f}")

assert torch.isfinite(loss), "Loss is not finite before backward."

# ----------------------------------------------------
# Backward pass
# ----------------------------------------------------

loss.backward()

# ----------------------------------------------------
# Gradient diagnostics
# ----------------------------------------------------

gradient_count = 0
finite_gradients = True
max_gradient = 0.0

for param in model.parameters():
    if param.grad is not None:
        gradient_count += 1

        if not torch.isfinite(param.grad).all():
            finite_gradients = False

        current_max = param.grad.detach().abs().max().item()

        if current_max > max_gradient:
            max_gradient = current_max

assert finite_gradients, "Non-finite gradient detected."

# ----------------------------------------------------
# Gradient clipping
# ----------------------------------------------------

grad_norm_before = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=GRAD_CLIP_NORM
)

print(f"Gradient norm before clipping: {float(grad_norm_before):.6f}")
print(f"Maximum gradient value:         {max_gradient:.6f}")
print(f"Parameters with gradients:      {gradient_count}")
print(f"Gradients finite:                {finite_gradients}")

# ----------------------------------------------------
# Optimizer step
# ----------------------------------------------------

optimizer.step()

# ----------------------------------------------------
# Verify model parameters
# ----------------------------------------------------

bad_parameters = []

for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        bad_parameters.append(name)

print()
print(f"Parameters with NaN/Inf after step: {len(bad_parameters)}")

assert len(bad_parameters) == 0, (
    f"Non-finite parameters detected: {bad_parameters[:10]}"
)

# ----------------------------------------------------
# Fresh forward pass after optimizer step
# ----------------------------------------------------

model.eval()

with torch.no_grad():

    fresh_batch = next(iter(train_loader))

    fresh_object_ids = fresh_batch["object_ids"].to(device)
    fresh_spatial = fresh_batch["spatial"].to(device)
    fresh_confidence = fresh_batch["confidence"].to(device)
    fresh_object_mask = fresh_batch["object_mask"].to(device)
    fresh_decoder_input = fresh_batch["decoder_input"].to(device)
    fresh_target = fresh_batch["target"].to(device)

    fresh_logits = model(
        fresh_object_ids,
        fresh_spatial,
        fresh_confidence,
        fresh_object_mask,
        fresh_decoder_input
    )

    fresh_loss = criterion(
        fresh_logits.reshape(-1, fresh_logits.size(-1)),
        fresh_target.reshape(-1)
    )

print(f"Fresh-batch loss AFTER step: {fresh_loss.item():.6f}")

assert torch.isfinite(fresh_loss), (
    "Fresh-batch loss became non-finite after optimizer step."
)

print()
print("✓ Forward pass successful.")
print("✓ Backward pass successful.")
print("✓ Gradients are finite.")
print("✓ Gradients were clipped.")
print("✓ Optimizer step successful.")
print("✓ Model parameters remain finite.")
print("✓ Fresh-batch loss remains finite.")

print("=" * 75)
print("CELL 41 COMPLETE")
print("=" * 75)

CELL 41 — CONTROLLED OPTIMIZER STEP TEST
Fresh batch loaded:
  object_ids:    (64, 10)
  spatial:       (64, 10, 9)
  confidence:    (64, 10)
  object_mask:   (64, 10)
  decoder_input: (64, 31)
  target:        (64, 31)

Logits shape: (64, 31, 12174)
Loss BEFORE backward: 9.548181
Gradient norm before clipping: 5.218484
Maximum gradient value:         0.449256
Parameters with gradients:      106
Gradients finite:                True

Parameters with NaN/Inf after step: 0
Fresh-batch loss AFTER step: 9.134203

✓ Forward pass successful.
✓ Backward pass successful.
✓ Gradients are finite.
✓ Gradients were clipped.
✓ Optimizer step successful.
✓ Model parameters remain finite.
✓ Fresh-batch loss remains finite.
CELL 41 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 42 — SHORT TRAINING SANITY RUN")
print("=" * 75)

model.train()

# ----------------------------------------------------
# Short controlled run
# ----------------------------------------------------

SANITY_STEPS = 100
running_loss = 0.0

for step, batch in enumerate(train_loader):

    if step >= SANITY_STEPS:
        break

    object_ids = batch["object_ids"].to(device)
    spatial = batch["spatial"].to(device)
    confidence = batch["confidence"].to(device)
    object_mask = batch["object_mask"].to(device)
    decoder_input = batch["decoder_input"].to(device)
    target = batch["target"].to(device)

    optimizer.zero_grad(set_to_none=True)

    # ------------------------------------------------
    # Forward
    # ------------------------------------------------

    logits = model(
        object_ids,
        spatial,
        confidence,
        object_mask,
        decoder_input
    )

    loss = criterion(
        logits.reshape(-1, logits.size(-1)),
        target.reshape(-1)
    )

    assert torch.isfinite(loss), (
        f"Non-finite loss at step {step + 1}: {loss.item()}"
    )

    # ------------------------------------------------
    # Backward
    # ------------------------------------------------

    loss.backward()

    # ------------------------------------------------
    # Gradient clipping
    # ------------------------------------------------

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=GRAD_CLIP_NORM
    )

    assert torch.isfinite(grad_norm), (
        f"Non-finite gradient norm at step {step + 1}"
    )

    # ------------------------------------------------
    # Optimizer step
    # ------------------------------------------------

    optimizer.step()

    # ------------------------------------------------
    # Parameter safety check
    # ------------------------------------------------

    for name, param in model.named_parameters():
        assert torch.isfinite(param).all(), (
            f"Non-finite parameter detected at step {step + 1}: {name}"
        )

    running_loss += loss.item()

    # ------------------------------------------------
    # Progress
    # ------------------------------------------------

    if (step + 1) % 10 == 0:
        average_loss = running_loss / (step + 1)

        print(
            f"Step {step + 1:3d}/{SANITY_STEPS} | "
            f"Loss: {loss.item():.4f} | "
            f"Average: {average_loss:.4f} | "
            f"GradNorm: {float(grad_norm):.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

# ----------------------------------------------------
# Final statistics
# ----------------------------------------------------

steps_done = step + 1
average_loss = running_loss / steps_done

print()
print(f"Steps completed: {steps_done}")
print(f"Average training loss: {average_loss:.6f}")

if torch.cuda.is_available():
    print(
        f"Peak GPU memory: "
        f"{torch.cuda.max_memory_allocated() / (1024 ** 3):.2f} GB"
    )

print()
print("✓ Training steps completed.")
print("✓ Loss remained finite.")
print("✓ Gradients remained finite.")
print("✓ Model parameters remained finite.")

print("=" * 75)
print("CELL 42 COMPLETE")
print("=" * 75)

CELL 42 — SHORT TRAINING SANITY RUN
Step  10/100 | Loss: 7.3070 | Average: 8.1930 | GradNorm: 13.9921 | LR: 1.00e-04
Step  20/100 | Loss: 6.4274 | Average: 7.5224 | GradNorm: 8.6608 | LR: 1.00e-04
Step  30/100 | Loss: 5.8459 | Average: 7.0542 | GradNorm: 4.8261 | LR: 1.00e-04
Step  40/100 | Loss: 5.6829 | Average: 6.7316 | GradNorm: 4.8987 | LR: 1.00e-04
Step  50/100 | Loss: 5.6429 | Average: 6.5094 | GradNorm: 2.9842 | LR: 1.00e-04
Step  60/100 | Loss: 5.6541 | Average: 6.3530 | GradNorm: 2.8406 | LR: 1.00e-04
Step  70/100 | Loss: 5.4826 | Average: 6.2224 | GradNorm: 2.3029 | LR: 1.00e-04
Step  80/100 | Loss: 5.3988 | Average: 6.1242 | GradNorm: 2.5380 | LR: 1.00e-04
Step  90/100 | Loss: 5.3166 | Average: 6.0354 | GradNorm: 2.7254 | LR: 1.00e-04
Step 100/100 | Loss: 5.4588 | Average: 5.9586 | GradNorm: 2.4479 | LR: 1.00e-04

Steps completed: 101
Average training loss: 5.899639
Peak GPU memory: 0.97 GB

✓ Training steps completed.
✓ Loss remained finite.
✓ Gradients remained finite.
✓ 

In [ ]:
import inspect

print("=" * 75)
print("VERIFYING LIVE ObjectAwareTransformer.forward()")
print("=" * 75)

print(inspect.signature(ObjectAwareTransformer.forward))

print()
print(inspect.getsource(ObjectAwareTransformer.forward))

print("=" * 75)
print("FORWARD SIGNATURE CHECK COMPLETE")
print("=" * 75)

VERIFYING LIVE ObjectAwareTransformer.forward()
(self, object_ids, spatial, confidence, object_mask, decoder_input)

    def forward(
        self,
        object_ids,
        spatial,
        confidence,
        object_mask,
        decoder_input
    ):

        # ====================================================
        # OBJECT ENCODER
        # ====================================================

        object_tokens = self.object_encoder(
            object_ids,
            spatial,
            confidence
        )

        # Add object-slot positional information
        object_tokens = self.object_position(
            object_tokens
        )

        # ====================================================
        # TRANSFORMER ENCODER
        # ====================================================

        memory = self.transformer_encoder(
            object_tokens,
            src_key_padding_mask=~object_mask
        )

        # ==========================================

In [ ]:
print("=" * 75)
print("CELL 44 — CREATING VALIDATION AND TEST DATALOADERS")
print("=" * 75)

# ----------------------------------------------------
# Create validation DataLoader
# ----------------------------------------------------

val_loader = DataLoader(
    val_dataset,
    batch_size=train_loader.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# ----------------------------------------------------
# Create test DataLoader
# ----------------------------------------------------

test_loader = DataLoader(
    test_dataset,
    batch_size=train_loader.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# ----------------------------------------------------
# Verify loaders
# ----------------------------------------------------

print(f"Train batches:      {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")
print(f"Test batches:       {len(test_loader):,}")

# Check one batch from each
val_batch = next(iter(val_loader))
test_batch = next(iter(test_loader))

print()
print("Validation batch:")
print(f"  object_ids:    {tuple(val_batch['object_ids'].shape)}")
print(f"  spatial:       {tuple(val_batch['spatial'].shape)}")
print(f"  confidence:    {tuple(val_batch['confidence'].shape)}")
print(f"  object_mask:   {tuple(val_batch['object_mask'].shape)}")
print(f"  decoder_input: {tuple(val_batch['decoder_input'].shape)}")
print(f"  target:        {tuple(val_batch['target'].shape)}")

print()
print("Test batch:")
print(f"  object_ids:    {tuple(test_batch['object_ids'].shape)}")
print(f"  spatial:       {tuple(test_batch['spatial'].shape)}")
print(f"  confidence:    {tuple(test_batch['confidence'].shape)}")
print(f"  object_mask:   {tuple(test_batch['object_mask'].shape)}")
print(f"  decoder_input: {tuple(test_batch['decoder_input'].shape)}")
print(f"  target:        {tuple(test_batch['target'].shape)}")

print()
print("✓ Validation DataLoader created.")
print("✓ Test DataLoader created.")
print("✓ Validation batch verified.")
print("✓ Test batch verified.")

print("=" * 75)
print("CELL 44 COMPLETE")
print("=" * 75)

CELL 44 — CREATING VALIDATION AND TEST DATALOADERS
Train batches:      2,493
Validation batches: 312
Test batches:       312

Validation batch:
  object_ids:    (64, 10)
  spatial:       (64, 10, 9)
  confidence:    (64, 10)
  object_mask:   (64, 10)
  decoder_input: (64, 31)
  target:        (64, 31)

Test batch:
  object_ids:    (64, 10)
  spatial:       (64, 10, 9)
  confidence:    (64, 10)
  object_mask:   (64, 10)
  decoder_input: (64, 31)
  target:        (64, 31)

✓ Validation DataLoader created.
✓ Test DataLoader created.
✓ Validation batch verified.
✓ Test batch verified.
CELL 44 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 45 — FINAL PRE-TRAINING CHECK")
print("=" * 75)

print("MODEL")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Device: {device}")

print()
print("DATA")
print(f"  Train captions: {len(train_dataset):,}")
print(f"  Validation captions: {len(val_dataset):,}")
print(f"  Test captions: {len(test_dataset):,}")
print(f"  Train batches: {len(train_loader):,}")
print(f"  Validation batches: {len(val_loader):,}")
print(f"  Test batches: {len(test_loader):,}")

print()
print("ARCHITECTURE")
print(f"  D_MODEL: {D_MODEL}")
print(f"  NHEAD: {NHEAD}")
print(f"  Encoder layers: {NUM_ENCODER_LAYERS}")
print(f"  Decoder layers: {NUM_DECODER_LAYERS}")
print(f"  Feedforward: {DIM_FEEDFORWARD}")
print(f"  Dropout: {DROPOUT}")
print(f"  Max caption length: {CAPTION_LENGTH}")
print(f"  Object tokens: {TOP_K}")

print()
print("TRAINING")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  Gradient clipping: {GRAD_CLIP_NORM}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"  Maximum epochs: {NUM_EPOCHS}")

print()
print("RUNTIME OBJECTS")

required_objects = [
    "torch",
    "ObjectAwareTransformer",
    "model",
    "optimizer",
    "criterion",
    "scheduler",
    "train_dataset",
    "val_dataset",
    "test_dataset",
    "train_loader",
    "val_loader",
    "test_loader"
]

missing = [
    name for name in required_objects
    if name not in globals()
]

print(f"  Missing objects: {missing}")

assert len(missing) == 0, (
    f"Missing required objects: {missing}"
)

# ----------------------------------------------------
# Model parameter safety
# ----------------------------------------------------

bad_parameters = [
    name
    for name, param in model.named_parameters()
    if not torch.isfinite(param).all()
]

print(f"  Non-finite model parameters: {len(bad_parameters)}")

assert len(bad_parameters) == 0

print()
print("✓ All required objects exist.")
print("✓ DataLoaders exist.")
print("✓ Model parameters are finite.")
print("✓ Runtime is ready for fresh initialization.")

print("=" * 75)
print("CELL 45 COMPLETE")
print("=" * 75)

CELL 45 — FINAL PRE-TRAINING CHECK
MODEL
  Parameters: 12,094,862
  Device: cuda

DATA
  Train captions: 159,494
  Validation captions: 19,935
  Test captions: 19,940
  Train batches: 2,493
  Validation batches: 312
  Test batches: 312

ARCHITECTURE
  D_MODEL: 256
  NHEAD: 8
  Encoder layers: 3
  Decoder layers: 3
  Feedforward: 1024
  Dropout: 0.2
  Max caption length: 32
  Object tokens: 10

TRAINING
  Learning rate: 0.0001
  Weight decay: 0.0001
  Label smoothing: 0.1
  Gradient clipping: 1.0
  Early stopping patience: 4
  Maximum epochs: 20

RUNTIME OBJECTS
  Missing objects: []
  Non-finite model parameters: 0

✓ All required objects exist.
✓ DataLoaders exist.
✓ Model parameters are finite.
✓ Runtime is ready for fresh initialization.
CELL 45 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 46 — FRESH MODEL INITIALIZATION FOR REAL TRAINING")
print("=" * 75)

# ----------------------------------------------------
# Fresh model
# ----------------------------------------------------

model = ObjectAwareTransformer(
    object_vocab_size=len(object_to_id),
    caption_vocab_size=len(caption_to_id),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_caption_length=CAPTION_LENGTH
).to(device)

# ----------------------------------------------------
# Fresh loss
# ----------------------------------------------------

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=LABEL_SMOOTHING
)

# ----------------------------------------------------
# Fresh optimizer
# ----------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ----------------------------------------------------
# Fresh scheduler
# ----------------------------------------------------

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

# ----------------------------------------------------
# Verify fresh initialization
# ----------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

bad_parameters = [
    name
    for name, param in model.named_parameters()
    if not torch.isfinite(param).all()
]

current_lr = optimizer.param_groups[0]["lr"]

print(f"Device: {device}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Initial learning rate: {current_lr:.2e}")
print(f"Parameters with NaN/Inf: {len(bad_parameters)}")

assert total_params == 12_094_862
assert trainable_params == 12_094_862
assert current_lr == LEARNING_RATE
assert len(bad_parameters) == 0

# ----------------------------------------------------
# Verify optimizer state is fresh
# ----------------------------------------------------

optimizer_state_entries = len(optimizer.state)

print(f"Optimizer state entries: {optimizer_state_entries}")

assert optimizer_state_entries == 0

print()
print("✓ Fresh model initialized.")
print("✓ Fresh loss initialized.")
print("✓ Fresh optimizer initialized.")
print("✓ Fresh scheduler initialized.")
print("✓ All parameters are finite.")
print("✓ Optimizer has no previous state.")
print("✓ Model is ready for REAL training.")

print("=" * 75)
print("CELL 46 COMPLETE")
print("=" * 75)

CELL 46 — FRESH MODEL INITIALIZATION FOR REAL TRAINING


/tmp/ipykernel_1090/3777785277.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(


Device: cuda
Total parameters: 12,094,862
Trainable parameters: 12,094,862
Initial learning rate: 1.00e-04
Parameters with NaN/Inf: 0
Optimizer state entries: 0

✓ Fresh model initialized.
✓ Fresh loss initialized.
✓ Fresh optimizer initialized.
✓ Fresh scheduler initialized.
✓ All parameters are finite.
✓ Optimizer has no previous state.
✓ Model is ready for REAL training.
CELL 46 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 47 — REAL TRAINING LOOP DEFINITION")
print("=" * 75)

# ----------------------------------------------------
# Training function
# ----------------------------------------------------

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_batches = 0

    for batch in loader:

        object_ids = batch["object_ids"].to(device)
        spatial = batch["spatial"].to(device)
        confidence = batch["confidence"].to(device)
        object_mask = batch["object_mask"].to(device)
        decoder_input = batch["decoder_input"].to(device)
        target = batch["target"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            object_ids,
            spatial,
            confidence,
            object_mask,
            decoder_input
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            target.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss detected: {loss.item()}"
            )

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_NORM
        )

        if not torch.isfinite(grad_norm):
            raise RuntimeError(
                f"Non-finite gradient norm detected: {float(grad_norm)}"
            )

        optimizer.step()

        # Verify parameters after every optimizer step
        for name, param in model.named_parameters():
            if not torch.isfinite(param).all():
                raise RuntimeError(
                    f"Non-finite parameter detected: {name}"
                )

        total_loss += loss.item()
        total_batches += 1

    return total_loss / total_batches


# ----------------------------------------------------
# Validation function
# ----------------------------------------------------

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    for batch in loader:

        object_ids = batch["object_ids"].to(device)
        spatial = batch["spatial"].to(device)
        confidence = batch["confidence"].to(device)
        object_mask = batch["object_mask"].to(device)
        decoder_input = batch["decoder_input"].to(device)
        target = batch["target"].to(device)

        logits = model(
            object_ids,
            spatial,
            confidence,
            object_mask,
            decoder_input
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            target.reshape(-1)
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite validation loss detected: {loss.item()}"
            )

        total_loss += loss.item()
        total_batches += 1

    return total_loss / total_batches


# ----------------------------------------------------
# Training history
# ----------------------------------------------------

training_history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "learning_rate": []
}

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

print("✓ train_one_epoch() defined.")
print("✓ validate_one_epoch() defined.")
print("✓ Training history initialized.")
print("✓ Best-model tracking initialized.")
print("✓ Early-stopping tracking initialized.")

print()
print("The model has NOT been trained by this cell.")

print("=" * 75)
print("CELL 47 COMPLETE")
print("=" * 75)

CELL 47 — REAL TRAINING LOOP DEFINITION
✓ train_one_epoch() defined.
✓ validate_one_epoch() defined.
✓ Training history initialized.
✓ Best-model tracking initialized.
✓ Early-stopping tracking initialized.

The model has NOT been trained by this cell.
CELL 47 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 48 — REAL TRANSFORMER TRAINING")
print("=" * 75)

# ----------------------------------------------------
# Checkpoint directory
# ----------------------------------------------------

CHECKPOINT_DIR = (
    DRIVE_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "Checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best_object_aware_transformer.pt"
)

print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Best checkpoint:      {BEST_CHECKPOINT_PATH}")

# ----------------------------------------------------
# Training
# ----------------------------------------------------

print()
print(f"Maximum epochs: {NUM_EPOCHS}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"Training batches/epoch: {len(train_loader):,}")
print(f"Validation batches/epoch: {len(val_loader):,}")
print()

for epoch in range(1, NUM_EPOCHS + 1):

    print("=" * 75)
    print(f"EPOCH {epoch}/{NUM_EPOCHS}")
    print("=" * 75)

    # ------------------------------------------------
    # Training
    # ------------------------------------------------

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    # ------------------------------------------------
    # Validation
    # ------------------------------------------------

    val_loss = validate_one_epoch(
        model,
        val_loader,
        criterion,
        device
    )

    # ------------------------------------------------
    # Current learning rate
    # ------------------------------------------------

    current_lr = optimizer.param_groups[0]["lr"]

    # ------------------------------------------------
    # Record history
    # ------------------------------------------------

    training_history["epoch"].append(epoch)
    training_history["train_loss"].append(train_loss)
    training_history["val_loss"].append(val_loss)
    training_history["learning_rate"].append(current_lr)

    # ------------------------------------------------
    # Scheduler
    # ------------------------------------------------

    scheduler.step(val_loss)

    # ------------------------------------------------
    # Check improvement
    # ------------------------------------------------

    improved = val_loss < best_val_loss

    if improved:

        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_loss": best_val_loss,
                "training_history": training_history,
                "config": {
                    "d_model": D_MODEL,
                    "nhead": NHEAD,
                    "num_encoder_layers": NUM_ENCODER_LAYERS,
                    "num_decoder_layers": NUM_DECODER_LAYERS,
                    "dim_feedforward": DIM_FEEDFORWARD,
                    "dropout": DROPOUT,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "label_smoothing": LABEL_SMOOTHING,
                    "gradient_clip_norm": GRAD_CLIP_NORM,
                    "max_caption_length": CAPTION_LENGTH,
                    "top_k": TOP_K
                }
            },
            BEST_CHECKPOINT_PATH
        )

        status = "✓ NEW BEST"

    else:

        epochs_without_improvement += 1
        status = (
            f"No improvement "
            f"({epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE})"
        )

    # ------------------------------------------------
    # Epoch report
    # ------------------------------------------------

    print()
    print(f"Train loss:       {train_loss:.6f}")
    print(f"Validation loss:  {val_loss:.6f}")
    print(f"Best val loss:    {best_val_loss:.6f}")
    print(f"Learning rate:    {current_lr:.2e}")
    print(f"Status:           {status}")

    # ------------------------------------------------
    # Early stopping
    # ------------------------------------------------

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:

        print()
        print(
            f"Early stopping triggered after epoch {epoch}."
        )
        break

# ----------------------------------------------------
# Final training summary
# ----------------------------------------------------

print()
print("=" * 75)
print("TRAINING COMPLETE")
print("=" * 75)

print(f"Epochs completed:     {len(training_history['epoch'])}")
print(f"Best epoch:           {best_epoch}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Checkpoint exists:    {BEST_CHECKPOINT_PATH.exists()}")

print()
print("✓ Training finished.")
print("✓ Best checkpoint saved.")
print("✓ Test set was NOT used during training.")

print("=" * 75)
print("CELL 48 COMPLETE")
print("=" * 75)

CELL 48 — REAL TRANSFORMER TRAINING
Checkpoint directory: /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/Checkpoints
Best checkpoint:      /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/Checkpoints/best_object_aware_transformer.pt

Maximum epochs: 20
Early stopping patience: 4
Training batches/epoch: 2,493
Validation batches/epoch: 312

EPOCH 1/20

Train loss:       4.559372
Validation loss:  4.156922
Best val loss:    4.156922
Learning rate:    1.00e-04
Status:           ✓ NEW BEST
EPOCH 2/20

Train loss:       4.153700
Validation loss:  4.001113
Best val loss:    4.001113
Learning rate:    1.00e-04
Status:           ✓ NEW BEST
EPOCH 3/20

Train loss:       4.025409
Validation loss:  3.917657
Best val loss:    3.917657
Learning rate:    1.00e-04
Status:           ✓ NEW BEST
EPOCH 4/20

Train loss:       3.946283
Validation loss:  3.871551
Best val loss:    3.871551
Learning rate:    1.00e-04
Status:           ✓ NEW BEST
EPOCH 5/20

Train l

In [ ]:
print("=" * 75)
print("CELL 49 — LOAD BEST CHECKPOINT")
print("=" * 75)

# ----------------------------------------------------
# Load best checkpoint
# ----------------------------------------------------

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

# ----------------------------------------------------
# Restore training information
# ----------------------------------------------------

loaded_epoch = checkpoint["epoch"]
loaded_best_val_loss = checkpoint["best_val_loss"]

print(f"Checkpoint epoch:       {loaded_epoch}")
print(f"Checkpoint val loss:    {loaded_best_val_loss:.6f}")
print(f"Model parameters:       {sum(p.numel() for p in model.parameters()):,}")

# ----------------------------------------------------
# Verify parameters
# ----------------------------------------------------

bad_parameters = [
    name
    for name, param in model.named_parameters()
    if not torch.isfinite(param).all()
]

print(f"Parameters with NaN/Inf: {len(bad_parameters)}")

assert len(bad_parameters) == 0

# ----------------------------------------------------
# Verify checkpoint history
# ----------------------------------------------------

checkpoint_history = checkpoint.get(
    "training_history",
    None
)

if checkpoint_history is not None:
    print()
    print("Training history:")
    print(f"  Epochs recorded: {len(checkpoint_history['epoch'])}")
    print(f"  Best validation loss: {min(checkpoint_history['val_loss']):.6f}")

print()
print("✓ Best checkpoint loaded successfully.")
print("✓ Model is in evaluation mode.")
print("✓ All model parameters are finite.")

print("=" * 75)
print("CELL 49 COMPLETE")
print("=" * 75)

CELL 49 — LOAD BEST CHECKPOINT
Checkpoint epoch:       20
Checkpoint val loss:    3.633360
Model parameters:       12,094,862
Parameters with NaN/Inf: 0

Training history:
  Epochs recorded: 20
  Best validation loss: 3.633360

✓ Best checkpoint loaded successfully.
✓ Model is in evaluation mode.
✓ All model parameters are finite.
CELL 49 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 50 — GREEDY CAPTION GENERATION TEST")
print("=" * 75)

# ----------------------------------------------------
# Special token IDs
# ----------------------------------------------------

START_ID = caption_to_id["<start>"]
END_ID = caption_to_id["<end>"]
PAD_ID = caption_to_id["<pad>"]

print(f"START ID: {START_ID}")
print(f"END ID:   {END_ID}")
print(f"PAD ID:   {PAD_ID}")

# ----------------------------------------------------
# Greedy decoding function
# ----------------------------------------------------

@torch.no_grad()
def generate_caption(
    model,
    object_ids,
    spatial,
    confidence,
    object_mask,
    max_length=CAPTION_LENGTH - 1
):
    model.eval()

    # Ensure batch dimension exists
    if object_ids.dim() == 1:
        object_ids = object_ids.unsqueeze(0)
        spatial = spatial.unsqueeze(0)
        confidence = confidence.unsqueeze(0)
        object_mask = object_mask.unsqueeze(0)

    # Start with <start>
    decoder_input = torch.full(
        (object_ids.size(0), 1),
        START_ID,
        dtype=torch.long,
        device=object_ids.device
    )

    finished = torch.zeros(
        object_ids.size(0),
        dtype=torch.bool,
        device=object_ids.device
    )

    for _ in range(max_length):

        logits = model(
            object_ids,
            spatial,
            confidence,
            object_mask,
            decoder_input
        )

        # Prediction for the newest token
        next_token = logits[:, -1, :].argmax(
            dim=-1,
            keepdim=True
        )

        # Keep END for sequences that already finished
        next_token = torch.where(
            finished.unsqueeze(1),
            torch.full_like(next_token, END_ID),
            next_token
        )

        decoder_input = torch.cat(
            [decoder_input, next_token],
            dim=1
        )

        finished = finished | (
            next_token.squeeze(1) == END_ID
        )

        if finished.all():
            break

    return decoder_input


# ----------------------------------------------------
# Test on one batch
# ----------------------------------------------------

test_batch = next(iter(test_loader))

test_object_ids = test_batch["object_ids"].to(device)
test_spatial = test_batch["spatial"].to(device)
test_confidence = test_batch["confidence"].to(device)
test_object_mask = test_batch["object_mask"].to(device)

generated_ids = generate_caption(
    model,
    test_object_ids,
    test_spatial,
    test_confidence,
    test_object_mask
)

print()
print(f"Generated tensor shape: {tuple(generated_ids.shape)}")

assert generated_ids.dim() == 2
assert generated_ids.size(0) == test_object_ids.size(0)
assert generated_ids.size(1) <= CAPTION_LENGTH

print()
print("First generated token IDs:")
print(generated_ids[0].detach().cpu().tolist())

# ----------------------------------------------------
# Convert first generated caption to words
# ----------------------------------------------------

generated_tokens = []

for token_id in generated_ids[0].detach().cpu().tolist():

    if token_id == START_ID:
        continue

    if token_id == END_ID:
        break

    if token_id == PAD_ID:
        continue

    generated_tokens.append(
        id_to_caption.get(
            int(token_id),
            "<unknown>"
        )
    )

generated_caption = " ".join(generated_tokens)

print()
print("Generated caption:")
print(generated_caption)

print()
print("✓ Greedy decoder executed.")
print("✓ Generated tensor has valid dimensions.")
print("✓ Caption successfully converted to words.")

print("=" * 75)
print("CELL 50 COMPLETE")
print("=" * 75)

CELL 50 — GREEDY CAPTION GENERATION TEST
START ID: 1
END ID:   2
PAD ID:   0

Generated tensor shape: (64, 31)

First generated token IDs:
[1, 4, 21, 9, 62, 52, 6, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Generated caption:
a dog is running through the <unk>

✓ Greedy decoder executed.
✓ Generated tensor has valid dimensions.
✓ Caption successfully converted to words.
CELL 50 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 51 — QUALITATIVE CAPTION INSPECTION")
print("=" * 75)

# ----------------------------------------------------
# Select first 10 test images
# ----------------------------------------------------

NUM_SAMPLES = 10

sample_indices = []

seen_images = set()

for idx in range(len(test_dataset)):

    image_id = test_dataset[idx]["image_id"]

    if image_id not in seen_images:
        seen_images.add(image_id)
        sample_indices.append(idx)

    if len(sample_indices) >= NUM_SAMPLES:
        break

# ----------------------------------------------------
# Generate and display captions
# ----------------------------------------------------

for sample_number, idx in enumerate(sample_indices, start=1):

    item = test_dataset[idx]

    image_id = item["image_id"]

    object_ids = item["object_ids"].unsqueeze(0).to(device)
    spatial = item["spatial"].unsqueeze(0).to(device)
    confidence = item["confidence"].unsqueeze(0).to(device)
    object_mask = item["object_mask"].unsqueeze(0).to(device)

    generated_ids = generate_caption(
        model,
        object_ids,
        spatial,
        confidence,
        object_mask
    )

    generated_tokens = []

    for token_id in generated_ids[0].detach().cpu().tolist():

        if token_id == START_ID:
            continue

        if token_id == END_ID:
            break

        if token_id == PAD_ID:
            continue

        generated_tokens.append(
            id_to_caption.get(
                int(token_id),
                "<unknown>"
            )
        )

    generated_text = " ".join(generated_tokens)

    # ------------------------------------------------
    # Ground-truth captions for this image
    # ------------------------------------------------

    image_caption_indices = [
        i
        for i, image in enumerate(test_dataset.image_ids)
        if image == image_id
    ]

    references = []

    for ref_idx in image_caption_indices[:5]:

        caption_ids = test_dataset[ref_idx]["target"]

        words = []

        for token_id in caption_ids.tolist():

            if token_id == PAD_ID:
                continue

            if token_id == END_ID:
                break

            token = id_to_caption.get(
                int(token_id),
                "<unknown>"
            )

            if token not in ("<start>", "<pad>"):
                words.append(token)

        if words:
            references.append(" ".join(words))

    print()
    print("-" * 75)
    print(f"SAMPLE {sample_number}")
    print(f"Image: {image_id}")
    print()
    print(f"GENERATED:")
    print(f"  {generated_text}")
    print()
    print("REFERENCE CAPTIONS:")

    for ref_number, reference in enumerate(references, start=1):
        print(f"  {ref_number}. {reference}")

print()
print("=" * 75)
print("CELL 51 COMPLETE")
print("=" * 75)

CELL 51 — QUALITATIVE CAPTION INSPECTION

---------------------------------------------------------------------------
SAMPLE 1
Image: Flickr8K/101654506_8eb26cfb60.jpg

GENERATED:
  a dog is running through the <unk>

REFERENCE CAPTIONS:
  1. a boy jumping in a <unk>
  2. a boy plays in the <unk>
  3. a child plays in a <unk>
  4. a little boy playing in the <unk>
  5. a young boy is jumping through water being sprayed up from the <unk>

---------------------------------------------------------------------------
SAMPLE 2
Image: Flickr8K/1032460886_4a598ed535.jpg

GENERATED:
  a man is standing in front of a building with a large <unk>

REFERENCE CAPTIONS:
  1. a man is playing a saxophone next to a fire <unk>
  2. a man plays saxophone next to a yellow fire <unk>
  3. a man stands by a wall playing a <unk>
  4. man playing musical instrument standing beside a yellow and red fire hydrant
  5. the man is standing next to a fire hydrant and playing a <unk>

-------------------------------

In [ ]:
print("=" * 75)
print("CELL 52 — VERIFY IMAGE/OBJECT/CAPTION ALIGNMENT")
print("=" * 75)

# ----------------------------------------------------
# Check Dataset internals
# ----------------------------------------------------

print("Dataset attributes:")

for name in [
    "image_ids",
    "caption_indices",
    "image_to_object_index",
    "object_image_ids"
]:
    print(
        f"  {name}: "
        f"{'EXISTS' if name in dir(test_dataset) else 'NOT FOUND'}"
    )

# ----------------------------------------------------
# Inspect first 10 samples
# ----------------------------------------------------

print()
print("First 10 test samples:")
print("-" * 75)

for idx in range(10):

    item = test_dataset[idx]

    dataset_image_id = item["image_id"]

    # Try to identify the object row used by the dataset
    object_index = None

    if hasattr(test_dataset, "image_to_object_index"):
        mapping = test_dataset.image_to_object_index

        if isinstance(mapping, dict):
            object_index = mapping.get(dataset_image_id)

    elif hasattr(test_dataset, "image_ids"):
        # Find matching image in object NPZ
        matches = np.where(
            object_image_ids == dataset_image_id
        )[0]

        if len(matches) > 0:
            object_index = int(matches[0])

    print(f"Sample {idx + 1}:")
    print(f"  Dataset image ID: {dataset_image_id}")
    print(f"  Object row index: {object_index}")

    # Object information actually returned by dataset
    valid_objects = item["object_mask"].bool()

    returned_object_ids = (
        item["object_ids"][valid_objects]
        .detach()
        .cpu()
        .tolist()
    )

    returned_object_names = [
        id_to_object.get(int(obj_id), "<unknown>")
        for obj_id in returned_object_ids
    ]

    print(f"  Objects returned: {returned_object_names}")

    # Caption target
    target_tokens = []

    for token_id in item["target"].detach().cpu().tolist():

        if token_id == PAD_ID:
            continue

        if token_id == END_ID:
            break

        if token_id == START_ID:
            continue

        target_tokens.append(
            id_to_caption.get(
                int(token_id),
                "<unknown>"
            )
        )

    print(f"  Target caption: {' '.join(target_tokens)}")
    print()

print("=" * 75)
print("CELL 52 COMPLETE")
print("=" * 75)

CELL 52 — VERIFY IMAGE/OBJECT/CAPTION ALIGNMENT
Dataset attributes:
  image_ids: EXISTS
  caption_indices: EXISTS
  image_to_object_index: NOT FOUND
  object_image_ids: NOT FOUND

First 10 test samples:
---------------------------------------------------------------------------
Sample 1:
  Dataset image ID: Flickr8K/101654506_8eb26cfb60.jpg
  Object row index: 10
  Objects returned: ['dog']
  Target caption: a brown and white dog is running through the <unk>

Sample 2:
  Dataset image ID: Flickr8K/101654506_8eb26cfb60.jpg
  Object row index: 10
  Objects returned: ['dog']
  Target caption: a dog is running in the snow

Sample 3:
  Dataset image ID: Flickr8K/101654506_8eb26cfb60.jpg
  Object row index: 10
  Objects returned: ['dog']
  Target caption: a dog running through <unk>

Sample 4:
  Dataset image ID: Flickr8K/101654506_8eb26cfb60.jpg
  Object row index: 10
  Objects returned: ['dog']
  Target caption: a white and brown dog is running through a snow covered <unk>

Sample 5:
  Dat

In [ ]:
print("=" * 75)
print("CELL 53 — TEST SET OBJECT REPRESENTATION ANALYSIS")
print("=" * 75)

# ----------------------------------------------------
# Count valid objects per test image
# ----------------------------------------------------

object_counts = []

for idx in range(len(test_dataset)):

    item = test_dataset[idx]

    count = int(
        item["object_mask"]
        .sum()
        .item()
    )

    object_counts.append(count)

object_counts = np.array(object_counts)

# ----------------------------------------------------
# Distribution
# ----------------------------------------------------

unique_counts, count_frequencies = np.unique(
    object_counts,
    return_counts=True
)

print("Objects per test caption sample:")
print()

for count, frequency in zip(
    unique_counts,
    count_frequencies
):
    print(
        f"  {count:2d} objects: "
        f"{frequency:,} samples"
    )

print()
print(f"Minimum objects: {object_counts.min()}")
print(f"Maximum objects: {object_counts.max()}")
print(f"Mean objects:    {object_counts.mean():.4f}")
print(f"Median objects:  {np.median(object_counts):.1f}")

# ----------------------------------------------------
# Images with only 1 object
# ----------------------------------------------------

one_object_samples = np.sum(
    object_counts == 1
)

zero_object_samples = np.sum(
    object_counts == 0
)

ten_object_samples = np.sum(
    object_counts == 10
)

print()
print(f"Zero-object samples: {zero_object_samples:,}")
print(f"One-object samples:  {one_object_samples:,}")
print(f"Ten-object samples:  {ten_object_samples:,}")

# ----------------------------------------------------
# Unique test images
# ----------------------------------------------------

unique_test_images = list(
    dict.fromkeys(test_dataset.image_ids)
)

print()
print(f"Unique test images: {len(unique_test_images):,}")

# ----------------------------------------------------
# Object vocabulary coverage
# ----------------------------------------------------

detected_object_ids = set()

for idx in range(len(test_dataset)):

    item = test_dataset[idx]

    valid = item["object_mask"].bool()

    ids = (
        item["object_ids"][valid]
        .detach()
        .cpu()
        .tolist()
    )

    detected_object_ids.update(
        int(x) for x in ids
    )

print(
    f"Unique object classes appearing "
    f"in test set: {len(detected_object_ids):,}"
)

print(
    f"Total object vocabulary classes: "
    f"{len(object_to_id) - 2:,}"
)

print()
print("✓ Test object representation analyzed.")

print("=" * 75)
print("CELL 53 COMPLETE")
print("=" * 75)

CELL 53 — TEST SET OBJECT REPRESENTATION ANALYSIS
Objects per test caption sample:

   0 objects: 5 samples
   1 objects: 1,835 samples
   2 objects: 2,770 samples
   3 objects: 2,060 samples
   4 objects: 1,525 samples
   5 objects: 1,295 samples
   6 objects: 1,030 samples
   7 objects: 950 samples
   8 objects: 680 samples
   9 objects: 625 samples
  10 objects: 7,165 samples

Minimum objects: 0
Maximum objects: 10
Mean objects:    6.1021
Median objects:  6.0

Zero-object samples: 5
One-object samples:  1,835
Ten-object samples:  7,165

Unique test images: 39,874
Unique object classes appearing in test set: 79
Total object vocabulary classes: 80

✓ Test object representation analyzed.
CELL 53 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 54 — OBJECT-CAPTION CONSISTENCY DIAGNOSTIC")
print("=" * 75)

# ----------------------------------------------------
# Test a limited number of unique test images
# ----------------------------------------------------

NUM_DIAGNOSTIC_IMAGES = 100

sample_indices = []

seen_images = set()

for idx in range(len(test_dataset)):

    image_id = test_dataset[idx]["image_id"]

    if image_id not in seen_images:
        seen_images.add(image_id)
        sample_indices.append(idx)

    if len(sample_indices) >= NUM_DIAGNOSTIC_IMAGES:
        break

# ----------------------------------------------------
# Generate captions
# ----------------------------------------------------

object_mentioned_count = 0
total_samples = 0

diagnostic_results = []

for idx in sample_indices:

    item = test_dataset[idx]

    image_id = item["image_id"]

    object_ids = item["object_ids"].unsqueeze(0).to(device)
    spatial = item["spatial"].unsqueeze(0).to(device)
    confidence = item["confidence"].unsqueeze(0).to(device)
    object_mask = item["object_mask"].unsqueeze(0).to(device)

    generated_ids = generate_caption(
        model,
        object_ids,
        spatial,
        confidence,
        object_mask
    )

    # ------------------------------------------------
    # Convert generated IDs to words
    # ------------------------------------------------

    generated_tokens = []

    for token_id in generated_ids[0].detach().cpu().tolist():

        if token_id in (START_ID, PAD_ID):
            continue

        if token_id == END_ID:
            break

        generated_tokens.append(
            id_to_caption.get(
                int(token_id),
                "<unknown>"
            )
        )

    generated_text = " ".join(generated_tokens)

    # ------------------------------------------------
    # Detector object names
    # ------------------------------------------------

    valid_objects = item["object_mask"].bool()

    detected_ids = (
        item["object_ids"][valid_objects]
        .detach()
        .cpu()
        .tolist()
    )

    detected_objects = set(
        id_to_object.get(
            int(object_id),
            "<unknown>"
        )
        for object_id in detected_ids
    )

    # ------------------------------------------------
    # Check whether generated caption mentions
    # any detected object class
    # ------------------------------------------------

    generated_words = set(
        generated_text.lower()
        .replace(".", "")
        .replace(",", "")
        .split()
    )

    matched_objects = sorted(
        obj
        for obj in detected_objects
        if obj.lower() in generated_words
    )

    has_match = len(matched_objects) > 0

    if has_match:
        object_mentioned_count += 1

    total_samples += 1

    diagnostic_results.append(
        (
            image_id,
            detected_objects,
            generated_text,
            matched_objects
        )
    )

# ----------------------------------------------------
# Results
# ----------------------------------------------------

consistency_rate = (
    object_mentioned_count / total_samples
) * 100

print(f"Images analyzed: {total_samples}")
print(
    f"Generated captions mentioning at least "
    f"one detected object: {object_mentioned_count}"
)
print(
    f"Object-caption consistency rate: "
    f"{consistency_rate:.2f}%"
)

# ----------------------------------------------------
# Show first 20 diagnostic examples
# ----------------------------------------------------

print()
print("=" * 75)
print("FIRST 20 EXAMPLES")
print("=" * 75)

for number, (
    image_id,
    detected_objects,
    generated_text,
    matched_objects
) in enumerate(
    diagnostic_results[:20],
    start=1
):

    print()
    print(f"{number}. {image_id}")
    print(
        f"   Detected: "
        f"{', '.join(sorted(detected_objects))}"
    )
    print(
        f"   Generated: "
        f"{generated_text}"
    )
    print(
        f"   Matched: "
        f"{', '.join(matched_objects) if matched_objects else 'NONE'}"
    )

print()
print("=" * 75)
print("CELL 54 COMPLETE")
print("=" * 75)

CELL 54 — OBJECT-CAPTION CONSISTENCY DIAGNOSTIC
Images analyzed: 100
Generated captions mentioning at least one detected object: 33
Object-caption consistency rate: 33.00%

FIRST 20 EXAMPLES

1. Flickr8K/101654506_8eb26cfb60.jpg
   Detected: dog
   Generated: a dog is running through the <unk>
   Matched: dog

2. Flickr8K/1032460886_4a598ed535.jpg
   Detected: person
   Generated: a man is standing in front of a building with a large <unk>
   Matched: NONE

3. Flickr8K/1042590306_95dea0916c.jpg
   Detected: person, potted plant
   Generated: three men are posing for a picture with a woman in a red shirt and a man in a black shirt and a woman in a black shirt and
   Matched: NONE

4. Flickr8K/1057210460_09c6f4c6c1.jpg
   Detected: person, tennis racket
   Generated: a man in a white shirt and black pants is playing <unk>
   Matched: NONE

5. Flickr8K/1067790824_f3cc97239b.jpg
   Detected: dog
   Generated: two dogs are playing in the <unk>
   Matched: NONE

6. Flickr8K/1072153132_53d2bb

In [ ]:
print("=" * 75)
print("CELL 55 — UNK TOKEN DIAGNOSTIC")
print("=" * 75)

# ----------------------------------------------------
# Basic vocabulary checks
# ----------------------------------------------------

print("Special tokens:")
print(f"  <pad>:   {caption_to_id.get('<pad>')}")
print(f"  <start>: {caption_to_id.get('<start>')}")
print(f"  <end>:   {caption_to_id.get('<end>')}")
print(f"  <unk>:   {caption_to_id.get('<unk>')}")

# ----------------------------------------------------
# Check common words from generated/reference captions
# ----------------------------------------------------

words_to_check = [
    "dog",
    "dogs",
    "man",
    "woman",
    "boy",
    "girl",
    "grass",
    "water",
    "snow",
    "ground",
    "street",
    "road",
    "field",
    "beach",
    "building",
    "tree",
    "ball",
    "bat",
    "horse",
    "bicycle",
    "bike",
    "car",
    "playing",
    "running",
    "jumping",
    "standing",
    "walking"
]

print()
print("Vocabulary membership:")
print("-" * 75)

for word in words_to_check:

    token_id = caption_to_id.get(word)

    if token_id is None:
        status = "NOT IN VOCAB"
    else:
        status = f"ID {token_id}"

    print(f"{word:15s} -> {status}")

# ----------------------------------------------------
# Count UNK in ground-truth sequences
# ----------------------------------------------------

unk_id = caption_to_id["<unk>"]

unk_counts = []

for row in caption_sequences:

    count = int(
        (row == unk_id).sum()
    )

    unk_counts.append(count)

unk_counts = np.array(unk_counts)

total_unk = int(unk_counts.sum())
captions_with_unk = int(
    (unk_counts > 0).sum()
)

print()
print("Ground-truth UNK statistics:")
print(f"  Total UNK tokens:        {total_unk:,}")
print(f"  Captions containing UNK: {captions_with_unk:,}")
print(f"  Total captions:          {len(caption_sequences):,}")
print(
    f"  Percentage captions with UNK: "
    f"{100 * captions_with_unk / len(caption_sequences):.2f}%"
)

# ----------------------------------------------------
# Show examples containing UNK
# ----------------------------------------------------

print()
print("Example encoded captions containing <unk>:")
print("-" * 75)

shown = 0

for row in caption_sequences:

    if unk_id not in row:
        continue

    words = []

    for token_id in row:

        token_id = int(token_id)

        if token_id == PAD_ID:
            continue

        if token_id == START_ID:
            continue

        if token_id == END_ID:
            break

        words.append(
            id_to_caption.get(
                token_id,
                "<missing>"
            )
        )

    print("  " + " ".join(words))

    shown += 1

    if shown >= 10:
        break

print()
print("=" * 75)
print("CELL 55 COMPLETE")
print("=" * 75)

CELL 55 — UNK TOKEN DIAGNOSTIC
Special tokens:
  <pad>:   0
  <start>: 1
  <end>:   2
  <unk>:   3

Vocabulary membership:
---------------------------------------------------------------------------
dog             -> ID 21
dogs            -> ID 72
man             -> ID 10
woman           -> ID 14
boy             -> ID 31
girl            -> ID 30
grass           -> ID 74
water           -> ID 40
snow            -> ID 80
ground          -> ID 170
street          -> ID 39
road            -> ID 142
field           -> ID 75
beach           -> ID 84
building        -> ID 86
tree            -> ID 156
ball            -> ID 60
bat             -> ID 721
horse           -> ID 206
bicycle         -> ID 165
bike            -> ID 95
car             -> ID 138
playing         -> ID 34
running         -> ID 62
jumping         -> ID 78
standing        -> ID 33
walking         -> ID 42

Ground-truth UNK statistics:
  Total UNK tokens:        234,767
  Captions containing UNK: 188,647
  Total captions:  

In [ ]:
print("=" * 75)
print("CELL 56 — IDENTIFYING THE WORDS BECOMING <UNK>")
print("=" * 75)

from collections import Counter

# ----------------------------------------------------
# Load processed captions
# ----------------------------------------------------

caption_csv = (
    DRIVE_ROOT
    / "Dataset"
    / "Processed"
    / "Captions"
    / "captions_processed.csv"
)

df_unk = pd.read_csv(caption_csv)

print(f"Caption rows loaded: {len(df_unk):,}")

# ----------------------------------------------------
# Find processed caption column
# ----------------------------------------------------

assert "processed_caption" in df_unk.columns

# ----------------------------------------------------
# Count words that are outside the vocabulary
# ----------------------------------------------------

vocab_words = set(caption_to_id.keys())

oov_counter = Counter()
total_word_count = 0

for caption in df_unk["processed_caption"].fillna(""):

    words = str(caption).split()

    for word in words:

        total_word_count += 1

        if word not in vocab_words:
            oov_counter[word] += 1

# ----------------------------------------------------
# Results
# ----------------------------------------------------

print()
print(f"Total processed words: {total_word_count:,}")
print(f"Unique OOV words:      {len(oov_counter):,}")
print(
    f"Total OOV word tokens: "
    f"{sum(oov_counter.values()):,}"
)

print()
print("TOP 50 WORDS BECOMING <unk>:")
print("-" * 75)

for rank, (word, count) in enumerate(
    oov_counter.most_common(50),
    start=1
):
    print(f"{rank:2d}. {word:25s} {count:8,}")

# ----------------------------------------------------
# Compare expected UNK count
# ----------------------------------------------------

sequence_unk_count = int(
    (caption_sequences == caption_to_id["<unk>"]).sum()
)

csv_oov_count = sum(oov_counter.values())

print()
print("CROSS-CHECK")
print("-" * 75)
print(f"UNK tokens in caption_sequences: {sequence_unk_count:,}")
print(f"OOV tokens found in CSV:         {csv_oov_count:,}")

print()
if sequence_unk_count == csv_oov_count:
    print("✓ UNK count matches the processed-caption OOV count.")
else:
    print(
        "⚠ Counts differ — tokenization/sequence construction "
        "needs further investigation."
    )

print()
print("=" * 75)
print("CELL 56 COMPLETE")
print("=" * 75)

CELL 56 — IDENTIFYING THE WORDS BECOMING <UNK>
Caption rows loaded: 199,369

Total processed words: 2,387,949
Unique OOV words:      18,142
Total OOV word tokens: 236,397

TOP 50 WORDS BECOMING <unk>:
---------------------------------------------------------------------------
 1. water.                       4,302
 2. street.                      4,035
 3. background.                  2,677
 4. 's                           2,628
 5. grass.                       2,411
 6. field.                       2,168
 7. building.                    2,127
 8. beach.                       2,124
 9. ball.                        2,067
10. snow.                        2,026
11. "                            1,720
12. him.                         1,628
13. camera.                      1,547
14. sidewalk.                    1,459
15. wall.                        1,412
16. on.                          1,347
17. mouth.                       1,333
18. park.                        1,330
19. outside.         

In [ ]:
print("=" * 75)
print("CELL 57 — INSPECT CAPTION SEQUENCE CONSTRUCTION")
print("=" * 75)

import inspect

# ----------------------------------------------------
# Search currently defined functions/classes
# ----------------------------------------------------

print("Currently defined caption-related functions:")
print()

for name, obj in list(globals().items()):

    if callable(obj):

        name_lower = name.lower()

        if (
            "caption" in name_lower
            or "vocab" in name_lower
            or "sequence" in name_lower
            or "token" in name_lower
        ):
            try:
                print(f"  {name}")
            except Exception:
                pass

print()
print("Current caption vocabulary size:")
print(f"  {len(caption_to_id):,}")

print()
print("Current processed-caption examples:")
print("-" * 75)

for caption in df_unk["processed_caption"].head(10):
    print(f"  {caption}")

print()
print("=" * 75)
print("CELL 57 COMPLETE")
print("=" * 75)

CELL 57 — INSPECT CAPTION SEQUENCE CONSTRUCTION
Currently defined caption-related functions:

  ObjectCaptionDataset
  CaptionEmbedding
  generate_caption

Current caption vocabulary size:
  12,174

Current processed-caption examples:
---------------------------------------------------------------------------
  a child in a pink dress is climbing up a set of stairs in an entry way.
  a girl going into a wooden building.
  a little girl climbing into a wooden playhouse.
  a little girl climbing the stairs to her playhouse.
  a little girl in a pink dress going into a wooden cabin.
  a black dog and a spotted dog are fighting
  a black dog and a tri-colored dog playing with each other on the road.
  a black dog and a white dog with brown spots are staring at each other in the street.
  two dogs of different breeds looking at each other on the road.
  two dogs on pavement moving toward each other.

CELL 57 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 58 — VOCABULARY vs PROCESSED-CAPTION TOKENIZATION")
print("=" * 75)

# ----------------------------------------------------
# Check representative tokens
# ----------------------------------------------------

test_tokens = [
    "water",
    "water.",
    "street",
    "street.",
    "dog",
    "dog.",
    "grass",
    "grass.",
    "ball",
    "ball.",
    "shirt",
    "shirt,",
    "'s",
    '"',
    ".",
]

print("\nTOKEN MEMBERSHIP")
print("-" * 75)

for token in test_tokens:
    print(f"{token!r:15} -> {'IN VOCAB' if token in caption_to_id else 'OOV'}", end="")

    if token in caption_to_id:
        print(f"   ID={caption_to_id[token]}")
    else:
        print()

# ----------------------------------------------------
# Inspect actual processed-caption tokenization
# ----------------------------------------------------

print("\n")
print("FIRST 10 CAPTIONS — RAW SPLIT TOKENS")
print("-" * 75)

for i, caption in enumerate(df_unk["processed_caption"].head(10), start=1):

    tokens = str(caption).split()

    print(f"\n{i}. {caption}")
    print("   Tokens:", tokens)

# ----------------------------------------------------
# Check punctuation attached to words
# ----------------------------------------------------

punctuation_oov = Counter()

for caption in df_unk["processed_caption"].fillna(""):
    for token in str(caption).split():

        if token not in caption_to_id:

            if token.endswith("."):
                punctuation_oov["trailing_period"] += 1

            if token.endswith(","):
                punctuation_oov["trailing_comma"] += 1

            if token.startswith('"') or token.endswith('"'):
                punctuation_oov["quote"] += 1

            if token == "'s":
                punctuation_oov["possessive_s"] += 1

print("\n")
print("PUNCTUATION-RELATED OOV TOKENS")
print("-" * 75)

for key, value in punctuation_oov.items():
    print(f"{key:20}: {value:,}")

print("\n")
print("=" * 75)
print("CELL 58 COMPLETE")
print("=" * 75)

CELL 58 — VOCABULARY vs PROCESSED-CAPTION TOKENIZATION

TOKEN MEMBERSHIP
---------------------------------------------------------------------------
'water'         -> IN VOCAB   ID=40
'water.'        -> OOV
'street'        -> IN VOCAB   ID=39
'street.'       -> OOV
'dog'           -> IN VOCAB   ID=21
'dog.'          -> OOV
'grass'         -> IN VOCAB   ID=74
'grass.'        -> OOV
'ball'          -> IN VOCAB   ID=60
'ball.'         -> OOV
'shirt'         -> IN VOCAB   ID=23
'shirt,'        -> OOV
"'s"            -> OOV
'"'             -> OOV
'.'             -> OOV


FIRST 10 CAPTIONS — RAW SPLIT TOKENS
---------------------------------------------------------------------------

1. a child in a pink dress is climbing up a set of stairs in an entry way.
   Tokens: ['a', 'child', 'in', 'a', 'pink', 'dress', 'is', 'climbing', 'up', 'a', 'set', 'of', 'stairs', 'in', 'an', 'entry', 'way.']

2. a girl going into a wooden building.
   Tokens: ['a', 'girl', 'going', 'into', 'a', 'wooden', 'bui

In [ ]:
print("=" * 75)
print("CELL 59 — REBUILD CORRECTED CAPTION VOCABULARY + SEQUENCES")
print("=" * 75)

import re
import json
import numpy as np
from collections import Counter

MIN_WORD_FREQ = 2
MAX_LEN = 32

PAD_ID = 0
START_ID = 1
END_ID = 2
UNK_ID = 3

SPECIAL_TOKENS = ["<pad>", "<start>", "<end>", "<unk>"]

# ----------------------------------------------------
# Consistent tokenizer
# ----------------------------------------------------
def tokenize_caption(text):
    text = str(text).lower().strip()

    # Normalize common punctuation so punctuation does not
    # become attached to words.
    text = re.sub(r"([.,!?;:\"()])", r" \1 ", text)

    # Keep possessive/contraction handling consistent.
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    # Remove standalone punctuation tokens.
    tokens = [
        tok for tok in tokens
        if tok not in {".", ",", "!", "?", ";", ":", '"', "(", ")"}
    ]

    return tokens


# ----------------------------------------------------
# Build vocabulary from ALL processed captions
# using the same tokenizer used for encoding.
# ----------------------------------------------------
freq = Counter()

for caption in df_unk["processed_caption"].fillna(""):
    freq.update(tokenize_caption(caption))

words = sorted(
    word for word, count in freq.items()
    if count >= MIN_WORD_FREQ
)

caption_to_id_new = {
    token: idx for idx, token in enumerate(SPECIAL_TOKENS)
}

for word in words:
    if word not in caption_to_id_new:
        caption_to_id_new[word] = len(caption_to_id_new)

id_to_caption_new = {
    idx: word
    for word, idx in caption_to_id_new.items()
}

print(f"Vocabulary size: {len(caption_to_id_new):,}")
print(f"Words before frequency filter: {len(freq):,}")
print(f"Words after frequency >= {MIN_WORD_FREQ}: {len(words):,}")


# ----------------------------------------------------
# Encode using EXACT SAME tokenizer
# ----------------------------------------------------
def encode_caption_corrected(text):
    tokens = tokenize_caption(text)

    ids = [START_ID]

    for tok in tokens[:MAX_LEN - 2]:
        ids.append(
            caption_to_id_new.get(tok, UNK_ID)
        )

    ids.append(END_ID)

    ids += [PAD_ID] * (MAX_LEN - len(ids))

    return ids[:MAX_LEN]


caption_sequences_new = np.asarray(
    [
        encode_caption_corrected(caption)
        for caption in df_unk["processed_caption"].fillna("")
    ],
    dtype=np.int32
)

print()
print("Caption sequence shape:", caption_sequences_new.shape)

# ----------------------------------------------------
# UNK diagnostic
# ----------------------------------------------------
unk_count = int(
    (caption_sequences_new == UNK_ID).sum()
)

non_pad_tokens = int(
    (caption_sequences_new != PAD_ID).sum()
)

captions_with_unk = int(
    np.any(caption_sequences_new == UNK_ID, axis=1).sum()
)

print()
print("CORRECTED UNK STATISTICS")
print("-" * 75)
print(f"UNK tokens:             {unk_count:,}")
print(f"Non-padding tokens:     {non_pad_tokens:,}")
print(f"Captions with UNK:      {captions_with_unk:,}")
print(
    f"Caption % with UNK:     "
    f"{100 * captions_with_unk / len(caption_sequences_new):.2f}%"
)

# ----------------------------------------------------
# Save corrected vocabulary
# ----------------------------------------------------
CAPTION_VOCAB_FILE_NEW = (
    OUTPUT_DIR / "caption_vocabulary_corrected.json"
)

with open(CAPTION_VOCAB_FILE_NEW, "w", encoding="utf-8") as f:
    json.dump(
        {
            "caption_to_id": caption_to_id_new,
            "id_to_caption": {
                str(k): v
                for k, v in id_to_caption_new.items()
            },
            "special_tokens": {
                "pad": PAD_ID,
                "start": START_ID,
                "end": END_ID,
                "unk": UNK_ID
            },
            "max_len": MAX_LEN,
            "min_word_freq": MIN_WORD_FREQ
        },
        f,
        indent=2,
        ensure_ascii=False
    )

# ----------------------------------------------------
# Save corrected sequences
# ----------------------------------------------------
CAPTION_SEQUENCE_FILE_NEW = (
    FEATURE_DIR / "caption_sequences_corrected.npz"
)

np.savez_compressed(
    CAPTION_SEQUENCE_FILE_NEW,
    image_ids=df_unk["image_id"].astype(str).to_numpy(),
    caption_ids=caption_sequences_new
)

print()
print("Saved vocabulary:")
print(CAPTION_VOCAB_FILE_NEW)

print()
print("Saved sequences:")
print(CAPTION_SEQUENCE_FILE_NEW)

print()
print("=" * 75)
print("CELL 59 COMPLETE")
print("=" * 75)

CELL 59 — REBUILD CORRECTED CAPTION VOCABULARY + SEQUENCES
Vocabulary size: 14,117
Words before frequency filter: 20,316
Words after frequency >= 2: 14,113

Caption sequence shape: (199369, 32)

CORRECTED UNK STATISTICS
---------------------------------------------------------------------------
UNK tokens:             6,152
Non-padding tokens:     2,776,670
Captions with UNK:      5,656
Caption % with UNK:     2.84%

Saved vocabulary:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/caption_vocabulary_corrected.json

Saved sequences:
/content/drive/MyDrive/Image_captioning/Features/Structured_Object/caption_sequences_corrected.npz

CELL 59 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 60 — SWITCH TO CORRECTED CAPTION VOCABULARY + SEQUENCES")
print("=" * 75)

# ----------------------------------------------------
# Load corrected vocabulary
# ----------------------------------------------------
with open(
    OUTPUT_DIR / "caption_vocabulary_corrected.json",
    "r",
    encoding="utf-8"
) as f:
    corrected_vocab_raw = json.load(f)

caption_to_id = {
    str(k): int(v)
    for k, v in corrected_vocab_raw["caption_to_id"].items()
}

id_to_caption = {
    int(v): str(k)
    for k, v in caption_to_id.items()
}

# ----------------------------------------------------
# Load corrected sequences
# ----------------------------------------------------
corrected_seq_file = (
    FEATURE_DIR / "caption_sequences_corrected.npz"
)

caption_data = np.load(
    corrected_seq_file,
    allow_pickle=True
)

caption_image_ids = caption_data["image_ids"].astype(str)
caption_sequences = caption_data["caption_ids"].astype(np.int32)

# ----------------------------------------------------
# Special token IDs
# ----------------------------------------------------
PAD_ID = caption_to_id["<pad>"]
START_ID = caption_to_id["<start>"]
END_ID = caption_to_id["<end>"]
UNK_ID = caption_to_id["<unk>"]

# ----------------------------------------------------
# Verification
# ----------------------------------------------------
print("Corrected vocabulary:")
print(f"  Size: {len(caption_to_id):,}")

print()
print("Corrected sequences:")
print(f"  Shape: {caption_sequences.shape}")

print()
print("Special tokens:")
print(f"  PAD:   {PAD_ID}")
print(f"  START: {START_ID}")
print(f"  END:   {END_ID}")
print(f"  UNK:   {UNK_ID}")

print()
print("Representative tokens:")
for word in ["dog", "water", "street", "grass", "ball", "building"]:
    print(
        f"  {word:10} -> "
        f"{caption_to_id.get(word, 'MISSING')}"
    )

print()
print("UNK tokens:")
print(
    f"  {int((caption_sequences == UNK_ID).sum()):,}"
)

print()
print("All sequence values finite:")
print(
    f"  {bool(np.isfinite(caption_sequences).all())}"
)

assert caption_sequences.shape == (199369, 32)
assert len(caption_image_ids) == 199369
assert PAD_ID == 0
assert START_ID == 1
assert END_ID == 2
assert UNK_ID == 3
assert np.isfinite(caption_sequences).all()

print()
print("✓ Corrected caption data is now ACTIVE.")
print("✓ Vocabulary size = 14,117")
print("✓ UNK tokens = 6,152")
print("✓ Old caption vocabulary is NOT being used.")

print()
print("=" * 75)
print("CELL 60 COMPLETE")
print("=" * 75)

CELL 60 — SWITCH TO CORRECTED CAPTION VOCABULARY + SEQUENCES
Corrected vocabulary:
  Size: 14,117

Corrected sequences:
  Shape: (199369, 32)

Special tokens:
  PAD:   0
  START: 1
  END:   2
  UNK:   3

Representative tokens:
  dog        -> 3703
  water      -> 13593
  street     -> 11958
  grass      -> 5371
  ball       -> 913
  building   -> 1751

UNK tokens:
  6,152

All sequence values finite:
  True

✓ Corrected caption data is now ACTIVE.
✓ Vocabulary size = 14,117
✓ UNK tokens = 6,152
✓ Old caption vocabulary is NOT being used.

CELL 60 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 61 — REBUILD TRAIN / VAL / TEST CAPTION INDICES")
print("=" * 75)

# ----------------------------------------------------
# Build image -> caption row indices
# ----------------------------------------------------
from collections import defaultdict

image_to_caption_indices = defaultdict(list)

for idx, image_id in enumerate(caption_image_ids):
    image_to_caption_indices[image_id].append(idx)

# ----------------------------------------------------
# Verify all caption rows are represented
# ----------------------------------------------------
total_indexed_captions = sum(
    len(v) for v in image_to_caption_indices.values()
)

print(f"Unique images:       {len(image_to_caption_indices):,}")
print(f"Caption rows indexed:{total_indexed_captions:,}")

assert total_indexed_captions == 199369

# ----------------------------------------------------
# Use the existing image splits
# ----------------------------------------------------
train_image_ids = list(train_ids)
val_image_ids = list(val_ids)
test_image_ids = list(test_ids)

# ----------------------------------------------------
# Convert image splits -> caption indices
# ----------------------------------------------------
train_caption_indices = [
    idx
    for image_id in train_image_ids
    for idx in image_to_caption_indices.get(image_id, [])
]

val_caption_indices = [
    idx
    for image_id in val_image_ids
    for idx in image_to_caption_indices.get(image_id, [])
]

test_caption_indices = [
    idx
    for image_id in test_image_ids
    for idx in image_to_caption_indices.get(image_id, [])
]

print()
print("Image split:")
print(f"  Train images: {len(train_image_ids):,}")
print(f"  Val images:   {len(val_image_ids):,}")
print(f"  Test images:  {len(test_image_ids):,}")

print()
print("Caption split:")
print(f"  Train captions: {len(train_caption_indices):,}")
print(f"  Val captions:   {len(val_caption_indices):,}")
print(f"  Test captions:  {len(test_caption_indices):,}")

print()
print("Total captions:")
print(
    f"  {len(train_caption_indices) + len(val_caption_indices) + len(test_caption_indices):,}"
)

assert len(train_image_ids) == 31899
assert len(val_image_ids) == 3987
assert len(test_image_ids) == 3988

assert (
    len(train_caption_indices)
    + len(val_caption_indices)
    + len(test_caption_indices)
    == 199369
)

print()
print("✓ Caption rows successfully mapped to image splits.")
print("✓ No captions were lost.")
print()
print("=" * 75)
print("CELL 61 COMPLETE")
print("=" * 75)

CELL 61 — REBUILD TRAIN / VAL / TEST CAPTION INDICES
Unique images:       39,874
Caption rows indexed:199,369

Image split:
  Train images: 31,899
  Val images:   3,987
  Test images:  3,988

Caption split:
  Train captions: 159,494
  Val captions:   19,935
  Test captions:  19,940

Total captions:
  199,369

✓ Caption rows successfully mapped to image splits.
✓ No captions were lost.

CELL 61 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 62 — REBUILD DATASETS + DATALOADERS")
print("=" * 75)

train_dataset = ObjectCaptionDataset(
    image_ids=train_image_ids,
    caption_indices=train_caption_indices,
    object_ids=object_ids,
    spatial=boxes,
    confidence=confidences,
    object_mask=object_mask,
    caption_sequences=caption_sequences
)

val_dataset = ObjectCaptionDataset(
    image_ids=val_image_ids,
    caption_indices=val_caption_indices,
    object_ids=object_ids,
    spatial=boxes,
    confidence=confidences,
    object_mask=object_mask,
    caption_sequences=caption_sequences
)

test_dataset = ObjectCaptionDataset(
    image_ids=test_image_ids,
    caption_indices=test_caption_indices,
    object_ids=object_ids,
    spatial=spatial,
    confidence=confidences,
    object_mask=object_mask,
    caption_sequences=caption_sequences
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Train dataset: {len(train_dataset):,}")
print(f"Val dataset:   {len(val_dataset):,}")
print(f"Test dataset:  {len(test_dataset):,}")

print()
print(f"Train batches: {len(train_loader):,}")
print(f"Val batches:   {len(val_loader):,}")
print(f"Test batches:  {len(test_loader):,}")

# ----------------------------------------------------
# Batch verification
# ----------------------------------------------------
batch = next(iter(train_loader))

print()
print("Batch shapes:")

for key, value in batch.items():
    if torch.is_tensor(value):
        print(f"  {key:15} {tuple(value.shape)} {value.dtype}")

print()
print("✓ Corrected caption sequences connected.")
print("✓ Train/Val/Test loaders rebuilt.")

print()
print("=" * 75)
print("CELL 62 COMPLETE")
print("=" * 75)

CELL 62 — REBUILD DATASETS + DATALOADERS


TypeError: ObjectCaptionDataset.__init__() got an unexpected keyword argument 'caption_indices'

In [ ]:
print("=" * 75)
print("CELL 63 — INSPECT DATASET ATTRIBUTES + ONE SAMPLE")
print("=" * 75)

# Use the exact constructor argument names discovered above.
# We only inspect the class signature and any existing class instances.
print("Class:", ObjectCaptionDataset)
print("Constructor:", inspect.signature(ObjectCaptionDataset.__init__))

# Check whether an existing dataset object from the earlier experiment
# is still available and inspect its attributes.
for name in ["train_dataset", "val_dataset", "test_dataset"]:
    if name in globals():
        obj = globals()[name]
        print()
        print(f"{name} exists:")
        print("  type:", type(obj))
        print("  attributes:", [
            x for x in dir(obj)
            if not x.startswith("__")
        ])

print()
print("=" * 75)
print("CELL 63 COMPLETE")
print("=" * 75)

CELL 63 — INSPECT DATASET ATTRIBUTES + ONE SAMPLE
Class: <class '__main__.ObjectCaptionDataset'>
Constructor: (self, image_ids, caption_sequences, split_image_ids, image_to_object_row)

train_dataset exists:
  type: <class '__main__.ObjectCaptionDataset'>
  attributes: ['caption_indices', 'caption_sequences', 'image_ids', 'image_to_object_row', 'split_image_ids']

val_dataset exists:
  type: <class '__main__.ObjectCaptionDataset'>
  attributes: ['caption_indices', 'caption_sequences', 'image_ids', 'image_to_object_row', 'split_image_ids']

test_dataset exists:
  type: <class '__main__.ObjectCaptionDataset'>
  attributes: ['caption_indices', 'caption_sequences', 'image_ids', 'image_to_object_row', 'split_image_ids']

CELL 63 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 64 — CONNECT CORRECTED CAPTION SEQUENCES")
print("=" * 75)

# Keep the already-validated dataset mappings.
# Only replace the old caption sequence matrix.

old_train_count = len(train_dataset.caption_sequences)
old_val_count = len(val_dataset.caption_sequences)
old_test_count = len(test_dataset.caption_sequences)

print("Existing dataset caption sequence references:")
print(f"  Train: {old_train_count:,}")
print(f"  Val:   {old_val_count:,}")
print(f"  Test:  {old_test_count:,}")

# Replace with corrected sequences
train_dataset.caption_sequences = caption_sequences
val_dataset.caption_sequences = caption_sequences
test_dataset.caption_sequences = caption_sequences

# Verify references
assert train_dataset.caption_sequences is caption_sequences
assert val_dataset.caption_sequences is caption_sequences
assert test_dataset.caption_sequences is caption_sequences

print()
print("Corrected sequence shape:")
print(f"  {caption_sequences.shape}")

print()
print("Dataset mappings preserved:")
print(f"  Train caption indices: {len(train_dataset.caption_indices):,}")
print(f"  Val caption indices:   {len(val_dataset.caption_indices):,}")
print(f"  Test caption indices:  {len(test_dataset.caption_indices):,}")

print()
print("✓ Existing image/object mappings preserved.")
print("✓ Corrected caption sequences connected.")
print("✓ No dataset reconstruction required.")

print()
print("=" * 75)
print("CELL 64 COMPLETE")
print("=" * 75)

CELL 64 — CONNECT CORRECTED CAPTION SEQUENCES
Existing dataset caption sequence references:
  Train: 199,369
  Val:   199,369
  Test:  199,369

Corrected sequence shape:
  (199369, 32)

Dataset mappings preserved:
  Train caption indices: 159,494
  Val caption indices:   19,935
  Test caption indices:  19,940

✓ Existing image/object mappings preserved.
✓ Corrected caption sequences connected.
✓ No dataset reconstruction required.

CELL 64 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 65 — VERIFY CORRECTED TRAINING BATCH")
print("=" * 75)

batch = next(iter(train_loader))

print("Batch shapes:")
for key, value in batch.items():
    if torch.is_tensor(value):
        print(f"  {key:15} {tuple(value.shape)} {value.dtype}")

# ----------------------------------------------------
# Verify caption IDs belong to corrected vocabulary
# ----------------------------------------------------
decoder_input = batch["decoder_input"]
target = batch["target"]

print()
print("Caption ID range:")
print(f"  Decoder input min: {int(decoder_input.min())}")
print(f"  Decoder input max: {int(decoder_input.max())}")
print(f"  Target min:        {int(target.min())}")
print(f"  Target max:        {int(target.max())}")

print()
print("Corrected vocabulary size:")
print(f"  {len(caption_to_id):,}")

assert decoder_input.shape == (BATCH_SIZE, MAX_LEN - 1)
assert target.shape == (BATCH_SIZE, MAX_LEN - 1)

assert int(decoder_input.min()) >= 0
assert int(target.min()) >= 0

assert int(decoder_input.max()) < len(caption_to_id)
assert int(target.max()) < len(caption_to_id)

assert torch.isfinite(
    decoder_input.float()
).all()

assert torch.isfinite(
    target.float()
).all()

print()
print("✓ Batch is using corrected caption IDs.")
print("✓ All caption IDs are within vocabulary range.")
print("✓ No invalid IDs detected.")

print()
print("=" * 75)
print("CELL 65 COMPLETE")
print("=" * 75)

CELL 65 — VERIFY CORRECTED TRAINING BATCH
Batch shapes:
  object_ids      (64, 10) torch.int64
  spatial         (64, 10, 9) torch.float32
  confidence      (64, 10) torch.float32
  object_mask     (64, 10) torch.bool
  decoder_input   (64, 31) torch.int64
  target          (64, 31) torch.int64

Caption ID range:
  Decoder input min: 0
  Decoder input max: 14073
  Target min:        0
  Target max:        14073

Corrected vocabulary size:
  14,117

✓ Batch is using corrected caption IDs.
✓ All caption IDs are within vocabulary range.
✓ No invalid IDs detected.

CELL 65 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 66 — INITIALIZE FRESH OBJECT-AWARE TRANSFORMER")
print("=" * 75)

# ----------------------------------------------------
# Fresh model with corrected vocabulary
# ----------------------------------------------------
model = ObjectAwareTransformer(
    object_vocab_size=len(object_to_id),
    caption_vocab_size=len(caption_to_id),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_caption_length=MAX_LEN
).to(DEVICE)

# ----------------------------------------------------
# Optimizer
# ----------------------------------------------------
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ----------------------------------------------------
# Loss
# ----------------------------------------------------
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=0.1
)

# ----------------------------------------------------
# Parameter verification
# ----------------------------------------------------
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Device:              {DEVICE}")
print(f"Object vocabulary:   {len(object_to_id):,}")
print(f"Caption vocabulary:  {len(caption_to_id):,}")
print(f"Total parameters:    {total_params:,}")
print(f"Trainable parameters:{trainable_params:,}")
print(f"Learning rate:       {optimizer.param_groups[0]['lr']}")
print(f"Weight decay:        {optimizer.param_groups[0]['weight_decay']}")

# ----------------------------------------------------
# Finite parameter check
# ----------------------------------------------------
all_finite = all(
    torch.isfinite(p).all().item()
    for p in model.parameters()
)

print()
print(f"All parameters finite: {all_finite}")

assert all_finite
assert len(caption_to_id) == 14117
assert trainable_params == total_params

print()
print("✓ Fresh model initialized.")
print("✓ Output projection matches corrected vocabulary.")
print("✓ No old checkpoint loaded.")
print("✓ All parameters finite.")

print()
print("=" * 75)
print("CELL 66 COMPLETE")
print("=" * 75)

CELL 66 — INITIALIZE FRESH OBJECT-AWARE TRANSFORMER


/tmp/ipykernel_1090/3777785277.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(


Device:              cuda
Object vocabulary:   82
Caption vocabulary:  14,117
Total parameters:    13,091,621
Trainable parameters:13,091,621
Learning rate:       0.0001
Weight decay:        0.0001

All parameters finite: True

✓ Fresh model initialized.
✓ Output projection matches corrected vocabulary.
✓ No old checkpoint loaded.
✓ All parameters finite.

CELL 66 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 67 — FRESH MODEL FORWARD + LOSS SANITY CHECK")
print("=" * 75)

model.train()

# ----------------------------------------------------
# Move batch to GPU
# ----------------------------------------------------
object_ids_batch = batch["object_ids"].to(DEVICE)
spatial_batch = batch["spatial"].to(DEVICE)
confidence_batch = batch["confidence"].to(DEVICE)
object_mask_batch = batch["object_mask"].to(DEVICE)
decoder_input_batch = batch["decoder_input"].to(DEVICE)
target_batch = batch["target"].to(DEVICE)

# ----------------------------------------------------
# Forward pass
# ----------------------------------------------------
with torch.no_grad():
    logits = model(
        object_ids_batch,
        spatial_batch,
        confidence_batch,
        object_mask_batch,
        decoder_input_batch
    )

# ----------------------------------------------------
# Verify logits
# ----------------------------------------------------
print("Logits shape:")
print(f"  {tuple(logits.shape)}")

print()
print("Expected shape:")
print(
    f"  ({BATCH_SIZE}, {MAX_LEN - 1}, {len(caption_to_id)})"
)

print()
print("Logits finite:")
print(f"  {bool(torch.isfinite(logits).all())}")

# ----------------------------------------------------
# Loss
# ----------------------------------------------------
loss = criterion(
    logits.reshape(-1, logits.size(-1)),
    target_batch.reshape(-1)
)

print()
print(f"Initial loss: {loss.item():.6f}")
print(f"Loss finite:  {bool(torch.isfinite(loss))}")

# ----------------------------------------------------
# Assertions
# ----------------------------------------------------
assert logits.shape == (
    BATCH_SIZE,
    MAX_LEN - 1,
    len(caption_to_id)
)

assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

print()
print("✓ Forward pass successful.")
print("✓ Logits match corrected vocabulary size.")
print("✓ Loss is finite.")
print("✓ Fresh model is ready for training.")

print()
print("=" * 75)
print("CELL 67 COMPLETE")
print("=" * 75)

CELL 67 — FRESH MODEL FORWARD + LOSS SANITY CHECK
Logits shape:
  (64, 31, 14117)

Expected shape:
  (64, 31, 14117)

Logits finite:
  True

Initial loss: 9.750975
Loss finite:  True

✓ Forward pass successful.
✓ Logits match corrected vocabulary size.
✓ Loss is finite.
✓ Fresh model is ready for training.

CELL 67 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 68 — 100-STEP TRAINING SANITY CHECK")
print("=" * 75)

model.train()

sanity_losses = []
sanity_steps = 100

for step, batch_s in enumerate(train_loader, start=1):

    # ------------------------------------------------
    # Move batch to GPU
    # ------------------------------------------------
    object_ids_s = batch_s["object_ids"].to(DEVICE)
    spatial_s = batch_s["spatial"].to(DEVICE)
    confidence_s = batch_s["confidence"].to(DEVICE)
    object_mask_s = batch_s["object_mask"].to(DEVICE)
    decoder_input_s = batch_s["decoder_input"].to(DEVICE)
    target_s = batch_s["target"].to(DEVICE)

    optimizer.zero_grad(set_to_none=True)

    # ------------------------------------------------
    # Forward
    # ------------------------------------------------
    logits_s = model(
        object_ids_s,
        spatial_s,
        confidence_s,
        object_mask_s,
        decoder_input_s
    )

    loss_s = criterion(
        logits_s.reshape(-1, logits_s.size(-1)),
        target_s.reshape(-1)
    )

    # ------------------------------------------------
    # Numerical safety
    # ------------------------------------------------
    if not torch.isfinite(loss_s):
        raise RuntimeError(
            f"Non-finite loss at sanity step {step}: "
            f"{loss_s.item()}"
        )

    loss_s.backward()

    # ------------------------------------------------
    # Gradient clipping
    # ------------------------------------------------
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    if not torch.isfinite(grad_norm):
        raise RuntimeError(
            f"Non-finite gradient norm at step {step}"
        )

    optimizer.step()

    # ------------------------------------------------
    # Parameter safety
    # ------------------------------------------------
    if not all(
        torch.isfinite(p).all().item()
        for p in model.parameters()
    ):
        raise RuntimeError(
            f"Non-finite model parameter at step {step}"
        )

    loss_value = loss_s.item()
    sanity_losses.append(loss_value)

    if step % 10 == 0:
        recent_avg = sum(sanity_losses[-10:]) / min(
            10, len(sanity_losses)
        )

        print(
            f"Step {step:3d} | "
            f"Loss: {loss_value:.4f} | "
            f"10-step avg: {recent_avg:.4f}"
        )

    if step >= sanity_steps:
        break

# ----------------------------------------------------
# Final statistics
# ----------------------------------------------------
avg_loss = sum(sanity_losses) / len(sanity_losses)

print()
print(f"Steps completed: {len(sanity_losses)}")
print(f"Average loss:    {avg_loss:.6f}")
print(f"First loss:      {sanity_losses[0]:.6f}")
print(f"Last loss:       {sanity_losses[-1]:.6f}")
print(f"Loss reduction:  {sanity_losses[0] - sanity_losses[-1]:.6f}")

print()

if torch.cuda.is_available():
    print(
        f"Peak GPU memory: "
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )

# ----------------------------------------------------
# Basic learning check
# ----------------------------------------------------
assert len(sanity_losses) == sanity_steps
assert all(np.isfinite(sanity_losses))

print()
print("✓ 100-step sanity training completed.")
print("✓ Loss remained finite.")
print("✓ Gradients remained finite.")
print("✓ Model parameters remained finite.")

print()
print("=" * 75)
print("CELL 68 COMPLETE")
print("=" * 75)

CELL 68 — 100-STEP TRAINING SANITY CHECK
Step  10 | Loss: 7.8698 | 10-step avg: 8.8178
Step  20 | Loss: 7.1017 | 10-step avg: 7.4457
Step  30 | Loss: 6.3566 | 10-step avg: 6.6137
Step  40 | Loss: 6.2491 | 10-step avg: 6.3345
Step  50 | Loss: 6.3193 | 10-step avg: 6.2433
Step  60 | Loss: 6.1609 | 10-step avg: 6.1674
Step  70 | Loss: 5.9783 | 10-step avg: 6.0868
Step  80 | Loss: 6.0007 | 10-step avg: 6.0448
Step  90 | Loss: 5.7755 | 10-step avg: 5.8858
Step 100 | Loss: 5.9478 | 10-step avg: 5.8643

Steps completed: 100
Average loss:    6.550416
First loss:      9.795728
Last loss:       5.947766
Loss reduction:  3.847962

Peak GPU memory: 1.32 GB

✓ 100-step sanity training completed.
✓ Loss remained finite.
✓ Gradients remained finite.
✓ Model parameters remained finite.

CELL 68 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 69 — RESET MODEL FOR REAL TRAINING")
print("=" * 75)

# ----------------------------------------------------
# Completely fresh model
# ----------------------------------------------------
model = ObjectAwareTransformer(
    object_vocab_size=len(object_to_id),
    caption_vocab_size=len(caption_to_id),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_caption_length=MAX_LEN
).to(DEVICE)

# ----------------------------------------------------
# Fresh optimizer
# ----------------------------------------------------
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ----------------------------------------------------
# Fresh scheduler
# ----------------------------------------------------
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

# ----------------------------------------------------
# Reset training history
# ----------------------------------------------------
history = {
    "train_loss": [],
    "val_loss": [],
    "learning_rate": []
}

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

# ----------------------------------------------------
# Parameter verification
# ----------------------------------------------------
total_params = sum(p.numel() for p in model.parameters())

print(f"Device:             {DEVICE}")
print(f"Caption vocabulary: {len(caption_to_id):,}")
print(f"Total parameters:   {total_params:,}")
print(f"Learning rate:      {optimizer.param_groups[0]['lr']}")
print(f"Optimizer state:    {len(optimizer.state)}")

all_finite = all(
    torch.isfinite(p).all().item()
    for p in model.parameters()
)

print(f"All parameters finite: {all_finite}")

assert all_finite
assert len(optimizer.state) == 0
assert len(history["train_loss"]) == 0

print()
print("✓ Model completely reset.")
print("✓ Optimizer completely reset.")
print("✓ Training history cleared.")
print("✓ No sanity-run weights remain.")
print("✓ Ready for the real training experiment.")

print()
print("=" * 75)
print("CELL 69 COMPLETE")
print("=" * 75)

CELL 69 — RESET MODEL FOR REAL TRAINING


/tmp/ipykernel_1090/3777785277.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(


Device:             cuda
Caption vocabulary: 14,117
Total parameters:   13,091,621
Learning rate:      0.0001
Optimizer state:    0
All parameters finite: True

✓ Model completely reset.
✓ Optimizer completely reset.
✓ Training history cleared.
✓ No sanity-run weights remain.
✓ Ready for the real training experiment.

CELL 69 COMPLETE


In [ ]:
print("=" * 75)
print("CELL 70 — REAL TRAINING: OBJECT-AWARE TRANSFORMER")
print("=" * 75)

NUM_EPOCHS = 20
PATIENCE = 4

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "learning_rate": []
}

CHECKPOINT_DIR = OUTPUT_DIR / "Checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = (
    CHECKPOINT_DIR /
    "best_object_aware_transformer_corrected.pt"
)

for epoch in range(1, NUM_EPOCHS + 1):

    # IMPORTANT:
    # Actual signature:
    # train_one_epoch(model, loader, criterion, optimizer, device)

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    val_loss = validate_one_epoch(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["learning_rate"].append(current_lr)

    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_loss": best_val_loss,
                "caption_vocab_size": len(caption_to_id),
                "object_vocab_size": len(object_to_id),
                "d_model": D_MODEL,
                "nhead": NHEAD,
                "num_encoder_layers": NUM_ENCODER_LAYERS,
                "num_decoder_layers": NUM_DECODER_LAYERS,
                "dim_feedforward": DIM_FEEDFORWARD,
                "dropout": DROPOUT,
                "max_caption_length": MAX_LEN
            },
            BEST_CHECKPOINT
        )

    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train: {train_loss:.6f} | "
        f"Val: {val_loss:.6f} | "
        f"LR: {current_lr:.2e}"
        + (" | BEST" if improved else "")
    )

    if epochs_without_improvement >= PATIENCE:
        print()
        print(
            f"Early stopping triggered after {epoch} epochs."
        )
        break

print()
print("=" * 75)
print("TRAINING COMPLETE")
print("=" * 75)

print(f"Epochs completed: {len(history['train_loss'])}")
print(f"Best epoch:       {best_epoch}")
print(f"Best val loss:    {best_val_loss:.6f}")
print(f"Checkpoint:       {BEST_CHECKPOINT}")

assert len(history["train_loss"]) > 0
assert len(history["train_loss"]) == len(history["val_loss"])
assert np.isfinite(history["train_loss"]).all()
assert np.isfinite(history["val_loss"]).all()

print()
print("✓ Training losses finite.")
print("✓ Validation losses finite.")
print("✓ Best checkpoint saved.")
print("=" * 75)

CELL 70 — REAL TRAINING: OBJECT-AWARE TRANSFORMER
Epoch 01/20 | Train: 4.950672 | Val: 4.467871 | LR: 1.00e-04 | BEST
Epoch 02/20 | Train: 4.458607 | Val: 4.284246 | LR: 1.00e-04 | BEST
Epoch 03/20 | Train: 4.299666 | Val: 4.179149 | LR: 1.00e-04 | BEST
Epoch 04/20 | Train: 4.202589 | Val: 4.107935 | LR: 1.00e-04 | BEST
Epoch 05/20 | Train: 4.131767 | Val: 4.061043 | LR: 1.00e-04 | BEST
Epoch 06/20 | Train: 4.075929 | Val: 4.021777 | LR: 1.00e-04 | BEST
Epoch 07/20 | Train: 4.030617 | Val: 3.997435 | LR: 1.00e-04 | BEST
Epoch 08/20 | Train: 3.991527 | Val: 3.969489 | LR: 1.00e-04 | BEST
Epoch 09/20 | Train: 3.957085 | Val: 3.947603 | LR: 1.00e-04 | BEST
Epoch 10/20 | Train: 3.927155 | Val: 3.927746 | LR: 1.00e-04 | BEST
Epoch 11/20 | Train: 3.901095 | Val: 3.917499 | LR: 1.00e-04 | BEST
Epoch 12/20 | Train: 3.876969 | Val: 3.903500 | LR: 1.00e-04 | BEST


KeyboardInterrupt: 

In [ ]:
# ============================================================================
# CELL 70 — RESUME REAL TRAINING FROM EPOCH 12
# ============================================================================

import torch
import numpy as np
from pathlib import Path

print("=" * 75)
print("CELL 70 — RESUME TRAINING FROM EPOCH 12")
print("=" * 75)

# ---------------------------------------------------------------------------
# CHECKPOINT
# ---------------------------------------------------------------------------

BEST_CHECKPOINT = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Outputs/Structured_Captioning/Checkpoints/"
    "best_object_aware_transformer_corrected.pt"
)

assert BEST_CHECKPOINT.exists(), (
    f"Checkpoint not found:\n{BEST_CHECKPOINT}"
)

print(f"Checkpoint: {BEST_CHECKPOINT}")

# ---------------------------------------------------------------------------
# LOAD CHECKPOINT
# ---------------------------------------------------------------------------

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

saved_epoch = checkpoint["epoch"]
saved_best_val = checkpoint["best_val_loss"]

print(f"Saved epoch: {saved_epoch}")
print(f"Saved best validation loss: {saved_best_val:.6f}")

assert saved_epoch == 12, (
    f"Expected Epoch 12 checkpoint, found Epoch {saved_epoch}"
)

# ---------------------------------------------------------------------------
# RESTORE MODEL
# ---------------------------------------------------------------------------

model.load_state_dict(checkpoint["model_state_dict"])

# ---------------------------------------------------------------------------
# RESTORE OPTIMIZER
# ---------------------------------------------------------------------------

optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# Make sure optimizer tensors are on the correct device
for state in optimizer.state.values():
    for key, value in state.items():
        if torch.is_tensor(value):
            state[key] = value.to(DEVICE)

# ---------------------------------------------------------------------------
# RESTORE SCHEDULER
# ---------------------------------------------------------------------------

scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

# ---------------------------------------------------------------------------
# RESTORE TRAINING STATE
# ---------------------------------------------------------------------------

best_val_loss = saved_best_val
best_epoch = saved_epoch
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "learning_rate": []
}

# Previous results are included for record keeping
previous_train = [
    4.950672,
    4.458607,
    4.299666,
    4.202589,
    4.131767,
    4.075929,
    4.030617,
    3.991527,
    3.957085,
    3.927155,
    3.901095,
    3.876969
]

previous_val = [
    4.467871,
    4.284246,
    4.179149,
    4.107935,
    4.061043,
    4.021777,
    3.997435,
    3.969489,
    3.947603,
    3.927746,
    3.917499,
    3.903500
]

history["train_loss"].extend(previous_train)
history["val_loss"].extend(previous_val)

# ---------------------------------------------------------------------------
# CONTINUE EPOCHS 13 → 20
# ---------------------------------------------------------------------------

NUM_EPOCHS = 20
PATIENCE = 4

CHECKPOINT_DIR = BEST_CHECKPOINT.parent
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print()
print("=" * 75)
print("RESUMING")
print("=" * 75)
print(f"Starting from: Epoch {saved_epoch + 1}")
print(f"Remaining epochs: {NUM_EPOCHS - saved_epoch}")
print(f"Best validation loss: {best_val_loss:.6f}")
print("=" * 75)

for epoch in range(saved_epoch + 1, NUM_EPOCHS + 1):

    # TRAIN
    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    # VALIDATION
    val_loss = validate_one_epoch(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    # LR scheduler
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["learning_rate"].append(current_lr)

    # Check improvement
    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_loss": best_val_loss,

                "caption_vocab_size": len(caption_to_id),
                "object_vocab_size": len(object_to_id),

                "d_model": D_MODEL,
                "nhead": NHEAD,
                "num_encoder_layers": NUM_ENCODER_LAYERS,
                "num_decoder_layers": NUM_DECODER_LAYERS,
                "dim_feedforward": DIM_FEEDFORWARD,
                "dropout": DROPOUT,
                "max_caption_length": MAX_LEN
            },
            BEST_CHECKPOINT
        )

    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train: {train_loss:.6f} | "
        f"Val: {val_loss:.6f} | "
        f"LR: {current_lr:.2e}"
        + (" | BEST" if improved else "")
    )

    # Early stopping
    if epochs_without_improvement >= PATIENCE:
        print()
        print(
            f"Early stopping triggered after {epoch} epochs."
        )
        break

print()
print("=" * 75)
print("TRAINING COMPLETE")
print("=" * 75)

print(f"Epochs completed: {len(history['train_loss'])}")
print(f"Best epoch:       {best_epoch}")
print(f"Best val loss:    {best_val_loss:.6f}")
print(f"Checkpoint:       {BEST_CHECKPOINT}")

assert np.isfinite(history["train_loss"]).all()
assert np.isfinite(history["val_loss"]).all()

print()
print("✓ Training losses finite")
print("✓ Validation losses finite")
print("✓ Best checkpoint saved")
print("=" * 75)

CELL 70 — RESUME TRAINING FROM EPOCH 12
Checkpoint: /content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/Checkpoints/best_object_aware_transformer_corrected.pt
Saved epoch: 12
Saved best validation loss: 3.903500

RESUMING
Starting from: Epoch 13
Remaining epochs: 8
Best validation loss: 3.903500
Epoch 13/20 | Train: 3.854603 | Val: 3.891815 | LR: 1.00e-04 | BEST
Epoch 14/20 | Train: 3.833208 | Val: 3.879235 | LR: 1.00e-04 | BEST
Epoch 15/20 | Train: 3.814208 | Val: 3.866494 | LR: 1.00e-04 | BEST
Epoch 16/20 | Train: 3.795456 | Val: 3.861458 | LR: 1.00e-04 | BEST
Epoch 17/20 | Train: 3.778848 | Val: 3.854424 | LR: 1.00e-04 | BEST
Epoch 18/20 | Train: 3.763199 | Val: 3.845340 | LR: 1.00e-04 | BEST
Epoch 19/20 | Train: 3.748592 | Val: 3.837948 | LR: 1.00e-04 | BEST
Epoch 20/20 | Train: 3.734360 | Val: 3.833584 | LR: 1.00e-04 | BEST

TRAINING COMPLETE
Epochs completed: 20
Best epoch:       20
Best val loss:    3.833584
Checkpoint:       /content/drive/MyDrive/Image_captio

In [ ]:
# ============================================================================
# CELL 71 — INSPECT ACTUAL TEST BATCH
# ============================================================================

print("=" * 75)
print("CELL 71 — TEST BATCH STRUCTURE")
print("=" * 75)

batch = next(iter(test_loader))

print("Batch type:", type(batch))
print("Number of returned items:", len(batch))

for i, item in enumerate(batch):
    if hasattr(item, "shape"):
        print(
            f"[{i}] type={type(item).__name__} "
            f"shape={tuple(item.shape)} "
            f"dtype={item.dtype}"
        )
    else:
        print(
            f"[{i}] type={type(item).__name__} "
            f"value={item}"
        )

print()
print("Test dataset:", len(test_dataset))
print("Test batches:", len(test_loader))

print("=" * 75)
print("CELL 71 COMPLETE")
print("=" * 75)

CELL 71 — TEST BATCH STRUCTURE
Batch type: <class 'dict'>
Number of returned items: 7
[0] type=str value=image_id
[1] type=str value=object_ids
[2] type=str value=spatial
[3] type=str value=confidence
[4] type=str value=object_mask
[5] type=str value=decoder_input
[6] type=str value=target

Test dataset: 19940
Test batches: 312
CELL 71 COMPLETE


In [ ]:
# ============================================================================
# CELL 72 — TEST SET CAPTION GENERATION (GREEDY DECODING)
# ============================================================================

import torch
import pandas as pd
from tqdm.auto import tqdm

print("=" * 75)
print("CELL 72 — GENERATING TEST CAPTIONS")
print("=" * 75)

model.eval()

generated_results = []

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Generating captions"):

        image_ids = batch["image_id"]

        object_ids = batch["object_ids"].to(DEVICE)
        spatial = batch["spatial"].to(DEVICE)
        confidence = batch["confidence"].to(DEVICE)
        object_mask = batch["object_mask"].to(DEVICE)

        batch_size = object_ids.size(0)

        # Start every caption with <start>
        generated = torch.full(
            (batch_size, 1),
            START_ID,
            dtype=torch.long,
            device=DEVICE
        )

        finished = torch.zeros(
            batch_size,
            dtype=torch.bool,
            device=DEVICE
        )

        # Generate up to MAX_LEN - 1 additional tokens
        for _ in range(MAX_LEN - 1):

            logits = model(
                object_ids,
                spatial,
                confidence,
                object_mask,
                generated
            )

            # Prediction for the newest position
            next_token = logits[:, -1, :].argmax(dim=-1)

            # Keep END tokens for already-finished sequences
            next_token = torch.where(
                finished,
                torch.tensor(END_ID, device=DEVICE),
                next_token
            )

            generated = torch.cat(
                [generated, next_token.unsqueeze(1)],
                dim=1
            )

            finished = finished | (next_token == END_ID)

            if finished.all():
                break

        # Convert generated IDs to words
        generated_cpu = generated.cpu().numpy()

        for i in range(batch_size):

            tokens = []

            for token_id in generated_cpu[i]:

                token_id = int(token_id)

                if token_id in (PAD_ID, START_ID):
                    continue

                if token_id == END_ID:
                    break

                word = id_to_caption.get(token_id, "<unk>")

                if word != "<unk>":
                    tokens.append(word)

            caption = " ".join(tokens).strip()

            generated_results.append({
                "image_id": str(image_ids[i]),
                "generated_caption": caption
            })

print()
print("=" * 75)
print("GENERATION COMPLETE")
print("=" * 75)

generated_df = pd.DataFrame(generated_results)

print("Generated captions:", len(generated_df))
print("Unique images:", generated_df["image_id"].nunique())

print()
print(generated_df.head(10).to_string(index=False))

assert len(generated_df) == len(test_dataset)
assert generated_df["image_id"].nunique() > 0

print()
print("✓ Test captions generated successfully")
print("=" * 75)

CELL 72 — GENERATING TEST CAPTIONS


Generating captions:   0%|          | 0/312 [00:00<?, ?it/s]


GENERATION COMPLETE
Generated captions: 19940
Unique images: 3988

                          image_id                                        generated_caption
 Flickr8K/101654506_8eb26cfb60.jpg                        a dog is running through the snow
 Flickr8K/101654506_8eb26cfb60.jpg                        a dog is running through the snow
 Flickr8K/101654506_8eb26cfb60.jpg                        a dog is running through the snow
 Flickr8K/101654506_8eb26cfb60.jpg                        a dog is running through the snow
 Flickr8K/101654506_8eb26cfb60.jpg                        a dog is running through the snow
Flickr8K/1032460886_4a598ed535.jpg a man is sitting in a chair in front of a large building
Flickr8K/1032460886_4a598ed535.jpg a man is sitting in a chair in front of a large building
Flickr8K/1032460886_4a598ed535.jpg a man is sitting in a chair in front of a large building
Flickr8K/1032460886_4a598ed535.jpg a man is sitting in a chair in front of a large building
Flickr8K/103

In [ ]:
# ============================================================================
# CELL 73 — BUILD TEST REFERENCE CAPTIONS
# SELF-CONTAINED: RECREATES REQUIRED PATHS
# ============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 75)
print("CELL 73 — BUILDING TEST REFERENCE CAPTIONS")
print("=" * 75)

# --------------------------------------------------------------------------
# 1. Recreate permanent Drive paths
# --------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/Image_captioning")

PROCESSED_CAPTIONS = (
    DRIVE_ROOT
    / "Dataset"
    / "Processed"
    / "Captions"
    / "captions_processed.csv"
)

print("Caption file:")
print(PROCESSED_CAPTIONS)

assert PROCESSED_CAPTIONS.exists(), (
    f"Caption file not found:\n{PROCESSED_CAPTIONS}"
)

# --------------------------------------------------------------------------
# 2. Load processed captions
# --------------------------------------------------------------------------

captions_eval_df = pd.read_csv(PROCESSED_CAPTIONS)

print()
print("Caption CSV shape:", captions_eval_df.shape)
print("Columns:", list(captions_eval_df.columns))

# --------------------------------------------------------------------------
# 3. Verify required columns
# --------------------------------------------------------------------------

assert "image_id" in captions_eval_df.columns, \
    "image_id column not found in captions CSV."

assert "caption" in captions_eval_df.columns, \
    "caption column not found in captions CSV."

captions_eval_df["image_id"] = (
    captions_eval_df["image_id"].astype(str)
)

# --------------------------------------------------------------------------
# 4. Get test image IDs from generated predictions
#    This avoids depending on test_image_ids from an old runtime.
# --------------------------------------------------------------------------

assert "generated_df" in globals(), \
    "generated_df is not available. Please rerun Cell 72 first."

generated_df["image_id"] = (
    generated_df["image_id"].astype(str)
)

test_image_set = set(
    generated_df["image_id"].unique()
)

print()
print("Generated test images:", len(test_image_set))

# --------------------------------------------------------------------------
# 5. Select all human captions belonging to test images
# --------------------------------------------------------------------------

test_refs_df = captions_eval_df[
    captions_eval_df["image_id"].isin(test_image_set)
].copy()

print("Test reference rows:", len(test_refs_df))
print("Test reference images:", test_refs_df["image_id"].nunique())

# --------------------------------------------------------------------------
# 6. Build image_id -> list of human reference captions
# --------------------------------------------------------------------------

reference_dict = {}

for image_id, group in test_refs_df.groupby("image_id"):

    refs = [
        str(caption).strip().lower()
        for caption in group["caption"]
        if pd.notna(caption) and str(caption).strip()
    ]

    reference_dict[image_id] = refs

# --------------------------------------------------------------------------
# 7. Verify reference counts
# --------------------------------------------------------------------------

reference_counts = pd.Series(
    {
        image_id: len(refs)
        for image_id, refs in reference_dict.items()
    }
)

print()
print("Reference captions per image:")
print(reference_counts.value_counts().sort_index())

print()
print("Images with 5 references:",
      int((reference_counts == 5).sum()))

print("Images with 4 references:",
      int((reference_counts == 4).sum()))

print("Images with fewer than 4 references:",
      int((reference_counts < 4).sum()))

# --------------------------------------------------------------------------
# 8. Verify every generated image has human references
# --------------------------------------------------------------------------

missing_refs = (
    test_image_set
    - set(reference_dict.keys())
)

print()
print("Images missing references:", len(missing_refs))

if missing_refs:
    print("First missing IDs:", list(missing_refs)[:10])

assert len(missing_refs) == 0, \
    "Some generated test images have no reference captions."

# --------------------------------------------------------------------------
# 9. Final consistency checks
# --------------------------------------------------------------------------

assert len(reference_dict) == len(test_image_set)

assert sum(len(v) for v in reference_dict.values()) == len(test_refs_df)

print()
print("✓ Reference mapping created successfully")
print("✓ Every generated test image has human reference captions")
print("✓ Ready for metric evaluation")

print("=" * 75)
print("CELL 73 COMPLETE")
print("=" * 75)

CELL 73 — BUILDING TEST REFERENCE CAPTIONS
Caption file:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv

Caption CSV shape: (199369, 7)
Columns: ['image_id', 'dataset', 'image_name', 'caption', 'processed_caption', 'model_caption', 'caption_length']

Generated test images: 3988
Test reference rows: 19940
Test reference images: 3988

Reference captions per image:
5    3988
Name: count, dtype: int64

Images with 5 references: 3988
Images with 4 references: 0
Images with fewer than 4 references: 0

Images missing references: 0

✓ Reference mapping created successfully
✓ Every generated test image has human reference captions
✓ Ready for metric evaluation
CELL 73 COMPLETE


In [ ]:
# ============================================================================
# CELL 74 — CHECK CAPTION EVALUATION LIBRARIES
# ============================================================================

print("=" * 75)
print("CELL 74 — CHECKING EVALUATION LIBRARIES")
print("=" * 75)

import importlib.util

packages = [
    "nltk",
    "rouge_score",
    "pycocoevalcap"
]

for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package:20s} : {'✓ INSTALLED' if installed else '✗ NOT INSTALLED'}")

print()
print("Python environment ready for evaluation-library setup.")
print("=" * 75)

CELL 74 — CHECKING EVALUATION LIBRARIES
nltk                 : ✓ INSTALLED
rouge_score          : ✗ NOT INSTALLED
pycocoevalcap        : ✗ NOT INSTALLED

Python environment ready for evaluation-library setup.


In [ ]:
# ============================================================================
# CELL 75 — VERIFY NLTK EVALUATION RESOURCES
# ============================================================================

import nltk

print("=" * 75)
print("CELL 75 — CHECKING NLTK RESOURCES")
print("=" * 75)

resources = {
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab",
    "wordnet": "corpora/wordnet",
    "omw-1.4": "corpora/omw-1.4",
}

for name, path in resources.items():
    try:
        nltk.data.find(path)
        print(f"{name:15s}: ✓ AVAILABLE")
    except LookupError:
        print(f"{name:15s}: ✗ MISSING")

print("=" * 75)
print("CELL 75 COMPLETE")
print("=" * 75)

CELL 75 — CHECKING NLTK RESOURCES
punkt          : ✗ MISSING
punkt_tab      : ✗ MISSING
wordnet        : ✗ MISSING
omw-1.4        : ✗ MISSING
CELL 75 COMPLETE


In [ ]:
# ============================================================================
# CELL 76 — INSTALL NLTK EVALUATION RESOURCES
# ============================================================================

import nltk

print("=" * 75)
print("CELL 76 — INSTALLING NLTK RESOURCES")
print("=" * 75)

resources = [
    "punkt",
    "punkt_tab",
    "wordnet",
    "omw-1.4"
]

for resource in resources:
    print(f"Downloading {resource}...")
    nltk.download(resource, quiet=True)
    print(f"✓ {resource}")

print()
print("✓ All required NLTK resources downloaded")
print("=" * 75)
print("CELL 76 COMPLETE")
print("=" * 75)

CELL 76 — INSTALLING NLTK RESOURCES
✓ punkt
✓ punkt_tab
✓ wordnet
✓ omw-1.4

✓ All required NLTK resources downloaded
CELL 76 COMPLETE


In [ ]:
# ============================================================================
# CELL 77 — INSTALL STANDARD COCO CAPTION EVALUATION TOOLKIT
# ============================================================================

import subprocess
import sys

print("=" * 75)
print("CELL 77 — INSTALLING PYCOCOEVALCAP")
print("=" * 75)

package = "git+https://github.com/salaniz/pycocoevalcap"

print(f"Installing: {package}")

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", package],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("INSTALL FAILED")
    print(result.stderr[-5000:])
    raise RuntimeError("pycocoevalcap installation failed.")

print("✓ pycocoevalcap installed")

print()
print("✓ ROUGE-SCORE was already installed")
print("✓ pycocoevalcap provides:")
print("  - BLEU-1 to BLEU-4")
print("  - METEOR")
print("  - ROUGE-L")
print("  - CIDEr")
print("  - SPICE")

print("=" * 75)
print("CELL 77 COMPLETE")
print("=" * 75)

CELL 77 — INSTALLING PYCOCOEVALCAP
Installing: git+https://github.com/salaniz/pycocoevalcap
✓ pycocoevalcap installed

✓ ROUGE-SCORE was already installed
✓ pycocoevalcap provides:
  - BLEU-1 to BLEU-4
  - METEOR
  - ROUGE-L
  - CIDEr
  - SPICE
CELL 77 COMPLETE


In [ ]:
# ============================================================================
# CELL 78 — VERIFY ALL CAPTION METRIC EVALUATORS
# ============================================================================

print("=" * 75)
print("CELL 78 — VERIFYING EVALUATION TOOLKIT")
print("=" * 75)

# --------------------------------------------------------------------------
# NLTK
# --------------------------------------------------------------------------

import nltk

from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score

print("✓ NLTK BLEU")
print("✓ NLTK METEOR")

# --------------------------------------------------------------------------
# ROUGE
# --------------------------------------------------------------------------

from rouge_score import rouge_scorer

print("✓ ROUGE-L")

# --------------------------------------------------------------------------
# COCO caption evaluation
# --------------------------------------------------------------------------

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

print("✓ COCO BLEU")
print("✓ COCO METEOR")
print("✓ COCO ROUGE-L")
print("✓ COCO CIDEr")
print("✓ COCO SPICE")

# --------------------------------------------------------------------------
# Check Java — required by the standard SPICE implementation
# --------------------------------------------------------------------------

import shutil
import subprocess

java_path = shutil.which("java")

if java_path is None:
    print("✗ Java NOT FOUND — SPICE cannot run yet")
else:
    print(f"✓ Java found: {java_path}")

    result = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True
    )

    version_output = result.stderr if result.stderr else result.stdout
    print(version_output.splitlines()[0] if version_output else "Java version detected")

print()
print("=" * 75)
print("CELL 78 COMPLETE")
print("=" * 75)

CELL 78 — VERIFYING EVALUATION TOOLKIT
✓ NLTK BLEU
✓ NLTK METEOR
✓ ROUGE-L
✓ COCO BLEU
✓ COCO METEOR
✓ COCO ROUGE-L
✓ COCO CIDEr
✓ COCO SPICE
✓ Java found: /usr/bin/java
openjdk version "21.0.12" 2026-07-21

CELL 78 COMPLETE


In [ ]:
# ============================================================================
# CELL 79 — PREPARE COCO-STYLE PREDICTIONS AND REFERENCES
# ============================================================================

import json
from pathlib import Path
import pandas as pd

print("=" * 75)
print("CELL 79 — PREPARING EVALUATION DATA")
print("=" * 75)

# --------------------------------------------------------------------------
# 1. One generated caption per image
# --------------------------------------------------------------------------

prediction_df = (
    generated_df
    .drop_duplicates(subset=["image_id"], keep="first")
    .copy()
)

prediction_df["image_id"] = prediction_df["image_id"].astype(str)

print("Generated rows:", len(generated_df))
print("Unique predictions:", len(prediction_df))

# --------------------------------------------------------------------------
# 2. Verify one prediction exists for every test image
# --------------------------------------------------------------------------

assert len(prediction_df) == 3988, (
    f"Expected 3988 test-image predictions, got {len(prediction_df)}"
)

# --------------------------------------------------------------------------
# 3. Build COCO evaluator dictionaries
#
# COCO caption evaluator expects:
#
# gts = {
#     image_id: [reference_caption_1, reference_caption_2, ...]
# }
#
# res = {
#     image_id: [generated_caption]
# }
# --------------------------------------------------------------------------

gts = {}
res = {}

for _, row in prediction_df.iterrows():

    image_id = str(row["image_id"])

    # Human references
    references = reference_dict[image_id]

    # Generated caption
    generated_caption = str(
        row["generated_caption"]
    ).strip().lower()

    gts[image_id] = references
    res[image_id] = [generated_caption]

# --------------------------------------------------------------------------
# 4. Verify structure
# --------------------------------------------------------------------------

assert len(gts) == 3988
assert len(res) == 3988

reference_count_check = {
    image_id: len(refs)
    for image_id, refs in gts.items()
}

assert all(
    count == 5
    for count in reference_count_check.values()
)

assert all(
    len(res[image_id]) == 1
    for image_id in res
)

# --------------------------------------------------------------------------
# 5. Display examples
# --------------------------------------------------------------------------

print()
print("Evaluation images:", len(gts))
print("References per image: 5")
print("Predictions per image: 1")

print()
print("Sample evaluation pairs:")

for i, image_id in enumerate(list(gts.keys())[:3]):

    print()
    print(f"IMAGE: {image_id}")

    print("Generated:")
    print(" ", res[image_id][0])

    print("References:")
    for j, ref in enumerate(gts[image_id], 1):
        print(f"  {j}. {ref}")

# --------------------------------------------------------------------------
# 6. Save evaluation data
# --------------------------------------------------------------------------

EVAL_DIR = (
    DRIVE_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "Evaluation"
)

EVAL_DIR.mkdir(parents=True, exist_ok=True)

with open(EVAL_DIR / "test_references.json", "w", encoding="utf-8") as f:
    json.dump(gts, f, ensure_ascii=False, indent=2)

with open(EVAL_DIR / "test_predictions.json", "w", encoding="utf-8") as f:
    json.dump(res, f, ensure_ascii=False, indent=2)

prediction_df.to_csv(
    EVAL_DIR / "test_predictions.csv",
    index=False
)

print()
print("Saved evaluation files to:")
print(EVAL_DIR)

print()
print("✓ One prediction per image")
print("✓ Five human references per image")
print("✓ COCO evaluation dictionaries ready")
print("=" * 75)
print("CELL 79 COMPLETE")
print("=" * 75)

CELL 79 — PREPARING EVALUATION DATA
Generated rows: 19940
Unique predictions: 3988

Evaluation images: 3988
References per image: 5
Predictions per image: 1

Sample evaluation pairs:

IMAGE: Flickr8K/101654506_8eb26cfb60.jpg
Generated:
  a dog is running through the snow
References:
  1. a brown and white dog is running through the snow .
  2. a dog is running in the snow
  3. a dog running through snow .
  4. a white and brown dog is running through a snow covered field .
  5. the white and brown dog is running over the surface of the snow .

IMAGE: Flickr8K/1032460886_4a598ed535.jpg
Generated:
  a man is sitting in a chair in front of a large building
References:
  1. a man is standing in front of a skyscraper
  2. a man stands in front of a skyscraper .
  3. a man stands in front of a very tall building .
  4. behind the man in a red shirt stands a large skyscraper .
  5. there is a skyscraper in the distance with a man walking in front of the camera .

IMAGE: Flickr8K/1042590306_95

In [ ]:
# ============================================================================
# CELL 81 — DIAGNOSE METEOR JAVA OUTPUT
# ============================================================================

import inspect
import pycocoevalcap.meteor.meteor as meteor_module

print("=" * 75)
print("CELL 81 — METEOR DIAGNOSTICS")
print("=" * 75)

meteor_file = inspect.getfile(meteor_module)

print("METEOR wrapper:")
print(meteor_file)

print()
print("METEOR command used by wrapper:")

from pycocoevalcap.meteor.meteor import Meteor

meteor_test = Meteor()

print(meteor_test.meteor_cmd)

print()
print("Java process running:", meteor_test.meteor_p.poll() is None)

print()
print("Testing one small METEOR SCORE request...")

test_refs = [
    "a dog is running through the snow",
    "a dog is running in the snow"
]

test_hyp = "a dog is running through the snow"

# Send a SCORE command exactly as the wrapper does
score_line = " ||| ".join(
    (
        "SCORE",
        " ||| ".join(test_refs),
        test_hyp
    )
)

meteor_test.meteor_p.stdin.write(
    (score_line + "\n").encode()
)

meteor_test.meteor_p.stdin.flush()

raw_output = meteor_test.meteor_p.stdout.readline()

print()
print("Raw Java output:")
print(repr(raw_output))

print()
print("Decoded output:")
print(raw_output.decode(errors="replace"))

print()
print("✓ Diagnostic request completed")

# Clean up
try:
    meteor_test.meteor_p.stdin.close()
    meteor_test.meteor_p.kill()
    meteor_test.meteor_p.wait()
except Exception:
    pass

print("=" * 75)
print("CELL 81 COMPLETE")
print("=" * 75)

CELL 81 — METEOR DIAGNOSTICS
METEOR wrapper:
/usr/local/lib/python3.13/dist-packages/pycocoevalcap/meteor/meteor.py

METEOR command used by wrapper:
['java', '-jar', '-Xmx2G', 'meteor-1.5.jar', '-', '-', '-stdio', '-l', 'en', '-norm']

Java process running: True

Testing one small METEOR SCORE request...

Raw Java output:
b"[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate\n"

Decoded output:
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


✓ Diagnostic request completed
CELL 81 COMPLETE


In [ ]:
# ============================================================================
# CELL 82 — METEOR STDOUT / STDERR DIAGNOSTIC
# ============================================================================

import subprocess
from pathlib import Path

print("=" * 75)
print("CELL 82 — METEOR JAVA OUTPUT DIAGNOSTIC")
print("=" * 75)

meteor_dir = Path(
    "/usr/local/lib/python3.13/dist-packages/pycocoevalcap/meteor"
)

meteor_jar = meteor_dir / "meteor-1.5.jar"

print("METEOR directory:")
print(meteor_dir)

print()
print("METEOR JAR exists:", meteor_jar.exists())
print("METEOR JAR:")
print(meteor_jar)

assert meteor_jar.exists(), "meteor-1.5.jar not found."

# --------------------------------------------------------------------------
# Start Java directly
# --------------------------------------------------------------------------

process = subprocess.Popen(
    [
        "java",
        "-Xmx2G",
        "-jar",
        str(meteor_jar),
        "-",
        "-",
        "-stdio",
        "-l",
        "en",
        "-norm"
    ],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# --------------------------------------------------------------------------
# Send a single SCORE request
# --------------------------------------------------------------------------

references = [
    "a dog is running through the snow",
    "a dog is running in the snow"
]

hypothesis = "a dog is running through the snow"

command = (
    "SCORE "
    + " ||| ".join(references)
    + " ||| "
    + hypothesis
    + "\n"
)

process.stdin.write(command.encode("utf-8"))
process.stdin.flush()

# --------------------------------------------------------------------------
# Read output
# --------------------------------------------------------------------------

stdout_line = process.stdout.readline()
stderr_available = process.stderr.peek() if hasattr(process.stderr, "peek") else b""

print()
print("STDOUT first line:")
print(repr(stdout_line))

print()
print("STDERR available:")
print(repr(stderr_available[:2000]))

# --------------------------------------------------------------------------
# Send EXIT
# --------------------------------------------------------------------------

try:
    process.stdin.write(b"EXIT\n")
    process.stdin.flush()
except Exception:
    pass

try:
    process.stdin.close()
except Exception:
    pass

try:
    process.terminate()
    process.wait(timeout=5)
except Exception:
    process.kill()

print()
print("✓ Direct METEOR diagnostic completed")
print("=" * 75)
print("CELL 82 COMPLETE")
print("=" * 75)


CELL 82 — METEOR JAVA OUTPUT DIAGNOSTIC
METEOR directory:
/usr/local/lib/python3.13/dist-packages/pycocoevalcap/meteor

METEOR JAR exists: True
METEOR JAR:
/usr/local/lib/python3.13/dist-packages/pycocoevalcap/meteor/meteor-1.5.jar


KeyboardInterrupt: 

In [ ]:
# ============================================================================
# CELL 82A — CHECK AVAILABLE JAVA VERSIONS
# ============================================================================

import os
import glob
import shutil
import subprocess

print("=" * 75)
print("CELL 82A — CHECKING JAVA VERSIONS")
print("=" * 75)

print("Current Java:")
print(shutil.which("java"))

result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(result.stderr if result.stderr else result.stdout)

print("Possible Java installations:")

java_candidates = []

for pattern in [
    "/usr/lib/jvm/*",
    "/usr/lib/jvm/*/bin/java",
    "/opt/*/bin/java",
    "/usr/local/*/bin/java"
]:
    java_candidates.extend(glob.glob(pattern))

for path in sorted(set(java_candidates)):
    print(" ", path)

print("=" * 75)
print("CELL 82A COMPLETE")
print("=" * 75)

CELL 82A — CHECKING JAVA VERSIONS
Current Java:
/usr/bin/java
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)

Possible Java installations:
  /usr/lib/jvm/java-1.21.0-openjdk-amd64
  /usr/lib/jvm/java-1.21.0-openjdk-amd64/bin/java
  /usr/lib/jvm/java-21-openjdk-amd64
  /usr/lib/jvm/java-21-openjdk-amd64/bin/java
CELL 82A COMPLETE


In [ ]:
# ============================================================================
# CELL 83 — METEOR EVALUATION (NLTK IMPLEMENTATION)
# ============================================================================

import time
import numpy as np
from nltk.translate.meteor_score import meteor_score

print("=" * 75)
print("CELL 83 — CALCULATING METEOR")
print("=" * 75)

start_time = time.time()

meteor_values = []

for image_id in gts:

    # Tokenize the 5 human references
    reference_tokens = [
        ref.split()
        for ref in gts[image_id]
    ]

    # Tokenize the generated caption
    hypothesis_tokens = res[image_id][0].split()

    # METEOR for this image
    score = meteor_score(
        reference_tokens,
        hypothesis_tokens
    )

    meteor_values.append(score)

meteor_mean = float(np.mean(meteor_values))

# Preserve the previously calculated BLEU scores
if "scores" not in globals():
    scores = {}

scores["METEOR"] = meteor_mean

print()
print(f"Images evaluated: {len(meteor_values)}")
print(f"METEOR: {meteor_mean:.4f}")
print(f"Time: {time.time() - start_time:.2f}s")

# Basic validity checks
assert len(meteor_values) == 3988
assert np.isfinite(meteor_mean)
assert 0.0 <= meteor_mean <= 1.0

print()
print("✓ METEOR calculated successfully")
print("✓ No Java process used")
print("✓ Score is the mean METEOR across the 3,988 test images")

print("=" * 75)
print("CELL 83 COMPLETE")
print("=" * 75)

CELL 83 — CALCULATING METEOR

Images evaluated: 3988
METEOR: 0.3268
Time: 16.42s

✓ METEOR calculated successfully
✓ No Java process used
✓ Score is the mean METEOR across the 3,988 test images
CELL 83 COMPLETE


In [ ]:
# ============================================================================
# CELL 84 — ROUGE-L EVALUATION
# ============================================================================

import time
import numpy as np

print("=" * 75)
print("CELL 84 — CALCULATING ROUGE-L")
print("=" * 75)

start_time = time.time()

def lcs_length(seq1, seq2):
    """Calculate Longest Common Subsequence length."""

    m = len(seq1)
    n = len(seq2)

    dp = [0] * (n + 1)

    for i in range(1, m + 1):
        previous = 0

        for j in range(1, n + 1):
            current = dp[j]

            if seq1[i - 1] == seq2[j - 1]:
                dp[j] = previous + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])

            previous = current

    return dp[n]


def rouge_l_for_caption(hypothesis, references):
    """ROUGE-L using the best matching reference."""

    hypothesis_tokens = hypothesis.split()

    if not hypothesis_tokens:
        return 0.0

    best_f1 = 0.0

    for reference in references:

        reference_tokens = reference.split()

        if not reference_tokens:
            continue

        lcs = lcs_length(
            hypothesis_tokens,
            reference_tokens
        )

        precision = lcs / len(hypothesis_tokens)
        recall = lcs / len(reference_tokens)

        if precision + recall == 0:
            f1 = 0.0
        else:
            f1 = (
                2 * precision * recall
                / (precision + recall)
            )

        best_f1 = max(best_f1, f1)

    return best_f1


rouge_values = []

for image_id in gts:

    hypothesis = res[image_id][0]
    references = gts[image_id]

    score = rouge_l_for_caption(
        hypothesis,
        references
    )

    rouge_values.append(score)

rouge_l_mean = float(np.mean(rouge_values))

scores["ROUGE-L"] = rouge_l_mean

print()
print(f"Images evaluated: {len(rouge_values)}")
print(f"ROUGE-L: {rouge_l_mean:.4f}")
print(f"Time: {time.time() - start_time:.2f}s")

assert len(rouge_values) == 3988
assert np.isfinite(rouge_l_mean)
assert 0.0 <= rouge_l_mean <= 1.0

print()
print("✓ ROUGE-L calculated successfully")

print("=" * 75)
print("CELL 84 COMPLETE")
print("=" * 75)

CELL 84 — CALCULATING ROUGE-L

Images evaluated: 3988
ROUGE-L: 0.3904
Time: 1.80s

✓ ROUGE-L calculated successfully
CELL 84 COMPLETE


In [ ]:
# ============================================================================
# CELL 85 — CIDEr EVALUATION
# ============================================================================

import time
import numpy as np
from pycocoevalcap.cider.cider import Cider

print("=" * 75)
print("CELL 85 — CALCULATING CIDEr")
print("=" * 75)

start_time = time.time()

cider_scorer = Cider()

cider_score, cider_per_image = cider_scorer.compute_score(
    gts,
    res
)

scores["CIDEr"] = float(cider_score)

print()
print(f"Images evaluated: {len(gts)}")
print(f"CIDEr: {cider_score:.4f}")
print(f"Time: {time.time() - start_time:.2f}s")

assert np.isfinite(cider_score)
assert cider_score >= 0.0

print()
print("✓ CIDEr calculated successfully")

print("=" * 75)
print("CELL 85 COMPLETE")
print("=" * 75)

CELL 85 — CALCULATING CIDEr

Images evaluated: 3988
CIDEr: 0.3318
Time: 9.27s

✓ CIDEr calculated successfully
CELL 85 COMPLETE


In [ ]:
# ============================================================================
# CELL 86A — SPICE SINGLE-IMAGE DIAGNOSTIC
# ============================================================================

import time
import traceback

from pycocoevalcap.spice.spice import Spice

print("=" * 75)
print("CELL 86A — SPICE SINGLE-IMAGE DIAGNOSTIC")
print("=" * 75)

# Use one already-prepared test image
test_id = next(iter(gts))

test_gts = {
    test_id: gts[test_id]
}

test_res = {
    test_id: res[test_id]
}

print("Test image:", test_id)
print()
print("Generated:")
print(" ", test_res[test_id][0])

print()
print("References:")
for i, ref in enumerate(test_gts[test_id], 1):
    print(f"  {i}. {ref}")

print()
print("Starting SPICE on ONE image...")
print()

start = time.time()

try:
    spice_scorer_test = Spice()

    score, per_image = spice_scorer_test.compute_score(
        test_gts,
        test_res
    )

    print()
    print("✓ SPICE SINGLE-IMAGE TEST PASSED")
    print(f"SPICE: {float(score):.4f}")
    print(f"Time: {time.time() - start:.2f}s")

except Exception as e:

    print()
    print("✗ SPICE SINGLE-IMAGE TEST FAILED")
    print("Error type:", type(e).__name__)
    print("Error:", str(e))

    print()
    print("Full traceback:")
    traceback.print_exc()

print()
print("=" * 75)
print("CELL 86A COMPLETE")
print("=" * 75)

CELL 86A — SPICE SINGLE-IMAGE DIAGNOSTIC
Test image: Flickr8K/101654506_8eb26cfb60.jpg

Generated:
  a dog is running through the snow

References:
  1. a brown and white dog is running through the snow .
  2. a dog is running in the snow
  3. a dog running through snow .
  4. a white and brown dog is running through a snow covered field .
  5. the white and brown dog is running over the surface of the snow .

Starting SPICE on ONE image...


✗ SPICE SINGLE-IMAGE TEST FAILED
Error type: CalledProcessError
Error: Command '['java', '-jar', '-Xmx8G', 'spice-1.0.jar', '/usr/local/lib/python3.13/dist-packages/pycocoevalcap/spice/tmp/tmp22z_ru5y', '-cache', '/usr/local/lib/python3.13/dist-packages/pycocoevalcap/spice/cache', '-out', '/usr/local/lib/python3.13/dist-packages/pycocoevalcap/spice/tmp/tmpr03ioudh', '-subset', '-silent']' returned non-zero exit status 1.

Full traceback:

CELL 86A COMPLETE


Traceback (most recent call last):
  File "/tmp/ipykernel_1090/2593515537.py", line 44, in <cell line: 0>
    score, per_image = spice_scorer_test.compute_score(
                       ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        test_gts,
        ^^^^^^^^^
        test_res
        ^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/pycocoevalcap/spice/spice.py", line 75, in compute_score
    subprocess.check_call(spice_cmd,
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
        cwd=os.path.dirname(os.path.abspath(__file__)))
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 419, in check_call
    raise CalledProcessError(retcode, cmd)
subprocess.CalledProcessError: Command '['java', '-jar', '-Xmx8G', 'spice-1.0.jar', '/usr/local/lib/python3.13/dist-packages/pycocoevalcap/spice/tmp/tmp22z_ru5y', '-cache', '/usr/local/lib/python3.13/dist-packages/pycocoevalcap/spice/cache', '-out', '/usr/local/lib/python3.13/dist-packages/pycocoe

In [ ]:
# ============================================================================
# CELL 87 — SAVE FINAL VALID EVALUATION RESULTS
# ============================================================================

import json
from pathlib import Path
from datetime import datetime

print("=" * 75)
print("CELL 87 — SAVING EVALUATION RESULTS")
print("=" * 75)

# --------------------------------------------------------------------------
# Record only metrics that were successfully evaluated
# --------------------------------------------------------------------------

final_scores = {
    "BLEU-1": float(scores["BLEU-1"]),
    "BLEU-2": float(scores["BLEU-2"]),
    "BLEU-3": float(scores["BLEU-3"]),
    "BLEU-4": float(scores["BLEU-4"]),
    "METEOR": float(scores["METEOR"]),
    "ROUGE-L": float(scores["ROUGE-L"]),
    "CIDEr": float(scores["CIDEr"]),
    "SPICE": None
}

evaluation_metadata = {
    "model": "Object-Aware Transformer",
    "detector": "RT-DETR-X",
    "dataset": "Flickr8K + Flickr30K",
    "test_images": 3988,
    "references_per_image": 5,
    "prediction_per_image": 1,
    "caption_max_length": 32,
    "object_top_k": 10,
    "object_confidence_threshold": 0.30,
    "metrics": final_scores,
    "SPICE_status": "Unavailable due to SPICE Java/CoreNLP runtime failure under Java 21",
    "METEOR_implementation": "NLTK",
    "evaluation_timestamp": datetime.now().isoformat()
}

# --------------------------------------------------------------------------
# Save
# --------------------------------------------------------------------------

results_path = (
    EVAL_DIR / "final_evaluation_results.json"
)

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_metadata,
        f,
        indent=2
    )

# --------------------------------------------------------------------------
# Display
# --------------------------------------------------------------------------

print()
print("FINAL VALID RESULTS")
print("-" * 40)

for metric, value in final_scores.items():

    if value is None:
        print(f"{metric:10s}: NOT AVAILABLE")
    else:
        print(f"{metric:10s}: {value:.4f}")

print()
print("Saved to:")
print(results_path)

print()
print("✓ Valid metrics saved")
print("✓ SPICE explicitly recorded as unavailable")
print("✓ No fabricated SPICE value")
print("=" * 75)
print("CELL 87 COMPLETE")
print("=" * 75)

CELL 87 — SAVING EVALUATION RESULTS

FINAL VALID RESULTS
----------------------------------------
BLEU-1    : 0.5773
BLEU-2    : 0.3879
BLEU-3    : 0.2546
BLEU-4    : 0.1692
METEOR    : 0.3268
ROUGE-L   : 0.3904
CIDEr     : 0.3318
SPICE     : NOT AVAILABLE

Saved to:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/Evaluation/final_evaluation_results.json

✓ Valid metrics saved
✓ SPICE explicitly recorded as unavailable
✓ No fabricated SPICE value
CELL 87 COMPLETE


In [ ]:
# ============================================================================
# CELL 88 — QUALITATIVE CAPTION EVALUATION
# ============================================================================

import random

print("=" * 75)
print("CELL 88 — QUALITATIVE CAPTION EVALUATION")
print("=" * 75)

# Reproducible selection
random.seed(42)

available_images = list(gts.keys())

selected_images = random.sample(
    available_images,
    min(10, len(available_images))
)

print(f"Selected images: {len(selected_images)}")
print()

for idx, image_id in enumerate(selected_images, 1):

    print("=" * 75)
    print(f"EXAMPLE {idx}")
    print("=" * 75)

    print("Image ID:")
    print(image_id)

    print()
    print("GENERATED CAPTION:")
    print("→", res[image_id][0])

    print()
    print("GROUND-TRUTH CAPTIONS:")

    for ref_idx, reference in enumerate(gts[image_id], 1):
        print(f"{ref_idx}. {reference}")

    print()

print("=" * 75)
print("CELL 88 COMPLETE")
print("=" * 75)

CELL 88 — QUALITATIVE CAPTION EVALUATION
Selected images: 10

EXAMPLE 1
Image ID:
Flickr30K/3954079669.jpg

GENERATED CAPTION:
→ a man in a blue shirt is carrying a large bag

GROUND-TRUTH CAPTIONS:
1. musician sitting on the ground plays his guitar as crowds of people walk around him disregarding the sign in his case which states " tell me why my music sucks " .
2. a street musician sitting playing guitar with an open case and a sign reading " tell me why my music sucks " .
3. street musicians are sitting on the ground as passersby go about their business .
4. 2 young people in the middle of a crowded area are playing music .
5. a street musician playing his guitar on the street .

EXAMPLE 2
Image ID:
Flickr8K/3169777863_d745865784.jpg

GENERATED CAPTION:
→ a group of young girls in matching outfits are dancing in a circle

GROUND-TRUTH CAPTIONS:
1. a group of dancers wearing black spandex are dancing in a studio .
2. a group of men and women in a dancing studio , their arms reaching 

In [ ]:
# ============================================================================
# CELL 89 — VERIFY CORRECTED VOCABULARY PROVENANCE
# ============================================================================

from pathlib import Path
import json

print("=" * 75)
print("CELL 89 — VERIFYING CORRECTED VOCABULARY")
print("=" * 75)

DRIVE_ROOT = Path("/content/drive/MyDrive/Image_captioning")

VOCAB_FILE = (
    DRIVE_ROOT
    / "Outputs"
    / "Structured_Captioning"
    / "caption_vocabulary_corrected.json"
)

SEQ_FILE = (
    DRIVE_ROOT
    / "Features"
    / "Structured_Object"
    / "caption_sequences_corrected.npz"
)

print("Vocabulary file:")
print(VOCAB_FILE)

print()
print("Sequence file:")
print(SEQ_FILE)

assert VOCAB_FILE.exists(), "Corrected vocabulary file not found."
assert SEQ_FILE.exists(), "Corrected caption sequence file not found."

# --------------------------------------------------------------------------
# Load vocabulary
# --------------------------------------------------------------------------

with open(VOCAB_FILE, "r", encoding="utf-8") as f:
    vocab = json.load(f)

if "caption_to_id" in vocab:
    caption_to_id_check = vocab["caption_to_id"]
else:
    caption_to_id_check = vocab

caption_to_id_check = {
    str(k): int(v)
    for k, v in caption_to_id_check.items()
}

print()
print("Vocabulary size:", len(caption_to_id_check))

# --------------------------------------------------------------------------
# Inspect notebook/runtime provenance variables if they still exist
# --------------------------------------------------------------------------

provenance_vars = [
    "caption_to_id",
    "caption_sequences",
    "caption_image_ids",
    "train_caption_indices",
    "val_caption_indices",
    "test_caption_indices"
]

print()
print("Relevant runtime variables:")

for name in provenance_vars:
    if name in globals():
        value = globals()[name]

        try:
            shape = value.shape
            print(f"✓ {name}: shape={shape}")
        except AttributeError:
            try:
                print(f"✓ {name}: length={len(value)}")
            except TypeError:
                print(f"✓ {name}: available")
    else:
        print(f"✗ {name}: not available")

print()
print("=" * 75)
print("IMPORTANT")
print("=" * 75)

print(
    "This cell does NOT assume how the vocabulary was built."
)
print(
    "It only checks the saved vocabulary and current runtime evidence."
)
print(
    "We need this provenance check before treating the evaluation as "
    "publication-ready."
)

print("=" * 75)
print("CELL 89 COMPLETE")
print("=" * 75)

CELL 89 — VERIFYING CORRECTED VOCABULARY
Vocabulary file:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/caption_vocabulary_corrected.json

Sequence file:
/content/drive/MyDrive/Image_captioning/Features/Structured_Object/caption_sequences_corrected.npz

Vocabulary size: 14117

Relevant runtime variables:
✓ caption_to_id: length=14117
✓ caption_sequences: shape=(199369, 32)
✓ caption_image_ids: shape=(199369,)
✓ train_caption_indices: length=159494
✓ val_caption_indices: length=19935
✓ test_caption_indices: length=19940

IMPORTANT
This cell does NOT assume how the vocabulary was built.
It only checks the saved vocabulary and current runtime evidence.
We need this provenance check before treating the evaluation as publication-ready.
CELL 89 COMPLETE


In [ ]:
# ============================================================================
# CELL 90 — FIND CORRECTED VOCABULARY CREATION CODE
# ============================================================================

print("=" * 75)
print("CELL 90 — FINDING CORRECTED VOCABULARY CREATION CODE")
print("=" * 75)

# Search IPython command history for relevant vocabulary-generation code
history = list(In)

keywords = [
    "caption_vocabulary_corrected",
    "caption_sequences_corrected",
    "14,117",
    "14117",
    "caption_to_id",
    "minimum_word_frequency"
]

matches = []

for cell_number, cell_code in enumerate(history):

    if not cell_code:
        continue

    code_lower = cell_code.lower()

    if any(keyword.lower() in code_lower for keyword in keywords):
        matches.append(
            (cell_number, cell_code)
        )

print("Matching cells found:", len(matches))

for cell_number, cell_code in matches[-10:]:

    print()
    print("=" * 75)
    print(f"HISTORY CELL: {cell_number}")
    print("=" * 75)

    print(cell_code[:10000])

print()
print("=" * 75)
print("CELL 90 COMPLETE")
print("=" * 75)

CELL 90 — FINDING CORRECTED VOCABULARY CREATION CODE
Matching cells found: 36

HISTORY CELL: 84
print("=" * 75)
print("CELL 60 — SWITCH TO CORRECTED CAPTION VOCABULARY + SEQUENCES")
print("=" * 75)

# ----------------------------------------------------
# Load corrected vocabulary
# ----------------------------------------------------
with open(
    OUTPUT_DIR / "caption_vocabulary_corrected.json",
    "r",
    encoding="utf-8"
) as f:
    corrected_vocab_raw = json.load(f)

caption_to_id = {
    str(k): int(v)
    for k, v in corrected_vocab_raw["caption_to_id"].items()
}

id_to_caption = {
    int(v): str(k)
    for k, v in caption_to_id.items()
}

# ----------------------------------------------------
# Load corrected sequences
# ----------------------------------------------------
corrected_seq_file = (
    FEATURE_DIR / "caption_sequences_corrected.npz"
)

caption_data = np.load(
    corrected_seq_file,
    allow_pickle=True
)

caption_image_ids = caption_data["image_ids"].astype(

In [ ]:
# ============================================================================
# CELL 90 — CHECK VOCABULARY TRAIN-ONLY PROVENANCE
# ============================================================================

from collections import Counter
import pandas as pd
import re

print("=" * 75)
print("CELL 90 — CHECKING VOCABULARY PROVENANCE")
print("=" * 75)

# --------------------------------------------------------------------------
# 1. Load the complete processed captions
# --------------------------------------------------------------------------

captions_check = pd.read_csv(PROCESSED_CAPTIONS)

captions_check["image_id"] = captions_check["image_id"].astype(str)

# --------------------------------------------------------------------------
# 2. Determine train / validation / test images from the CURRENT evaluation
# --------------------------------------------------------------------------

train_image_set_check = set()

# We already reconstructed the split earlier.
# train_caption_indices contains the exact training caption rows.
for idx in train_caption_indices:
    train_image_set_check.add(
        str(caption_image_ids[idx])
    )

print("Training images:", len(train_image_set_check))

# --------------------------------------------------------------------------
# 3. Extract training captions only
# --------------------------------------------------------------------------

train_captions_check = captions_check[
    captions_check["image_id"].isin(train_image_set_check)
]["caption"].fillna("").astype(str)

print("Training caption rows:", len(train_captions_check))

# --------------------------------------------------------------------------
# 4. Build vocabulary from TRAINING captions only
#    Same basic tokenization used by the original pipeline.
# --------------------------------------------------------------------------

train_frequency = Counter()

for text in train_captions_check:
    tokens = text.lower().strip().split()
    train_frequency.update(tokens)

MIN_WORD_FREQ_CHECK = 2

train_vocab_words = {
    word
    for word, frequency in train_frequency.items()
    if frequency >= MIN_WORD_FREQ_CHECK
}

# Add special tokens
train_only_vocab = {
    "<pad>",
    "<start>",
    "<end>",
    "<unk>"
}

train_only_vocab.update(train_vocab_words)

# --------------------------------------------------------------------------
# 5. Compare saved corrected vocabulary against train-only vocabulary
# --------------------------------------------------------------------------

saved_vocab = set(
    str(word)
    for word in caption_to_id.keys()
)

saved_normal_words = {
    word
    for word in saved_vocab
    if word not in {"<pad>", "<start>", "<end>", "<unk>"}
}

extra_saved_words = saved_normal_words - train_vocab_words

missing_saved_words = train_vocab_words - saved_normal_words

print()
print("=" * 75)
print("VOCABULARY COMPARISON")
print("=" * 75)

print(f"Saved corrected vocabulary:       {len(saved_vocab):,}")
print(f"Expected train-only vocabulary:   {len(train_only_vocab):,}")
print(f"Saved normal words:               {len(saved_normal_words):,}")
print(f"Train-only normal words:          {len(train_vocab_words):,}")

print()
print("Saved words NOT present in train-only vocabulary:",
      len(extra_saved_words))

print("Train-only words missing from saved vocabulary:",
      len(missing_saved_words))

# --------------------------------------------------------------------------
# 6. Show examples
# --------------------------------------------------------------------------

print()
print("Examples of saved words absent from training vocabulary:")

for word in sorted(extra_saved_words)[:50]:
    print(" ", word)

# --------------------------------------------------------------------------
# 7. Final interpretation
# --------------------------------------------------------------------------

print()
print("=" * 75)

if len(extra_saved_words) == 0:
    print("✓ RESULT: No evidence of vocabulary leakage.")
    print("✓ Saved vocabulary is consistent with train-only vocabulary.")

else:
    print("⚠ RESULT: Saved vocabulary contains words absent from")
    print("  the training-only vocabulary.")
    print("⚠ This indicates the saved corrected vocabulary may have")
    print("  been constructed using validation/test captions as well.")

print("=" * 75)
print("CELL 90 COMPLETE")
print("=" * 75)

CELL 90 — CHECKING VOCABULARY PROVENANCE
Training images: 31899
Training caption rows: 159494

VOCABULARY COMPARISON
Saved corrected vocabulary:       14,117
Expected train-only vocabulary:   12,666
Saved normal words:               14,113
Train-only normal words:          12,662

Saved words NOT present in train-only vocabulary: 1486
Train-only words missing from saved vocabulary: 35

Examples of saved words absent from training vocabulary:
  'slide
  -ependent
  00
  000
  08
  1299
  161
  1800
  1915
  1923
  1st
  3-wheeler
  35
  36
  4-wheel
  42
  43
  49ers
  57
  63
  661
  99
  `
  aaa
  abarrotes
  abs
  accompaniment
  acrouss
  activists
  advertizing
  affair
  agains
  agony
  agricultural
  aide
  airliner
  airtime
  ale
  alertly
  aligator
  allows
  alpaca
  amazes
  amble
  ammunition
  ampitheater
  amputated
  angele
  angerly
  ankle-high

⚠ RESULT: Saved vocabulary contains words absent from
  the training-only vocabulary.
⚠ This indicates the saved corrected 

In [ ]:
# ============================================================================
# CELL 91 — BUILD CORRECT TRAIN-ONLY VOCABULARY
# ============================================================================

from collections import Counter
import json
import numpy as np

print("=" * 75)
print("CELL 91 — BUILDING TRAIN-ONLY VOCABULARY")
print("=" * 75)

# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------

MIN_WORD_FREQ = 2
SPECIAL = ["<pad>", "<start>", "<end>", "<unk>"]

# --------------------------------------------------------------------------
# Build frequency table from TRAINING CAPTIONS ONLY
# --------------------------------------------------------------------------

train_frequency = Counter()

for text in train_captions_check:
    tokens = text.lower().strip().split()
    train_frequency.update(tokens)

# --------------------------------------------------------------------------
# Keep words meeting minimum frequency
# --------------------------------------------------------------------------

train_words = sorted(
    word
    for word, frequency in train_frequency.items()
    if frequency >= MIN_WORD_FREQ
    and word not in SPECIAL
)

# --------------------------------------------------------------------------
# Create deterministic vocabulary
# --------------------------------------------------------------------------

caption_to_id_clean = {
    token: idx
    for idx, token in enumerate(SPECIAL)
}

for word in train_words:
    if word not in caption_to_id_clean:
        caption_to_id_clean[word] = len(caption_to_id_clean)

id_to_caption_clean = {
    idx: token
    for token, idx in caption_to_id_clean.items()
}

# --------------------------------------------------------------------------
# Statistics
# --------------------------------------------------------------------------

print("Training captions:", len(train_captions_check))
print("Unique tokens before frequency filter:", len(train_frequency))
print("Words after frequency >= 2:", len(train_words))
print("Special tokens:", len(SPECIAL))
print("Final vocabulary size:", len(caption_to_id_clean))

print()
print("Special token IDs:")
for token in SPECIAL:
    print(f"  {token:10} -> {caption_to_id_clean[token]}")

print()
print("Representative tokens:")

for word in [
    "dog",
    "water",
    "street",
    "grass",
    "ball",
    "building",
    "person",
    "woman",
    "man"
]:
    print(
        f"  {word:10} -> "
        f"{caption_to_id_clean.get(word, 'MISSING')}"
    )

# --------------------------------------------------------------------------
# Verify vocabulary is genuinely train-only
# --------------------------------------------------------------------------

saved_clean_words = set(caption_to_id_clean.keys())

assert len(caption_to_id_clean) == 12666

print()
print("✓ Vocabulary size matches expected train-only size: 12,666")
print("✓ Vocabulary constructed from TRAIN captions only")
print("✓ Minimum word frequency:", MIN_WORD_FREQ)

# --------------------------------------------------------------------------
# Save clean vocabulary
# --------------------------------------------------------------------------

CLEAN_VOCAB_FILE = (
    OUTPUT_DIR / "caption_vocabulary_train_only.json"
)

with open(CLEAN_VOCAB_FILE, "w", encoding="utf-8") as f:
    json.dump(
        {
            "caption_to_id": caption_to_id_clean,
            "id_to_caption": {
                str(k): v
                for k, v in id_to_caption_clean.items()
            },
            "special_tokens": SPECIAL,
            "max_len": MAX_LEN,
            "min_word_freq": MIN_WORD_FREQ,
            "vocabulary_source": "training_captions_only",
            "training_image_count": len(train_image_set_check),
            "training_caption_count": len(train_captions_check)
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print()
print("Saved:")
print(CLEAN_VOCAB_FILE)

print()
print("=" * 75)
print("CELL 91 COMPLETE")
print("=" * 75)


CELL 91 — BUILDING TRAIN-ONLY VOCABULARY
Training captions: 159494
Unique tokens before frequency filter: 18965
Words after frequency >= 2: 12662
Special tokens: 4
Final vocabulary size: 12666

Special token IDs:
  <pad>      -> 0
  <start>    -> 1
  <end>      -> 2
  <unk>      -> 3

Representative tokens:
  dog        -> 3345
  water      -> 12214
  street     -> 10736
  grass      -> 4854
  ball       -> 843
  building   -> 1588
  person     -> 7950
  woman      -> 12493
  man        -> 6570

✓ Vocabulary size matches expected train-only size: 12,666
✓ Vocabulary constructed from TRAIN captions only
✓ Minimum word frequency: 2

Saved:
/content/drive/MyDrive/Image_captioning/Outputs/Structured_Captioning/caption_vocabulary_train_only.json

CELL 91 COMPLETE


In [ ]:
# ============================================================================
# CELL 92 — RE-ENCODE ALL CAPTIONS WITH TRAIN-ONLY VOCABULARY
# ============================================================================

print("=" * 75)
print("CELL 92 — ENCODING CAPTIONS WITH CLEAN VOCABULARY")
print("=" * 75)

# --------------------------------------------------------------------------
# Special token IDs
# --------------------------------------------------------------------------

PAD_ID_CLEAN = caption_to_id_clean["<pad>"]
START_ID_CLEAN = caption_to_id_clean["<start>"]
END_ID_CLEAN = caption_to_id_clean["<end>"]
UNK_ID_CLEAN = caption_to_id_clean["<unk>"]

# --------------------------------------------------------------------------
# Encode using the SAME normalization/tokenization convention
# --------------------------------------------------------------------------

def encode_caption_clean(text):

    tokens = str(text).lower().strip().split()

    ids = [START_ID_CLEAN]

    for tok in tokens[:MAX_LEN - 2]:
        ids.append(
            caption_to_id_clean.get(tok, UNK_ID_CLEAN)
        )

    ids.append(END_ID_CLEAN)

    if len(ids) < MAX_LEN:
        ids += [PAD_ID_CLEAN] * (MAX_LEN - len(ids))

    return ids[:MAX_LEN]


# --------------------------------------------------------------------------
# Encode all captions
# --------------------------------------------------------------------------

clean_caption_ids = [
    encode_caption_clean(text)
    for text in captions_check["caption"]
]

clean_caption_sequences = np.asarray(
    clean_caption_ids,
    dtype=np.int32
)

clean_caption_image_ids = (
    captions_check["image_id"]
    .astype(str)
    .to_numpy()
)

# --------------------------------------------------------------------------
# Statistics
# --------------------------------------------------------------------------

unk_count = int(
    (clean_caption_sequences == UNK_ID_CLEAN).sum()
)

non_padding_count = int(
    (clean_caption_sequences != PAD_ID_CLEAN).sum()
)

captions_with_unk = int(
    np.any(
        clean_caption_sequences == UNK_ID_CLEAN,
        axis=1
    ).sum()
)

print("Caption sequence shape:",
      clean_caption_sequences.shape)

print("Image ID count:",
      len(clean_caption_image_ids))

print()
print("Clean vocabulary size:",
      len(caption_to_id_clean))

print("UNK token count:",
      f"{unk_count:,}")

print("Non-padding token count:",
      f"{non_padding_count:,}")

print(
    "Captions containing UNK:",
    f"{captions_with_unk:,}",
    f"/ {len(clean_caption_sequences):,}",
    f"({100 * captions_with_unk / len(clean_caption_sequences):.2f}%)"
)

# --------------------------------------------------------------------------
# Validate
# --------------------------------------------------------------------------

assert clean_caption_sequences.shape == (
    199369,
    MAX_LEN
)

assert len(clean_caption_image_ids) == 199369

assert PAD_ID_CLEAN == 0
assert START_ID_CLEAN == 1
assert END_ID_CLEAN == 2
assert UNK_ID_CLEAN == 3

assert clean_caption_sequences.min() >= 0
assert clean_caption_sequences.max() < len(
    caption_to_id_clean
)

assert np.isfinite(
    clean_caption_sequences
).all()

# --------------------------------------------------------------------------
# Save
# --------------------------------------------------------------------------

CLEAN_SEQUENCE_FILE = (
    FEATURE_DIR / "caption_sequences_train_only.npz"
)

np.savez_compressed(
    CLEAN_SEQUENCE_FILE,
    image_ids=clean_caption_image_ids,
    caption_ids=clean_caption_sequences
)

print()
print("Saved:")
print(CLEAN_SEQUENCE_FILE)

print()
print("✓ 199,369 captions encoded")
print("✓ Train-only vocabulary used")
print("✓ All IDs within vocabulary range")
print("✓ All sequence values finite")
print("✓ Clean caption sequences saved")

print("=" * 75)
print("CELL 92 COMPLETE")
print("=" * 75)

CELL 92 — ENCODING CAPTIONS WITH CLEAN VOCABULARY
Caption sequence shape: (199369, 32)
Image ID count: 199369

Clean vocabulary size: 12666
UNK token count: 9,480
Non-padding token count: 2,990,721
Captions containing UNK: 8,640 / 199,369 (4.33%)

Saved:
/content/drive/MyDrive/Image_captioning/Features/Structured_Object/caption_sequences_train_only.npz

✓ 199,369 captions encoded
✓ Train-only vocabulary used
✓ All IDs within vocabulary range
✓ All sequence values finite
✓ Clean caption sequences saved
CELL 92 COMPLETE


In [ ]:
# ============================================================================
# CELL 93 — SWITCH DATASETS TO TRAIN-ONLY CAPTION SEQUENCES
# ============================================================================

print("=" * 75)
print("CELL 93 — SWITCHING TO CLEAN CAPTION SEQUENCES")
print("=" * 75)

# Activate clean vocabulary
caption_to_id = caption_to_id_clean
id_to_caption = id_to_caption_clean

PAD_ID = PAD_ID_CLEAN
START_ID = START_ID_CLEAN
END_ID = END_ID_CLEAN
UNK_ID = UNK_ID_CLEAN

# Activate clean sequences
caption_image_ids = clean_caption_image_ids
caption_sequences = clean_caption_sequences

# Existing datasets retain their correct image/caption indices.
# Only replace the caption sequence array.

train_dataset.caption_sequences = caption_sequences
val_dataset.caption_sequences = caption_sequences
test_dataset.caption_sequences = caption_sequences

print("Vocabulary size:", len(caption_to_id))
print("Caption sequences:", caption_sequences.shape)

print()
print("Dataset caption indices:")
print("  Train:", len(train_dataset.caption_indices))
print("  Val:  ", len(val_dataset.caption_indices))
print("  Test: ", len(test_dataset.caption_indices))

# --------------------------------------------------------------------------
# Verify split counts
# --------------------------------------------------------------------------

assert len(caption_to_id) == 12666
assert caption_sequences.shape == (199369, 32)

assert len(train_dataset.caption_indices) == 159494
assert len(val_dataset.caption_indices) == 19935
assert len(test_dataset.caption_indices) == 19940

print()
print("✓ Clean vocabulary activated")
print("✓ Clean caption sequences activated")
print("✓ Existing dataset splits preserved")
print("✓ No detection data changed")

print("=" * 75)
print("CELL 93 COMPLETE")
print("=" * 75)

CELL 93 — SWITCHING TO CLEAN CAPTION SEQUENCES
Vocabulary size: 12666
Caption sequences: (199369, 32)

Dataset caption indices:
  Train: 159494
  Val:   19935
  Test:  19940

✓ Clean vocabulary activated
✓ Clean caption sequences activated
✓ Existing dataset splits preserved
✓ No detection data changed
CELL 93 COMPLETE


In [ ]:
# ============================================================================
# CELL 94 — VERIFY CLEAN TRAINING BATCH
# ============================================================================

print("=" * 75)
print("CELL 94 — VERIFYING CLEAN TRAINING BATCH")
print("=" * 75)

batch = next(iter(train_loader))

print("Batch type:", type(batch))

for key, value in batch.items():
    if torch.is_tensor(value):
        print(
            f"{key:15s} "
            f"shape={tuple(value.shape)} "
            f"dtype={value.dtype}"
        )

decoder_input = batch["decoder_input"]
target = batch["target"]

print()
print("Caption ID ranges:")
print("Decoder input min:", int(decoder_input.min()))
print("Decoder input max:", int(decoder_input.max()))
print("Target min:       ", int(target.min()))
print("Target max:       ", int(target.max()))

print()
print("Vocabulary size:", len(caption_to_id))

# --------------------------------------------------------------------------
# Assertions
# --------------------------------------------------------------------------

assert decoder_input.shape == (BATCH_SIZE, MAX_LEN - 1)
assert target.shape == (BATCH_SIZE, MAX_LEN - 1)

assert int(decoder_input.min()) >= 0
assert int(target.min()) >= 0

assert int(decoder_input.max()) < len(caption_to_id)
assert int(target.max()) < len(caption_to_id)

assert torch.isfinite(decoder_input.float()).all()
assert torch.isfinite(target.float()).all()

assert len(caption_to_id) == 12666

print()
print("✓ Batch uses clean 12,666-token vocabulary")
print("✓ Caption shapes correct")
print("✓ All caption IDs within range")
print("✓ No invalid values")

print("=" * 75)
print("CELL 94 COMPLETE")
print("=" * 75)

CELL 94 — VERIFYING CLEAN TRAINING BATCH
Batch type: <class 'dict'>
object_ids      shape=(64, 10) dtype=torch.int64
spatial         shape=(64, 10, 9) dtype=torch.float32
confidence      shape=(64, 10) dtype=torch.float32
object_mask     shape=(64, 10) dtype=torch.bool
decoder_input   shape=(64, 31) dtype=torch.int64
target          shape=(64, 31) dtype=torch.int64

Caption ID ranges:
Decoder input min: 0
Decoder input max: 12628
Target min:        0
Target max:        12628

Vocabulary size: 12666

✓ Batch uses clean 12,666-token vocabulary
✓ Caption shapes correct
✓ All caption IDs within range
✓ No invalid values
CELL 94 COMPLETE


In [ ]:
# ============================================================================
# CELL 95 — INITIALIZE CLEAN OBJECT-AWARE TRANSFORMER
# ============================================================================

print("=" * 75)
print("CELL 95 — INITIALIZING CLEAN TRANSFORMER")
print("=" * 75)

# --------------------------------------------------------------------------
# Fresh model
# --------------------------------------------------------------------------

model = ObjectAwareTransformer(
    object_vocab_size=len(object_to_id),
    caption_vocab_size=len(caption_to_id),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_caption_length=MAX_LEN
).to(DEVICE)

# --------------------------------------------------------------------------
# Optimizer
# --------------------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# --------------------------------------------------------------------------
# Loss
# --------------------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=0.1
)

# --------------------------------------------------------------------------
# Parameter verification
# --------------------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Device:               ", DEVICE)
print("Object vocabulary:    ", len(object_to_id))
print("Caption vocabulary:   ", len(caption_to_id))
print("Total parameters:     ", f"{total_params:,}")
print("Trainable parameters: ", f"{trainable_params:,}")
print("Learning rate:        ", optimizer.param_groups[0]["lr"])
print("Weight decay:         ", optimizer.param_groups[0]["weight_decay"])

all_finite = all(
    torch.isfinite(p).all().item()
    for p in model.parameters()
)

print()
print("All parameters finite:", all_finite)

assert len(caption_to_id) == 12666
assert trainable_params == total_params
assert all_finite

print()
print("✓ Fresh model initialized")
print("✓ Output projection = 12,666 vocabulary classes")
print("✓ No old 14,117-vocabulary checkpoint loaded")
print("✓ Parameters finite")

print("=" * 75)
print("CELL 95 COMPLETE")
print("=" * 75)

CELL 95 — INITIALIZING CLEAN TRANSFORMER


/tmp/ipykernel_1090/3777785277.py:64: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(


Device:                cuda
Object vocabulary:     82
Caption vocabulary:    12666
Total parameters:      12,347,258
Trainable parameters:  12,347,258
Learning rate:         0.0001
Weight decay:          0.0001

All parameters finite: True

✓ Fresh model initialized
✓ Output projection = 12,666 vocabulary classes
✓ No old 14,117-vocabulary checkpoint loaded
✓ Parameters finite
CELL 95 COMPLETE


In [ ]:
# ============================================================================
# CELL 96 — CLEAN MODEL FORWARD + LOSS SANITY CHECK
# ============================================================================

print("=" * 75)
print("CELL 96 — FORWARD + LOSS SANITY CHECK")
print("=" * 75)

model.train()

# --------------------------------------------------------------------------
# Move batch to GPU
# --------------------------------------------------------------------------

object_ids_batch = batch["object_ids"].to(DEVICE)
spatial_batch = batch["spatial"].to(DEVICE)
confidence_batch = batch["confidence"].to(DEVICE)
object_mask_batch = batch["object_mask"].to(DEVICE)
decoder_input_batch = batch["decoder_input"].to(DEVICE)
target_batch = batch["target"].to(DEVICE)

# --------------------------------------------------------------------------
# Forward
# --------------------------------------------------------------------------

with torch.no_grad():

    logits = model(
        object_ids_batch,
        spatial_batch,
        confidence_batch,
        object_mask_batch,
        decoder_input_batch
    )

print("Logits shape:")
print(" ", tuple(logits.shape))

print()
print("Expected:")
print(
    f"  ({BATCH_SIZE}, {MAX_LEN - 1}, {len(caption_to_id)})"
)

print()
print("Logits finite:",
      bool(torch.isfinite(logits).all()))

# --------------------------------------------------------------------------
# Loss
# --------------------------------------------------------------------------

loss = criterion(
    logits.reshape(-1, logits.size(-1)),
    target_batch.reshape(-1)
)

print()
print(f"Initial loss: {loss.item():.6f}")
print("Loss finite:", bool(torch.isfinite(loss)))

# --------------------------------------------------------------------------
# Assertions
# --------------------------------------------------------------------------

assert logits.shape == (
    BATCH_SIZE,
    MAX_LEN - 1,
    12666
)

assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

print()
print("✓ Forward pass successful")
print("✓ Logits match clean vocabulary")
print("✓ Loss is finite")

print("=" * 75)
print("CELL 96 COMPLETE")
print("=" * 75)

CELL 96 — FORWARD + LOSS SANITY CHECK
Logits shape:
  (64, 31, 12666)

Expected:
  (64, 31, 12666)

Logits finite: True

Initial loss: 9.681318
Loss finite: True

✓ Forward pass successful
✓ Logits match clean vocabulary
✓ Loss is finite
CELL 96 COMPLETE


In [ ]:
# ============================================================================
# CELL 97 — CLEAN MODEL 100-STEP TRAINING SANITY CHECK
# ============================================================================

print("=" * 75)
print("CELL 97 — 100-STEP TRAINING SANITY CHECK")
print("=" * 75)

model.train()

sanity_losses = []

data_iterator = iter(train_loader)

for step in range(1, 101):

    try:
        batch = next(data_iterator)
    except StopIteration:
        data_iterator = iter(train_loader)
        batch = next(data_iterator)

    object_ids_batch = batch["object_ids"].to(DEVICE)
    spatial_batch = batch["spatial"].to(DEVICE)
    confidence_batch = batch["confidence"].to(DEVICE)
    object_mask_batch = batch["object_mask"].to(DEVICE)
    decoder_input_batch = batch["decoder_input"].to(DEVICE)
    target_batch = batch["target"].to(DEVICE)

    optimizer.zero_grad(set_to_none=True)

    logits = model(
        object_ids_batch,
        spatial_batch,
        confidence_batch,
        object_mask_batch,
        decoder_input_batch
    )

    loss = criterion(
        logits.reshape(-1, logits.size(-1)),
        target_batch.reshape(-1)
    )

    assert torch.isfinite(loss)

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    sanity_losses.append(loss.item())

    if step % 10 == 0:
        avg_loss = np.mean(sanity_losses[-10:])

        print(
            f"Step {step:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"10-step avg: {avg_loss:.4f}"
        )

# --------------------------------------------------------------------------
# Final checks
# --------------------------------------------------------------------------

first_loss = sanity_losses[0]
last_loss = sanity_losses[-1]
reduction = first_loss - last_loss

print()
print("First loss:", f"{first_loss:.6f}")
print("Last loss: ", f"{last_loss:.6f}")
print("Reduction: ", f"{reduction:.6f}")

assert len(sanity_losses) == 100
assert np.isfinite(sanity_losses).all()
assert reduction > 0

print()
print("✓ 100 optimization steps completed")
print("✓ Loss remained finite")
print("✓ Loss decreased")
print("✓ Clean model is learning")

print("=" * 75)
print("CELL 97 COMPLETE")
print("=" * 75)

CELL 97 — 100-STEP TRAINING SANITY CHECK
Step 010 | Loss: 7.8943 | 10-step avg: 8.6852
Step 020 | Loss: 6.7843 | 10-step avg: 7.2557
Step 030 | Loss: 6.3557 | 10-step avg: 6.4487
Step 040 | Loss: 6.0312 | 10-step avg: 6.1317
Step 050 | Loss: 5.8500 | 10-step avg: 6.0041
Step 060 | Loss: 5.6804 | 10-step avg: 5.8557
Step 070 | Loss: 5.8113 | 10-step avg: 5.7969
Step 080 | Loss: 5.6770 | 10-step avg: 5.7140
Step 090 | Loss: 5.5967 | 10-step avg: 5.6343
Step 100 | Loss: 5.5387 | 10-step avg: 5.5923

First loss: 9.649721
Last loss:  5.538666
Reduction:  4.111055

✓ 100 optimization steps completed
✓ Loss remained finite
✓ Loss decreased
✓ Clean model is learning
CELL 97 COMPLETE


In [ ]:
# ============================================================
# CELL 98A — RESTORE OBJECT-AWARE TRANSFORMER CLASS
# ============================================================

import math
import torch
import torch.nn as nn

print("=" * 75)
print("CELL 98A — RESTORING OBJECT-AWARE TRANSFORMER")
print("=" * 75)


class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len):
        super().__init__()

        position = torch.arange(
            max_len,
            dtype=torch.float32
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32
            )
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model,
            dtype=torch.float32
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class ObjectFeatureEncoder(nn.Module):

    def __init__(
        self,
        object_vocab_size,
        d_model
    ):
        super().__init__()

        self.object_embedding = nn.Embedding(
            object_vocab_size,
            d_model,
            padding_idx=0
        )

        self.spatial_mlp = nn.Sequential(
            nn.Linear(9, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.confidence_mlp = nn.Sequential(
            nn.Linear(1, 128),
            nn.ReLU(),
            nn.Linear(128, d_model)
        )

        self.fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.LayerNorm(d_model)
        )

    def forward(
        self,
        object_ids,
        spatial,
        confidence
    ):

        object_embedding = self.object_embedding(
            object_ids
        )

        spatial_embedding = self.spatial_mlp(
            spatial
        )

        confidence_embedding = self.confidence_mlp(
            confidence.unsqueeze(-1)
        )

        combined = torch.cat(
            [
                object_embedding,
                spatial_embedding,
                confidence_embedding
            ],
            dim=-1
        )

        return self.fusion(combined)


class ObjectAwareTransformer(nn.Module):

    def __init__(
        self,
        object_vocab_size,
        caption_vocab_size,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=1024,
        dropout=0.2,
        max_caption_length=32
    ):
        super().__init__()

        self.object_encoder = ObjectFeatureEncoder(
            object_vocab_size,
            d_model
        )

        self.object_position = PositionalEncoding(
            d_model,
            TOP_K
        )

        self.caption_embedding = nn.Embedding(
            caption_vocab_size,
            d_model,
            padding_idx=PAD_ID
        )

        self.caption_position = PositionalEncoding(
            d_model,
            max_caption_length
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_decoder_layers
        )

        self.output_projection = nn.Linear(
            d_model,
            caption_vocab_size
        )

    @staticmethod
    def generate_causal_mask(size, device):

        mask = torch.triu(
            torch.ones(
                size,
                size,
                device=device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        return mask

    def forward(
        self,
        object_ids,
        spatial,
        confidence,
        object_mask,
        decoder_input
    ):

        object_tokens = self.object_encoder(
            object_ids,
            spatial,
            confidence
        )

        object_tokens = self.object_position(
            object_tokens
        )

        memory = self.transformer_encoder(
            object_tokens,
            src_key_padding_mask=~object_mask
        )

        decoder_tokens = self.caption_embedding(
            decoder_input
        )

        decoder_tokens = self.caption_position(
            decoder_tokens
        )

        target_mask = self.generate_causal_mask(
            decoder_input.size(1),
            decoder_input.device
        )

        decoded = self.transformer_decoder(
            decoder_tokens,
            memory,
            tgt_mask=target_mask,
            memory_key_padding_mask=~object_mask
        )

        logits = self.output_projection(
            decoded
        )

        return logits


print()
print("✓ ObjectFeatureEncoder restored")
print("✓ PositionalEncoding restored")
print("✓ ObjectAwareTransformer restored")
print("✓ Forward signature restored")
print("=" * 75)
print("CELL 98A COMPLETE")
print("=" * 75)

CELL 98A — RESTORING OBJECT-AWARE TRANSFORMER

✓ ObjectFeatureEncoder restored
✓ PositionalEncoding restored
✓ ObjectAwareTransformer restored
✓ Forward signature restored
CELL 98A COMPLETE


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Image_captioning")

print("=" * 75)
print("SEARCHING FOR VOCABULARY FILES")
print("=" * 75)

matches = []

for p in DRIVE_ROOT.rglob("*.json"):
    name = p.name.lower()
    if "vocab" in name:
        matches.append(p)

for p in matches:
    print(p)

print(f"\nFound: {len(matches)} vocabulary JSON files")

SEARCHING FOR VOCABULARY FILES

Found: 0 vocabulary JSON files


In [ ]:
caption_candidates = []
object_candidates = []

for p in DRIVE_ROOT.rglob("*.json"):
    name = p.name.lower()

    if "caption" in name and "vocab" in name:
        caption_candidates.append(p)

    if "object" in name and "vocab" in name:
        object_candidates.append(p)

print("=" * 75)
print("CAPTION VOCABULARIES")
print("=" * 75)

for p in caption_candidates:
    print(p)

print("\n" + "=" * 75)
print("OBJECT VOCABULARIES")
print("=" * 75)

for p in object_candidates:
    print(p)

CAPTION VOCABULARIES

OBJECT VOCABULARIES


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Image_captioning")

json_files = list(DRIVE_ROOT.rglob("*.json"))

print("=" * 75)
print(f"TOTAL JSON FILES FOUND: {len(json_files)}")
print("=" * 75)

for i, p in enumerate(json_files):
    print(f"[{i}] {p}")

TOTAL JSON FILES FOUND: 0


In [ ]:
# ============================================================
# CELL 99 — VERIFY CLEAN BEST CHECKPOINT
# ============================================================

from pathlib import Path
import torch
import json

CHECKPOINT_PATH = DRIVE_ROOT / "Outputs" / "Structured_Captioning" / \
                  "Checkpoints" / "best_object_aware_transformer_train_only.pt"

print("=" * 75)
print("CELL 99 — VERIFYING CLEAN BEST CHECKPOINT")
print("=" * 75)

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found:\n{CHECKPOINT_PATH}"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Checkpoint keys: {list(checkpoint.keys())}")

# Basic metadata checks
if "caption_vocab_size" in checkpoint:
    print(f"Checkpoint caption vocabulary: {checkpoint['caption_vocab_size']}")
    assert checkpoint["caption_vocab_size"] == 12666

if "object_vocab_size" in checkpoint:
    print(f"Checkpoint object vocabulary: {checkpoint['object_vocab_size']}")
    assert checkpoint["object_vocab_size"] == 82

if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif "model_state" in checkpoint:
    state_dict = checkpoint["model_state"]
else:
    raise KeyError("Could not find model state dictionary in checkpoint.")

# Verify output projection shape
projection_keys = [
    k for k in state_dict.keys()
    if "output_projection" in k and k.endswith("weight")
]

assert len(projection_keys) > 0, "Output projection weight not found."

projection_key = projection_keys[0]
projection_shape = tuple(state_dict[projection_key].shape)

print(f"Output projection shape: {projection_shape}")
print("Expected first dimension: 12666")

assert projection_shape[0] == 12666

print()
print("✓ Checkpoint exists")
print("✓ Clean 12,666-token vocabulary confirmed")
print("✓ Object vocabulary = 82")
print("✓ Output projection matches clean vocabulary")
print("✓ Checkpoint is readable")
print("=" * 75)
print("CELL 99 COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# CELL 100 — LOAD CLEAN BEST MODEL
# ============================================================

print("=" * 75)
print("CELL 100 — LOADING CLEAN BEST MODEL")
print("=" * 75)

clean_model = ObjectAwareTransformer(
    object_vocab_size=len(object_to_id),
    caption_vocab_size=len(caption_to_id),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_caption_length=MAX_LEN
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

if "model_state_dict" in checkpoint:
    clean_model.load_state_dict(checkpoint["model_state_dict"])
elif "model_state" in checkpoint:
    clean_model.load_state_dict(checkpoint["model_state"])
else:
    raise KeyError("Model state dictionary not found in checkpoint.")

clean_model.eval()

# Verify all parameters
all_finite = all(
    torch.isfinite(parameter).all().item()
    for parameter in clean_model.parameters()
)

print(f"Device: {DEVICE}")
print(f"Caption vocabulary: {len(caption_to_id)}")
print(f"Object vocabulary: {len(object_to_id)}")
print(f"Parameters: {sum(p.numel() for p in clean_model.parameters()):,}")
print(f"All parameters finite: {all_finite}")

assert len(caption_to_id) == 12666
assert len(object_to_id) == 82
assert all_finite

print()
print("✓ Clean best checkpoint loaded")
print("✓ Model in evaluation mode")
print("✓ No old 14,117-token model used")
print("✓ All parameters finite")
print("=" * 75)
print("CELL 100 COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# CELL 101 — VERIFY TEST DATA FOR CLEAN GENERATION
# ============================================================

print("=" * 75)
print("CELL 101 — VERIFYING TEST DATA")
print("=" * 75)

test_batch = next(iter(test_loader))

required_keys = {
    "image_id",
    "object_ids",
    "spatial",
    "confidence",
    "object_mask",
    "decoder_input",
    "target"
}

print("Batch keys:")
for key in test_batch.keys():
    print(f"  {key}")

assert required_keys.issubset(set(test_batch.keys()))

B = test_batch["object_ids"].shape[0]

print()
print(f"Batch size:       {B}")
print(f"Object IDs:       {tuple(test_batch['object_ids'].shape)}")
print(f"Spatial:          {tuple(test_batch['spatial'].shape)}")
print(f"Confidence:       {tuple(test_batch['confidence'].shape)}")
print(f"Object mask:      {tuple(test_batch['object_mask'].shape)}")
print(f"Decoder input:    {tuple(test_batch['decoder_input'].shape)}")
print(f"Target:           {tuple(test_batch['target'].shape)}")

assert test_batch["object_ids"].shape[1] == TOP_K
assert test_batch["spatial"].shape[1:] == (TOP_K, 9)
assert test_batch["confidence"].shape[1] == TOP_K
assert test_batch["object_mask"].shape[1] == TOP_K
assert test_batch["decoder_input"].shape[1] == MAX_LEN - 1
assert test_batch["target"].shape[1] == MAX_LEN - 1

# Check object IDs
assert test_batch["object_ids"].min().item() >= 0
assert test_batch["object_ids"].max().item() < len(object_to_id)

# Check caption IDs
assert test_batch["decoder_input"].min().item() >= 0
assert test_batch["decoder_input"].max().item() < len(caption_to_id)

assert test_batch["target"].min().item() >= 0
assert test_batch["target"].max().item() < len(caption_to_id)

print()
print(f"Unique test images: {len(test_dataset.split_image_ids):,}")
print(f"Test caption samples: {len(test_dataset):,}")

print()
print("✓ Test loader uses verified dictionary format")
print("✓ Object tensor dimensions correct")
print("✓ Spatial dimensions correct")
print("✓ Caption dimensions correct")
print("✓ All IDs within clean vocabulary ranges")
print("=" * 75)
print("CELL 101 COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# CELL 102 — GENERATE CLEAN TEST CAPTIONS
# ============================================================

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print("=" * 75)
print("CELL 102 — GENERATING CLEAN TEST CAPTIONS")
print("=" * 75)

# ------------------------------------------------------------
# Special token IDs
# ------------------------------------------------------------

PAD_ID = caption_to_id["<pad>"]
START_ID = caption_to_id["<start>"]
END_ID = caption_to_id["<end>"]
UNK_ID = caption_to_id["<unk>"]

id_to_caption_clean = {
    int(v): str(k)
    for k, v in caption_to_id.items()
}

print(f"PAD:   {PAD_ID}")
print(f"START: {START_ID}")
print(f"END:   {END_ID}")
print(f"UNK:   {UNK_ID}")

# ------------------------------------------------------------
# Greedy decoding
# ------------------------------------------------------------

def generate_caption_greedy(
    model,
    object_ids,
    spatial,
    confidence,
    object_mask,
    max_length=MAX_LEN
):
    """
    Autoregressive greedy decoding for the verified
    ObjectAwareTransformer.forward() signature.
    """

    device = object_ids.device
    batch_size = object_ids.size(0)

    decoder_input = torch.full(
        (batch_size, 1),
        START_ID,
        dtype=torch.long,
        device=device
    )

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=device
    )

    with torch.no_grad():

        for _ in range(max_length - 1):

            logits = model(
                object_ids,
                spatial,
                confidence,
                object_mask,
                decoder_input
            )

            # Last generated position
            next_token_logits = logits[:, -1, :]

            next_token = torch.argmax(
                next_token_logits,
                dim=-1
            )

            # Keep END for sequences already finished
            next_token = torch.where(
                finished,
                torch.tensor(END_ID, device=device),
                next_token
            )

            decoder_input = torch.cat(
                [decoder_input, next_token.unsqueeze(1)],
                dim=1
            )

            finished = finished | (next_token == END_ID)

            if finished.all():
                break

    return decoder_input


# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

clean_predictions = []

with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Generating test captions"
    ):

        object_ids = batch["object_ids"].to(DEVICE)
        spatial = batch["spatial"].to(DEVICE)
        confidence = batch["confidence"].to(DEVICE)
        object_mask = batch["object_mask"].to(DEVICE)

        token_ids = generate_caption_greedy(
            clean_model,
            object_ids,
            spatial,
            confidence,
            object_mask,
            max_length=MAX_LEN
        )

        image_ids = batch["image_id"]

        for i in range(token_ids.size(0)):

            ids = token_ids[i].detach().cpu().tolist()

            words = []

            for token_id in ids:

                if token_id == START_ID:
                    continue

                if token_id == END_ID:
                    break

                if token_id == PAD_ID:
                    continue

                word = id_to_caption_clean.get(
                    int(token_id),
                    "<unk>"
                )

                if word != "<unk>":
                    words.append(word)

            caption = " ".join(words).strip()

            clean_predictions.append({
                "image_id": str(image_ids[i]),
                "model_caption": caption
            })

# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

clean_predictions_df = pd.DataFrame(clean_predictions)

print()
print(f"Generated rows: {len(clean_predictions_df):,}")
print(
    f"Unique images:  "
    f"{clean_predictions_df['image_id'].nunique():,}"
)

assert len(clean_predictions_df) == len(test_dataset)
assert (
    clean_predictions_df["image_id"].nunique()
    == len(test_dataset.split_image_ids)
)

assert clean_predictions_df["model_caption"].notna().all()

# ------------------------------------------------------------
# Save raw clean predictions
# ------------------------------------------------------------

EVAL_DIR = DRIVE_ROOT / "Outputs" / "Structured_Captioning" / "Evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PREDICTIONS_PATH = EVAL_DIR / "clean_test_predictions_train_only.csv"

clean_predictions_df.to_csv(
    CLEAN_PREDICTIONS_PATH,
    index=False
)

print()
print("Example predictions:")
print(clean_predictions_df.head(10).to_string(index=False))

print()
print(f"Saved: {CLEAN_PREDICTIONS_PATH}")

print()
print("✓ Caption generation completed")
print("✓ One prediction generated for every test caption sample")
print("✓ Test image coverage verified")
print("✓ Clean vocabulary used")
print("=" * 75)
print("CELL 102 COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# CELL 103 — BUILD CLEAN COCO-STYLE EVALUATION SET
# ============================================================

import json
import pandas as pd

print("=" * 75)
print("CELL 103 — BUILDING CLEAN EVALUATION SET")
print("=" * 75)

CAPTIONS_PATH = PROCESSED_CAPTIONS

captions_df = pd.read_csv(CAPTIONS_PATH)

print(f"Full captions CSV: {captions_df.shape}")

# ------------------------------------------------------------
# Test image IDs
# ------------------------------------------------------------

test_image_ids = set(
    str(x) for x in test_dataset.split_image_ids
)

print(f"Test images: {len(test_image_ids):,}")

# ------------------------------------------------------------
# Build references
# ------------------------------------------------------------

test_refs_df = captions_df[
    captions_df["image_id"].astype(str).isin(test_image_ids)
].copy()

print(f"Test reference rows: {len(test_refs_df):,}")

# Use processed captions for consistent evaluation
assert "processed_caption" in test_refs_df.columns

gts = {}

for image_id, group in test_refs_df.groupby(
    test_refs_df["image_id"].astype(str)
):

    refs = [
        str(x).strip()
        for x in group["processed_caption"].tolist()
        if str(x).strip()
    ]

    gts[image_id] = refs

# ------------------------------------------------------------
# Collapse predictions to one per image
# ------------------------------------------------------------

pred_grouped = (
    clean_predictions_df
    .groupby("image_id", sort=False)["model_caption"]
    .first()
    .to_dict()
)

res = {
    str(image_id): [str(caption)]
    for image_id, caption in pred_grouped.items()
}

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(gts) == len(test_image_ids)
assert len(res) == len(test_image_ids)

reference_counts = {
    image_id: len(refs)
    for image_id, refs in gts.items()
}

print()
print(f"Images with references:  {len(gts):,}")
print(f"Images with predictions: {len(res):,}")

print(
    "Reference counts:",
    pd.Series(list(reference_counts.values())).value_counts().sort_index().to_dict()
)

# Flickr8K/Flickr30K combined test should normally provide 5 refs/image
assert all(count == 5 for count in reference_counts.values())

# ------------------------------------------------------------
# Save JSON files
# ------------------------------------------------------------

CLEAN_GTS_PATH = EVAL_DIR / "clean_test_references_train_only.json"
CLEAN_RES_PATH = EVAL_DIR / "clean_test_predictions_train_only.json"

with open(CLEAN_GTS_PATH, "w", encoding="utf-8") as f:
    json.dump(gts, f, ensure_ascii=False, indent=2)

with open(CLEAN_RES_PATH, "w", encoding="utf-8") as f:
    json.dump(res, f, ensure_ascii=False, indent=2)

print()
print(f"Saved references:  {CLEAN_GTS_PATH}")
print(f"Saved predictions: {CLEAN_RES_PATH}")

print()
print("✓ One prediction per unique test image")
print("✓ Five ground-truth references per test image")
print("✓ Image counts match")
print("✓ COCO-style evaluation dictionaries ready")
print("=" * 75)
print("CELL 103 COMPLETE")
print("=" * 75)

In [ ]:
# ============================================================
# CELL 104 — FINAL CLEAN MODEL EVALUATION
# ============================================================

import json
import math
import numpy as np

from nltk.translate.bleu_score import (
    corpus_bleu,
    SmoothingFunction
)
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

print("=" * 75)
print("CELL 104 — FINAL CLEAN MODEL EVALUATION")
print("=" * 75)

# ------------------------------------------------------------
# Load saved evaluation dictionaries
# ------------------------------------------------------------

with open(CLEAN_GTS_PATH, "r", encoding="utf-8") as f:
    gts = json.load(f)

with open(CLEAN_RES_PATH, "r", encoding="utf-8") as f:
    res = json.load(f)

assert set(gts.keys()) == set(res.keys())

image_ids = list(gts.keys())

print(f"Evaluation images: {len(image_ids):,}")

# ------------------------------------------------------------
# Tokenized references / hypotheses
# ------------------------------------------------------------

references = []
hypotheses = []

for image_id in image_ids:

    refs = [
        ref.split()
        for ref in gts[image_id]
    ]

    hyp = res[image_id][0].split()

    references.append(refs)
    hypotheses.append(hyp)

# ------------------------------------------------------------
# BLEU
# ------------------------------------------------------------

smoothing = SmoothingFunction().method1

bleu1 = corpus_bleu(
    references,
    hypotheses,
    weights=(1.0, 0, 0, 0),
    smoothing_function=smoothing
)

bleu2 = corpus_bleu(
    references,
    hypotheses,
    weights=(0.5, 0.5, 0, 0),
    smoothing_function=smoothing
)

bleu3 = corpus_bleu(
    references,
    hypotheses,
    weights=(1/3, 1/3, 1/3, 0),
    smoothing_function=smoothing
)

bleu4 = corpus_bleu(
    references,
    hypotheses,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smoothing
)

# ------------------------------------------------------------
# METEOR — NLTK
# ------------------------------------------------------------

meteor_scores = []

for image_id in image_ids:

    hyp = res[image_id][0].split()

    ref_tokens = [
        ref.split()
        for ref in gts[image_id]
    ]

    score = max(
        meteor_score(
            [ref],
            hyp
        )
        for ref in ref_tokens
    )

    meteor_scores.append(score)

meteor = float(np.mean(meteor_scores))

# ------------------------------------------------------------
# ROUGE-L
# ------------------------------------------------------------

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_scores = []

for image_id in image_ids:

    hypothesis = res[image_id][0]

    score_values = []

    for reference in gts[image_id]:

        score = rouge.score(
            reference,
            hypothesis
        )["rougeL"].fmeasure

        score_values.append(score)

    rouge_scores.append(max(score_values))

rouge_l = float(np.mean(rouge_scores))

# ------------------------------------------------------------
# CIDEr
# ------------------------------------------------------------

cider = None
cider_error = None

try:

    from pycocoevalcap.cider.cider import Cider

    cider_scorer = Cider()

    cider_score, cider_per_image = cider_scorer.compute_score(
        gts,
        res
    )

    cider = float(cider_score)

except Exception as e:

    cider_error = repr(e)

# ------------------------------------------------------------
# SPICE
# ------------------------------------------------------------

spice = None
spice_status = (
    "Unavailable: Stanford CoreNLP/SPICE Java compatibility "
    "issue previously confirmed in this Colab environment."
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

final_results = {
    "model": "RT-DETR-X + Structured Object Transformer",
    "dataset": "Combined Flickr8K + Flickr30K",
    "test_images": len(image_ids),

    "caption_vocabulary_size": len(caption_to_id),
    "object_vocabulary_size": len(object_to_id),

    "BLEU-1": float(bleu1),
    "BLEU-2": float(bleu2),
    "BLEU-3": float(bleu3),
    "BLEU-4": float(bleu4),

    "METEOR_NLTK": float(meteor),
    "ROUGE-L": float(rouge_l),

    "CIDEr": cider,
    "CIDEr_error": cider_error,

    "SPICE": spice,
    "SPICE_status": spice_status
}

print()
print("FINAL CLEAN MODEL RESULTS")
print("-" * 50)

for key, value in final_results.items():

    if key.endswith("_error") or key.endswith("_status"):
        continue

    if isinstance(value, float):
        print(f"{key:25s}: {value:.6f}")
    else:
        print(f"{key:25s}: {value}")

if cider_error:
    print()
    print("CIDEr warning:")
    print(cider_error)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

FINAL_RESULTS_PATH = EVAL_DIR / "final_clean_evaluation_results.json"

with open(
    FINAL_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print()
print(f"Saved final results:")
print(FINAL_RESULTS_PATH)

print()
print("✓ BLEU-1/2/3/4 calculated")
print("✓ METEOR calculated using NLTK")
print("✓ ROUGE-L calculated")
print("✓ CIDEr attempted")
print("✓ SPICE explicitly recorded as unavailable")
print("✓ No old leaked-vocabulary scores used")
print("=" * 75)
print("CELL 104 COMPLETE")
print("=" * 75)